In [ ]:
import subprocess, sys

def pip(*pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])

pip(
    "transformers>=4.38.0",
    "sentence-transformers>=2.6.0",
    "datasets>=2.18.0",
    "accelerate>=0.28.0",
    "scikit-learn>=1.4.0",
    "pandas>=2.0.0",
    "numpy>=1.26.0",
    "matplotlib>=3.8.0",
    "seaborn>=0.13.0",
    "torch>=2.1.0",
    "tqdm>=4.66.0",
    "scipy>=1.12.0",
    "sentencepiece",
    "protobuf",
    "evaluate",
    "rouge_score",
)

print("✅ All dependencies installed.")

In [ ]:
import os, random, warnings, json, time, zipfile, shutil
import numpy as np
import torch

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

BASE_DIR   = "/content"
DATA_FILE  = os.path.join(BASE_DIR, "pmc_pair_dataset_balanced_clean.csv")
if not os.path.exists(DATA_FILE):
    alt1 = os.path.join(BASE_DIR, "pmc_pair_dataset_balanced.csv")
    alt2 = os.path.join(BASE_DIR, "paper_level_with_subdomains.csv")
    if os.path.exists(alt1):
        DATA_FILE = alt1
    elif os.path.exists(alt2):
        DATA_FILE = alt2

PROCESSED_DIR = os.path.join(BASE_DIR, "processed_splits")
RESULTS_BASE   = os.path.join(BASE_DIR, "saved_models")
MODELS_DIR     = RESULTS_BASE
MODEL_NAME     = "Base"
PIPELINE_NAME  = "All Models"

ROOT_DIR       = os.path.join(MODELS_DIR, MODEL_NAME)
MODEL_DIR      = os.path.join(ROOT_DIR, "model")
TOKENIZER_DIR  = os.path.join(ROOT_DIR, "tokenizer")
RESULTS_DIR    = os.path.join(ROOT_DIR, "results")

BEST_CKPT      = os.path.join(ROOT_DIR, "best_model.pt")
METRICS_PATH   = os.path.join(ROOT_DIR, "metrics.json")
LABEL_MAP_PATH  = os.path.join(ROOT_DIR, "label_map.json")
THRESH_PATH    = os.path.join(ROOT_DIR, "threshold.json")
INFER_PATH     = os.path.join(ROOT_DIR, "inference.py")
CONFIG_PATH    = os.path.join(ROOT_DIR, "training_config.json")
PACKAGE_ZIP    = os.path.join(MODELS_DIR, f"{MODEL_NAME}_package.zip")

for d in [PROCESSED_DIR, ROOT_DIR, MODEL_DIR, TOKENIZER_DIR, RESULTS_DIR]:
    os.makedirs(d, exist_ok=True)

MAX_ABS_WORDS   = 300
MAX_CONCL_WORDS = 300
MAX_TOKEN_LEN   = 512

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {DEVICE}")
print(f"Seed   : {SEED}")
print(f"Data   : {DATA_FILE}")
print(f"Root   : {ROOT_DIR}")
print(f"Model  : {MODEL_DIR}")
print(f"Tok    : {TOKENIZER_DIR}")
print(f"Plots  : {RESULTS_DIR}")

In [ ]:
import pandas as pd
import os

REQUIRED_COLS = ["label", "abstract_raw", "conclusion_raw", "abstract_subdomain", "conclusion_subdomain"]
OPTIONAL_COLS = [
    "pair_type", "abstract_paper_id", "conclusion_paper_id",
    "abstract_title", "conclusion_title",
    "abstract_model", "conclusion_model",
    "abstract_wc", "conclusion_wc", "input_text",
    "abstract_subdomains_all", "conclusion_subdomains_all"
]

assert os.path.exists(DATA_FILE), f"❌ Dataset not found at {DATA_FILE}"

df = pd.read_csv(DATA_FILE, low_memory=False)
print(f"✅ Loaded {len(df):,} rows × {df.shape[1]} cols")
print("Columns:", df.columns.tolist())

missing = [c for c in REQUIRED_COLS if c not in df.columns]
assert not missing, f"❌ Missing required columns: {missing}"
print("✅ All required columns present.")

vc = df["label"].value_counts()
print("\nLabel distribution:")
print(vc.to_string())
print(f"Positive rate: {vc.get(1,0)/len(df)*100:.1f}%")

In [ ]:
import re
import unicodedata

def clean_text(text: str, max_words: int = None) -> str:
    if not isinstance(text, str) or not text.strip():
        return ""
    text = unicodedata.normalize("NFC", text)
    text = re.sub(r"\[\d[\d,;\s\-]*\]", " ", text)
    text = re.sub(r"\(\d[\d,;\s\-]*\)", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    if max_words and max_words > 0:
        words = text.split()
        if len(words) > max_words:
            text = " ".join(words[:max_words])
    return text

def pick_first_existing(row, cols):
    for c in cols:
        if c in row and pd.notna(row[c]):
            val = row[c]
            if isinstance(val, str) and val.strip():
                return val
            if not isinstance(val, str):
                return val
    return ""

print("Cleaning texts…")

abs_src_cols = ["abstract_raw", "abstract_clean", "abstract_model", "abstract"]
con_src_cols = ["conclusion_raw", "conclusion_clean", "conclusion_model", "conclusion"]

if "abstract_raw" not in df.columns:
    df["abstract_raw"] = df.apply(lambda r: pick_first_existing(r, abs_src_cols), axis=1)
if "conclusion_raw" not in df.columns:
    df["conclusion_raw"] = df.apply(lambda r: pick_first_existing(r, con_src_cols), axis=1)

df["abstract_clean"] = df["abstract_raw"].apply(lambda x: clean_text(x, max_words=MAX_ABS_WORDS))
df["conclusion_clean"] = df["conclusion_raw"].apply(lambda x: clean_text(x, max_words=MAX_CONCL_WORDS))

before = len(df)
df = df[df["abstract_clean"].str.len() > 0].copy()
df = df[df["conclusion_clean"].str.len() > 0].copy()
df = df.dropna(subset=["label"]).copy()
df["label"] = df["label"].astype(int)

print(f"Dropped {before - len(df):,} rows with empty/missing text.")
print(f"Remaining: {len(df):,} rows")

df["input_text_clean"] = "Abstract: " + df["abstract_clean"] + " [SEP] Conclusion: " + df["conclusion_clean"]
df["abs_wc"] = df["abstract_clean"].str.split().str.len()
df["concl_wc"] = df["conclusion_clean"].str.split().str.len()

print("\nAbstract word-count stats:")
print(df["abs_wc"].describe().to_string())
print("\nConclusion word-count stats:")
print(df["concl_wc"].describe().to_string())

print("\nExample row:")
print(df[["label", "abstract_subdomain", "conclusion_subdomain", "abstract_clean", "conclusion_clean"]].head(1).to_dict("records")[0])

In [ ]:
from sklearn.model_selection import train_test_split

TRAIN_FRAC = 0.7
VAL_FRAC   = 0.1
TEST_FRAC  = 0.2

def make_safe_stratify_key(frame, min_count=4):
    """
    Primary stratification key = label + abstract_subdomain.
    Rare strata are collapsed to label-only to avoid skew / split failures.
    """
    base = (
        frame["label"].astype(str)
        + "|"
        + frame["abstract_subdomain"].fillna("UNK").astype(str)
    )
    counts = base.value_counts()
    safe = base.where(base.map(counts) >= min_count, frame["label"].astype(str))
    return safe

def safe_split(df, test_size, stratify_series, random_state=SEED):
    try:
        return train_test_split(
            df,
            test_size=test_size,
            stratify=stratify_series,
            random_state=random_state
        )
    except ValueError:
        print("⚠️ Stratified split failed on composite strata; falling back to label-only stratification.")
        return train_test_split(
            df,
            test_size=test_size,
            stratify=df["label"],
            random_state=random_state
        )

df = df.reset_index(drop=True)

# 20% test
train_val_df, test_df = safe_split(
    df,
    test_size=TEST_FRAC,
    stratify_series=make_safe_stratify_key(df),
    random_state=SEED
)

# 10% val from total = 10 / 80 = 12.5% of train_val
relative_val_size = VAL_FRAC / (TRAIN_FRAC + VAL_FRAC)

train_df, val_df = safe_split(
    train_val_df,
    test_size=relative_val_size,
    stratify_series=make_safe_stratify_key(train_val_df),
    random_state=SEED
)

# Final shuffle within each split
train_df = train_df.sample(frac=1, random_state=SEED).reset_index(drop=True)
val_df   = val_df.sample(frac=1, random_state=SEED).reset_index(drop=True)
test_df  = test_df.sample(frac=1, random_state=SEED).reset_index(drop=True)

print("Split sizes:")
print(f"  Train      : {len(train_df):,} ({len(train_df)/len(df)*100:.1f}%)")
print(f"  Validation : {len(val_df):,} ({len(val_df)/len(df)*100:.1f}%)")
print(f"  Test       : {len(test_df):,} ({len(test_df)/len(df)*100:.1f}%)")

for name, split in [("train", train_df), ("val", val_df), ("test", test_df)]:
    vc = split["label"].value_counts()
    print(f"{name:5s} → 0:{vc.get(0,0):,}  1:{vc.get(1,0):,}  pos_rate={vc.get(1,0)/len(split)*100:.1f}%")
    print(split["abstract_subdomain"].value_counts().head(5).to_string())

train_df.to_csv(os.path.join(PROCESSED_DIR, "train.csv"), index=False)
val_df.to_csv(os.path.join(PROCESSED_DIR, "val.csv"), index=False)
test_df.to_csv(os.path.join(PROCESSED_DIR, "test.csv"), index=False)
print(f"\n✅ Splits saved to {PROCESSED_DIR}")

In [ ]:
import json, time, shutil, zipfile
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, log_loss,
    confusion_matrix, classification_report,
    roc_curve, precision_recall_curve, auc
)
from sklearn.utils.class_weight import compute_class_weight

sns.set_theme(style="whitegrid", font_scale=1.0)
PALETTE = ["#4C72B0", "#DD8452", "#55A868", "#C44E52", "#8172B2", "#937860"]

ALL_RESULTS = {}

def compute_all_metrics(y_true, y_pred, y_prob=None):
    report = classification_report(y_true, y_pred, output_dict=True, zero_division=0)
    m = {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision_macro": report["macro avg"]["precision"],
        "recall_macro": report["macro avg"]["recall"],
        "f1_macro": report["macro avg"]["f1-score"],
        "precision_weighted": report["weighted avg"]["precision"],
        "recall_weighted": report["weighted avg"]["recall"],
        "f1_weighted": report["weighted avg"]["f1-score"],
        "precision_class0": report["0"]["precision"],
        "recall_class0": report["0"]["recall"],
        "f1_class0": report["0"]["f1-score"],
        "precision_class1": report["1"]["precision"],
        "recall_class1": report["1"]["recall"],
        "f1_class1": report["1"]["f1-score"],
    }

    if y_prob is not None:
        try:
            m["roc_auc"] = roc_auc_score(y_true, y_prob)
        except Exception:
            m["roc_auc"] = float("nan")
        try:
            m["pr_auc"] = average_precision_score(y_true, y_prob)
        except Exception:
            m["pr_auc"] = float("nan")
        try:
            m["log_loss"] = log_loss(y_true, np.column_stack([1 - np.array(y_prob), y_prob]))
        except Exception:
            m["log_loss"] = float("nan")

    cm = confusion_matrix(y_true, y_pred)
    if cm.shape == (2, 2):
        tn, fp, fn, tp = cm.ravel()
        m["fpr"] = fp / (fp + tn) if (fp + tn) > 0 else 0.0
        m["fnr"] = fn / (fn + tp) if (fn + tp) > 0 else 0.0

    return m

def compute_hard_negative_accuracy(y_true, y_pred, abs_wc, concl_wc):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    abs_wc = np.array(abs_wc)
    concl_wc = np.array(concl_wc)

    # hard negatives = negative examples with similar length and predicted as positive
    mask = (y_true == 0) & (np.abs(abs_wc - concl_wc) < 50)
    idx = np.where(mask)[0]
    if len(idx) == 0:
        return {"hard_neg_count": 0, "hard_neg_accuracy": float("nan")}
    return {
        "hard_neg_count": int(len(idx)),
        "hard_neg_accuracy": float(accuracy_score(y_true[idx], y_pred[idx])),
    }

def print_metrics(m, model_name="Model"):
    print(f"\n{'='*60}")
    print(f"{model_name} — Metrics")
    print(f"{'='*60}")
    for k, v in m.items():
        try:
            print(f"{k:<30s}: {v:.4f}")
        except Exception:
            print(f"{k:<30s}: {v}")

def save_metrics(metrics, path):
    with open(path, "w") as f:
        json.dump({k: (float(v) if isinstance(v, (np.floating, float)) and np.isfinite(v) else v)
                   for k, v in metrics.items()}, f, indent=2)

def plot_confusion_matrix(y_true, y_pred, model_name, save_dir):
    cm = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax,
                xticklabels=["Diff (0)", "Same (1)"],
                yticklabels=["Diff (0)", "Same (1)"])
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_title(f"{model_name} — Confusion Matrix")
    path = os.path.join(save_dir, f"{model_name}_confusion_matrix.png")
    plt.tight_layout()
    plt.savefig(path, dpi=160, bbox_inches="tight")
    plt.show()
    plt.close()
    print(f"Saved → {path}")

def plot_roc_curve(y_true, y_prob, model_name, save_dir):
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    auc_score = roc_auc_score(y_true, y_prob)
    fig, ax = plt.subplots(figsize=(5, 4))
    ax.plot(fpr, tpr, lw=2, label=f"AUC={auc_score:.3f}")
    ax.plot([0,1], [0,1], "k--", lw=1)
    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.set_title(f"{model_name} — ROC Curve")
    ax.legend(loc="lower right")
    path = os.path.join(save_dir, f"{model_name}_roc_curve.png")
    plt.tight_layout()
    plt.savefig(path, dpi=160, bbox_inches="tight")
    plt.show()
    plt.close()
    print(f"Saved → {path}")

def plot_pr_curve(y_true, y_prob, model_name, save_dir):
    precision, recall, _ = precision_recall_curve(y_true, y_prob)
    ap = average_precision_score(y_true, y_prob)
    fig, ax = plt.subplots(figsize=(5, 4))
    ax.plot(recall, precision, lw=2, label=f"AP={ap:.3f}")
    ax.set_xlabel("Recall")
    ax.set_ylabel("Precision")
    ax.set_title(f"{model_name} — Precision-Recall Curve")
    ax.legend(loc="upper right")
    path = os.path.join(save_dir, f"{model_name}_pr_curve.png")
    plt.tight_layout()
    plt.savefig(path, dpi=160, bbox_inches="tight")
    plt.show()
    plt.close()
    print(f"Saved → {path}")

def plot_training_curves(train_losses, val_losses, train_accs, val_accs, model_name, save_dir):
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    epochs = range(1, len(train_losses) + 1)

    axes[0].plot(epochs, train_losses, "o-", label="Train")
    if val_losses:
        axes[0].plot(epochs, val_losses, "s--", label="Val")
    axes[0].set_title(f"{model_name} — Loss")
    axes[0].set_xlabel("Epoch")
    axes[0].legend()

    axes[1].plot(epochs, train_accs, "o-", label="Train")
    if val_accs:
        axes[1].plot(epochs, val_accs, "s--", label="Val")
    axes[1].set_title(f"{model_name} — Accuracy")
    axes[1].set_xlabel("Epoch")
    axes[1].legend()

    path = os.path.join(save_dir, f"{model_name}_training_curves.png")
    plt.tight_layout()
    plt.savefig(path, dpi=160, bbox_inches="tight")
    plt.show()
    plt.close()
    print(f"Saved → {path}")

def plot_subdomain_performance(y_true, y_pred, subdomains, model_name, save_dir):
    df_e = pd.DataFrame({"true": y_true, "pred": y_pred, "subdomain": subdomains})
    grp = df_e.groupby("subdomain").apply(
        lambda g: accuracy_score(g["true"], g["pred"]) if len(g) >= 3 else np.nan
    ).dropna().sort_values(ascending=False)

    if grp.empty:
        print("Not enough samples for subdomain plot.")
        return

    fig, ax = plt.subplots(figsize=(max(8, len(grp) * 0.75), 4))
    grp.plot(kind="bar", ax=ax, color=PALETTE[0])
    ax.axhline(accuracy_score(y_true, y_pred), color="red", ls="--", label="Overall")
    ax.set_ylabel("Accuracy")
    ax.set_title(f"{model_name} — Subdomain Accuracy")
    ax.legend()
    plt.xticks(rotation=45, ha="right")
    path = os.path.join(save_dir, f"{model_name}_subdomain_performance.png")
    plt.tight_layout()
    plt.savefig(path, dpi=160, bbox_inches="tight")
    plt.show()
    plt.close()
    print(f"Saved → {path}")

def plot_length_bucket_performance(y_true, y_pred, lengths, model_name, save_dir):
    lengths = np.array(lengths)
    bins = [0, 100, 200, 300, 400, 600, 100000]
    labels = ["<100", "100-200", "200-300", "300-400", "400-600", "600+"]
    buckets = pd.cut(lengths, bins=bins, labels=labels, right=False)

    df_e = pd.DataFrame({"true": y_true, "pred": y_pred, "bucket": buckets})
    grp = df_e.groupby("bucket", observed=True).apply(
        lambda g: accuracy_score(g["true"], g["pred"]) if len(g) >= 3 else np.nan
    ).dropna()

    if grp.empty:
        print("Not enough samples for length-bucket plot.")
        return

    fig, ax = plt.subplots(figsize=(8, 4))
    grp.plot(kind="bar", ax=ax, color=PALETTE[1])
    ax.set_ylim(0, 1)
    ax.set_ylabel("Accuracy")
    ax.set_title(f"{model_name} — Accuracy by Length Bucket")
    ax.set_xlabel("Word-count bucket")
    plt.xticks(rotation=30, ha="right")
    path = os.path.join(save_dir, f"{model_name}_length_bucket.png")
    plt.tight_layout()
    plt.savefig(path, dpi=160, bbox_inches="tight")
    plt.show()
    plt.close()
    print(f"Saved → {path}")

def save_label_map(root_dir):
    label_map = {"0": "Different subdomain", "1": "Same subdomain"}
    with open(os.path.join(root_dir, "label_map.json"), "w") as f:
        json.dump(label_map, f, indent=2)

def save_threshold(root_dir, threshold, val_f1):
    payload = {"threshold": float(threshold), "val_f1_macro": float(val_f1)}
    with open(os.path.join(root_dir, "threshold.json"), "w") as f:
        json.dump(payload, f, indent=2)

def save_training_config(root_dir):
    cfg = {
        "pipeline_name": PIPELINE_NAME,
        "model_name": MODEL_NAME,
        "backbone": "sentence-transformers/all-MiniLM-L6-v2",
        "max_abs_words": MAX_ABS_WORDS,
        "max_concl_words": MAX_CONCL_WORDS,
        "max_token_len": MAX_TOKEN_LEN,
        "train_frac": TRAIN_FRAC,
        "val_frac": VAL_FRAC,
        "test_frac": TEST_FRAC,
        "seed": SEED,
    }
    with open(os.path.join(root_dir, "training_config.json"), "w") as f:
        json.dump(cfg, f, indent=2)

def package_model(root_dir, model_name):
    zip_base = os.path.join(MODELS_DIR, f"{model_name}_package")
    zip_path = shutil.make_archive(zip_base, "zip", root_dir=MODELS_DIR, base_dir=model_name)
    return zip_path

print("✅ Helpers loaded.")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SBERT training + export in one cell
# Updated for stronger performance, safer caching, and better T4 utilization
# ─────────────────────────────────────────────────────────────────────────────

!pip -q install sentence-transformers

import os
import re
import json
import time
import shutil
import random
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.cuda.amp import autocast, GradScaler
from torch.utils.data import DataLoader, TensorDataset

from sentence_transformers import SentenceTransformer
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, precision_recall_curve, average_precision_score,
    confusion_matrix, roc_curve, auc, classification_report, log_loss
)
from sklearn.utils.class_weight import compute_class_weight
import matplotlib.pyplot as plt
import seaborn as sns

# ── Reproducibility ──────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
try:
    torch.set_float32_matmul_precision("high")
except Exception:
    pass

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

# ── Paths ───────────────────────────────────────────────────────────────────
BASE_DIR = "/content"
PROCESSED_DIR = os.path.join(BASE_DIR, "processed_splits")
MODELS_DIR = globals().get("MODELS_DIR", os.path.join(BASE_DIR, "saved_models"))

MODEL_NAME = "SBERT"
ROOT_DIR = os.path.join(MODELS_DIR, MODEL_NAME)
MODEL_DIR = os.path.join(ROOT_DIR, "model")
TOKENIZER_DIR = os.path.join(ROOT_DIR, "tokenizer")
RESULTS_DIR = os.path.join(ROOT_DIR, "results")
CACHE_DIR = os.path.join(ROOT_DIR, "cache")

BEST_CKPT = os.path.join(ROOT_DIR, "best_model.pt")
METRICS_PATH = os.path.join(ROOT_DIR, "metrics.json")
LABEL_MAP_PATH = os.path.join(ROOT_DIR, "label_map.json")
THRESH_PATH = os.path.join(ROOT_DIR, "threshold.json")
INFER_PATH = os.path.join(ROOT_DIR, "inference.py")
CONFIG_PATH = os.path.join(ROOT_DIR, "training_config.json")
PACKAGE_ZIP = os.path.join(MODELS_DIR, f"{MODEL_NAME}_package.zip")

for d in [ROOT_DIR, MODEL_DIR, TOKENIZER_DIR, RESULTS_DIR, CACHE_DIR]:
    os.makedirs(d, exist_ok=True)

PALETTE = ["#4C72B0", "#DD8452", "#55A868", "#C44E52", "#8172B2", "#937860"]

print(f"Root     : {ROOT_DIR}")
print(f"Model    : {MODEL_DIR}")
print(f"Tokenizer: {TOKENIZER_DIR}")
print(f"Results  : {RESULTS_DIR}")
print(f"Archive  : {PACKAGE_ZIP}")

# ── Data loading ────────────────────────────────────────────────────────────
train_df = pd.read_csv(os.path.join(PROCESSED_DIR, "train.csv"))
val_df   = pd.read_csv(os.path.join(PROCESSED_DIR, "val.csv"))
test_df  = pd.read_csv(os.path.join(PROCESSED_DIR, "test.csv"))

print(f"Loaded splits — train:{len(train_df)}  val:{len(val_df)}  test:{len(test_df)}")

def resolve_text_columns(df):
    abs_candidates = ["abstract_clean", "abstract_raw", "abstract_model", "abstract"]
    con_candidates = ["conclusion_clean", "conclusion_raw", "conclusion_model", "conclusion"]

    abs_col = next((c for c in abs_candidates if c in df.columns), None)
    con_col = next((c for c in con_candidates if c in df.columns), None)

    if abs_col is None or con_col is None:
        raise ValueError("Could not find abstract/conclusion columns in the split files.")

    return abs_col, con_col

ABS_COL, CONCL_COL = resolve_text_columns(train_df)
print(f"Using text columns: {ABS_COL}, {CONCL_COL}")

# ── Cleaning ────────────────────────────────────────────────────────────────
MAX_ABS_WORDS = 300
MAX_CONCL_WORDS = 300
MAX_TOKEN_LEN = 256

def clean_text(text: str, max_words: int = None) -> str:
    if not isinstance(text, str) or not text.strip():
        return ""
    text = unicodedata.normalize("NFC", text)
    text = re.sub(r"\[\d[\d,;\s\-]*\]", " ", text)
    text = re.sub(r"\(\d[\d,;\s\-]*\)", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    if max_words and max_words > 0:
        words = text.split()
        if len(words) > max_words:
            text = " ".join(words[:max_words])
    return text

for split_name, split_df in [("train", train_df), ("val", val_df), ("test", test_df)]:
    split_df[ABS_COL] = split_df[ABS_COL].apply(lambda x: clean_text(x, max_words=MAX_ABS_WORDS))
    split_df[CONCL_COL] = split_df[CONCL_COL].apply(lambda x: clean_text(x, max_words=MAX_CONCL_WORDS))
    split_df.dropna(subset=["label"], inplace=True)
    split_df["label"] = split_df["label"].astype(int)

    before = len(split_df)
    split_df = split_df[(split_df[ABS_COL].str.len() > 0) & (split_df[CONCL_COL].str.len() > 0)].copy()
    split_df["abs_wc"] = split_df[ABS_COL].str.split().str.len()
    split_df["concl_wc"] = split_df[CONCL_COL].str.split().str.len()
    split_df = split_df[(split_df["abs_wc"] <= 1000) & (split_df["concl_wc"] <= 1000)].copy()

    if split_name == "train":
        train_df = split_df.reset_index(drop=True)
    elif split_name == "val":
        val_df = split_df.reset_index(drop=True)
    else:
        test_df = split_df.reset_index(drop=True)

    print(f"{split_name}: dropped {before - len(split_df):,} rows; remaining {len(split_df):,}")

for split_df in [train_df, val_df, test_df]:
    split_df["input_text_clean"] = "Abstract: " + split_df[ABS_COL] + " [SEP] Conclusion: " + split_df[CONCL_COL]

TOTAL_N = len(train_df) + len(val_df) + len(test_df)
TRAIN_FRAC = len(train_df) / TOTAL_N if TOTAL_N else 0
VAL_FRAC = len(val_df) / TOTAL_N if TOTAL_N else 0
TEST_FRAC = len(test_df) / TOTAL_N if TOTAL_N else 0

# ── Backbone / hyperparameters ──────────────────────────────────────────────
SBERT_CKPT = "sentence-transformers/all-mpnet-base-v2"

# Larger batches for better T4 utilization
ENCODE_BS = 256
BATCH_SIZE = 256
GRAD_ACCUM = 1

LR = 2e-4
WEIGHT_DECAY = 1e-2
N_EPOCHS = 16
PATIENCE = 4
MAX_GRAD_NORM = 1.0
DROPOUT = 0.25
LABEL_SMOOTHING = 0.01
NUM_WORKERS = 2
USE_FP16 = (DEVICE.type == "cuda")

# ── Class weights ───────────────────────────────────────────────────────────
y_train = train_df["label"].astype(int).values
classes = np.array([0, 1])
class_w = compute_class_weight(class_weight="balanced", classes=classes, y=y_train)
w_tensor = torch.tensor(class_w, dtype=torch.float32, device=DEVICE)
print("Class weights:", {0: float(class_w[0]), 1: float(class_w[1])})

# ── Encoder ────────────────────────────────────────────────────────────────
print(f"Loading SBERT backbone: {SBERT_CKPT}")
sbert = SentenceTransformer(SBERT_CKPT, device=str(DEVICE))
sbert.max_seq_length = MAX_TOKEN_LEN
sbert.eval()

class PairDataset(torch.utils.data.Dataset):
    def __init__(self, df, abs_col, concl_col):
        self.abstracts = df[abs_col].astype(str).tolist()
        self.conclusions = df[concl_col].astype(str).tolist()
        self.labels = df["label"].astype(int).tolist()

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, i):
        return self.abstracts[i], self.conclusions[i], self.labels[i]

class SBERTClassifier(nn.Module):
    def __init__(self, emb_dim=768, dropout=0.25):
        super().__init__()
        feat_dim = emb_dim * 4 + 1  # a, c, |a-c|, a*c, cosine similarity
        self.net = nn.Sequential(
            nn.Linear(feat_dim, 1024),
            nn.GELU(),
            nn.LayerNorm(1024),
            nn.Dropout(dropout),

            nn.Linear(1024, 256),
            nn.GELU(),
            nn.Dropout(dropout),

            nn.Linear(256, 64),
            nn.GELU(),
            nn.Dropout(dropout),

            nn.Linear(64, 2)
        )

    def forward(self, a, c):
        a = nn.functional.normalize(a, p=2, dim=-1)
        c = nn.functional.normalize(c, p=2, dim=-1)
        diff = torch.abs(a - c)
        prod = a * c
        cos = torch.sum(a * c, dim=-1, keepdim=True)
        x = torch.cat([a, c, diff, prod, cos], dim=-1)
        return self.net(x)

# ── Cache helpers ───────────────────────────────────────────────────────────
def cache_file(name):
    # versioned cache key to avoid stale embeddings when settings change
    safe_ckpt = SBERT_CKPT.replace("/", "_")
    return os.path.join(
        CACHE_DIR,
        f"{name}_{safe_ckpt}_abs{MAX_ABS_WORDS}_con{MAX_CONCL_WORDS}_tok{MAX_TOKEN_LEN}.npz"
    )

def encode_split(df, name):
    fpath = cache_file(name)
    if os.path.exists(fpath):
        data = np.load(fpath)
        abs_embs = data["abs_embs"]
        con_embs = data["con_embs"]
        labels = data["labels"]
        print(f"Loaded cached embeddings for {name}: {abs_embs.shape}")
    else:
        print(f"Encoding {name} split...")
        abs_embs = sbert.encode(
            df[ABS_COL].astype(str).tolist(),
            batch_size=ENCODE_BS,
            show_progress_bar=True,
            convert_to_numpy=True,
            normalize_embeddings=True,
            device=str(DEVICE)
        )
        con_embs = sbert.encode(
            df[CONCL_COL].astype(str).tolist(),
            batch_size=ENCODE_BS,
            show_progress_bar=True,
            convert_to_numpy=True,
            normalize_embeddings=True,
            device=str(DEVICE)
        )
        labels = df["label"].astype(int).values
        np.savez(
            fpath,
            abs_embs=abs_embs.astype(np.float32),
            con_embs=con_embs.astype(np.float32),
            labels=labels.astype(np.int64),
        )
        print(f"Saved cache: {fpath}")

    return (
        torch.tensor(abs_embs, dtype=torch.float32),
        torch.tensor(con_embs, dtype=torch.float32),
        torch.tensor(labels, dtype=torch.long),
    )

trn_abs, trn_con, trn_lbl = encode_split(train_df, "train")
val_abs, val_con, val_lbl = encode_split(val_df, "val")
tst_abs, tst_con, tst_lbl = encode_split(test_df, "test")

loader_kwargs = dict(
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    pin_memory=(DEVICE.type == "cuda"),
    persistent_workers=(NUM_WORKERS > 0),
)

trn_loader = DataLoader(TensorDataset(trn_abs, trn_con, trn_lbl), shuffle=True, **loader_kwargs)
val_loader = DataLoader(TensorDataset(val_abs, val_con, val_lbl), shuffle=False, **loader_kwargs)
tst_loader = DataLoader(TensorDataset(tst_abs, tst_con, tst_lbl), shuffle=False, **loader_kwargs)

EMB_DIM = trn_abs.shape[1]
print("Embedding dim:", EMB_DIM)

clf = SBERTClassifier(emb_dim=EMB_DIM, dropout=DROPOUT).to(DEVICE)

try:
    clf = torch.compile(clf)
    print("torch.compile enabled for classifier.")
except Exception:
    pass

criterion = nn.CrossEntropyLoss(weight=w_tensor, label_smoothing=LABEL_SMOOTHING)

try:
    optimizer = torch.optim.AdamW(
        clf.parameters(),
        lr=LR,
        weight_decay=WEIGHT_DECAY,
        betas=(0.9, 0.999),
        eps=1e-8,
        fused=True
    )
    print("Using fused AdamW.")
except TypeError:
    optimizer = torch.optim.AdamW(
        clf.parameters(),
        lr=LR,
        weight_decay=WEIGHT_DECAY,
        betas=(0.9, 0.999),
        eps=1e-8
    )

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(N_EPOCHS, 1))
scaler = GradScaler(enabled=USE_FP16)

best_val_f1 = 0.0
best_val_threshold = 0.5
patience_ctr = 0
train_losses, val_losses, train_accs, val_accs = [], [], [], []

def predict_loader(model, loader):
    model.eval()
    all_probs, all_preds, all_true = [], [], []
    total_loss, total_n = 0.0, 0

    with torch.inference_mode():
        for ab, co, lb in loader:
            ab = ab.to(DEVICE, non_blocking=True)
            co = co.to(DEVICE, non_blocking=True)
            lb = lb.to(DEVICE, non_blocking=True)

            with autocast(enabled=USE_FP16):
                logits = model(ab, co)
                loss = criterion(logits, lb)

            probs = torch.softmax(logits, dim=-1)[:, 1]
            preds = logits.argmax(dim=-1)

            bs = lb.size(0)
            total_loss += loss.item() * bs
            total_n += bs

            all_probs.append(probs.detach().cpu().numpy())
            all_preds.append(preds.detach().cpu().numpy())
            all_true.append(lb.detach().cpu().numpy())

    y_true = np.concatenate(all_true)
    probs = np.concatenate(all_probs)
    preds = np.concatenate(all_preds)
    avg_loss = total_loss / max(total_n, 1)
    return y_true, probs, preds, avg_loss

def best_threshold_for_f1(y_true, probs):
    thresholds = np.linspace(0.05, 0.95, 181)
    best_thr, best_f1 = 0.5, -1.0
    y_true = np.asarray(y_true, dtype=int)
    probs = np.asarray(probs, dtype=float)
    for thr in thresholds:
        preds = (probs >= thr).astype(int)
        f1 = f1_score(y_true, preds, average="macro", zero_division=0)
        if f1 > best_f1:
            best_f1 = f1
            best_thr = float(thr)
    return best_thr, float(best_f1)

def compute_hard_negative_accuracy(y_true, y_pred, abs_wc, concl_wc):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    abs_wc = np.array(abs_wc)
    concl_wc = np.array(concl_wc)

    mask = (y_true == 0) & (np.abs(abs_wc - concl_wc) < 50)
    idx = np.where(mask)[0]
    if len(idx) == 0:
        return {"hard_neg_count": 0, "hard_neg_accuracy": float("nan")}
    return {
        "hard_neg_count": int(len(idx)),
        "hard_neg_accuracy": float(accuracy_score(y_true[idx], y_pred[idx])),
    }

def save_metrics(metrics, path):
    clean = {}
    for k, v in metrics.items():
        if isinstance(v, (np.floating, float)):
            clean[k] = float(v) if np.isfinite(v) else None
        elif isinstance(v, (np.integer, int)):
            clean[k] = int(v)
        else:
            clean[k] = v
    with open(path, "w") as f:
        json.dump(clean, f, indent=2)

def save_label_map(root_dir):
    label_map = {"0": "Different subdomain", "1": "Same subdomain"}
    with open(os.path.join(root_dir, "label_map.json"), "w") as f:
        json.dump(label_map, f, indent=2)

def save_threshold(root_dir, threshold, val_f1):
    payload = {"threshold": float(threshold), "val_f1_macro": float(val_f1)}
    with open(os.path.join(root_dir, "threshold.json"), "w") as f:
        json.dump(payload, f, indent=2)

def save_training_config(root_dir):
    cfg = {
        "pipeline_name": PIPELINE_NAME,
        "model_name": MODEL_NAME,
        "backbone": SBERT_CKPT,
        "max_abs_words": MAX_ABS_WORDS,
        "max_concl_words": MAX_CONCL_WORDS,
        "max_token_len": MAX_TOKEN_LEN,
        "train_frac": TRAIN_FRAC,
        "val_frac": VAL_FRAC,
        "test_frac": TEST_FRAC,
        "seed": SEED,
        "batch_size": BATCH_SIZE,
        "encode_batch_size": ENCODE_BS,
        "lr": LR,
        "weight_decay": WEIGHT_DECAY,
        "epochs": N_EPOCHS,
        "patience": PATIENCE,
        "dropout": DROPOUT,
        "label_smoothing": LABEL_SMOOTHING,
        "grad_accum": GRAD_ACCUM,
        "max_grad_norm": MAX_GRAD_NORM,
        "num_workers": NUM_WORKERS,
    }
    with open(os.path.join(root_dir, "training_config.json"), "w") as f:
        json.dump(cfg, f, indent=2)

def plot_confusion_matrix(y_true, y_pred, model_name, save_dir):
    cm = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(5, 4))
    sns.heatmap(
        cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax,
        xticklabels=["Diff (0)", "Same (1)"],
        yticklabels=["Diff (0)", "Same (1)"]
    )
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_title(f"{model_name} — Confusion Matrix")
    path = os.path.join(save_dir, f"{model_name}_confusion_matrix.png")
    plt.tight_layout()
    plt.savefig(path, dpi=160, bbox_inches="tight")
    plt.show()
    plt.close()
    print(f"Saved → {path}")

def plot_roc_curve(y_true, y_prob, model_name, save_dir):
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    auc_score = roc_auc_score(y_true, y_prob)
    fig, ax = plt.subplots(figsize=(5, 4))
    ax.plot(fpr, tpr, lw=2, label=f"AUC={auc_score:.3f}")
    ax.plot([0,1], [0,1], "k--", lw=1)
    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.set_title(f"{model_name} — ROC Curve")
    ax.legend(loc="lower right")
    path = os.path.join(save_dir, f"{model_name}_roc_curve.png")
    plt.tight_layout()
    plt.savefig(path, dpi=160, bbox_inches="tight")
    plt.show()
    plt.close()
    print(f"Saved → {path}")

def plot_pr_curve(y_true, y_prob, model_name, save_dir):
    precision, recall, _ = precision_recall_curve(y_true, y_prob)
    ap = average_precision_score(y_true, y_prob)
    fig, ax = plt.subplots(figsize=(5, 4))
    ax.plot(recall, precision, lw=2, label=f"AP={ap:.3f}")
    ax.set_xlabel("Recall")
    ax.set_ylabel("Precision")
    ax.set_title(f"{model_name} — Precision-Recall Curve")
    ax.legend(loc="upper right")
    path = os.path.join(save_dir, f"{model_name}_pr_curve.png")
    plt.tight_layout()
    plt.savefig(path, dpi=160, bbox_inches="tight")
    plt.show()
    plt.close()
    print(f"Saved → {path}")

def plot_training_curves(train_losses, val_losses, train_accs, val_accs, model_name, save_dir):
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    epochs = range(1, len(train_losses) + 1)

    axes[0].plot(epochs, train_losses, "o-", label="Train")
    if val_losses:
        axes[0].plot(epochs, val_losses, "s--", label="Val")
    axes[0].set_title(f"{model_name} — Loss")
    axes[0].set_xlabel("Epoch")
    axes[0].legend()

    axes[1].plot(epochs, train_accs, "o-", label="Train")
    if val_accs:
        axes[1].plot(epochs, val_accs, "s--", label="Val")
    axes[1].set_title(f"{model_name} — Accuracy")
    axes[1].set_xlabel("Epoch")
    axes[1].legend()

    path = os.path.join(save_dir, f"{model_name}_training_curves.png")
    plt.tight_layout()
    plt.savefig(path, dpi=160, bbox_inches="tight")
    plt.show()
    plt.close()
    print(f"Saved → {path}")

def plot_subdomain_performance(y_true, y_pred, subdomains, model_name, save_dir):
    df_e = pd.DataFrame({"true": y_true, "pred": y_pred, "subdomain": subdomains})
    grp = df_e.groupby("subdomain").apply(
        lambda g: accuracy_score(g["true"], g["pred"]) if len(g) >= 3 else np.nan
    ).dropna().sort_values(ascending=False)

    if grp.empty:
        print("Not enough samples for subdomain plot.")
        return

    fig, ax = plt.subplots(figsize=(max(8, len(grp) * 0.75), 4))
    grp.plot(kind="bar", ax=ax, color=PALETTE[0])
    ax.axhline(accuracy_score(y_true, y_pred), color="red", ls="--", label="Overall")
    ax.set_ylabel("Accuracy")
    ax.set_title(f"{model_name} — Subdomain Accuracy")
    ax.legend()
    plt.xticks(rotation=45, ha="right")
    path = os.path.join(save_dir, f"{model_name}_subdomain_performance.png")
    plt.tight_layout()
    plt.savefig(path, dpi=160, bbox_inches="tight")
    plt.show()
    plt.close()
    print(f"Saved → {path}")

def plot_length_bucket_performance(y_true, y_pred, lengths, model_name, save_dir):
    lengths = np.asarray(lengths)
    bins = [0, 100, 200, 300, 400, 600, 100000]
    labels = ["<100", "100-200", "200-300", "300-400", "400-600", "600+"]
    buckets = pd.cut(lengths, bins=bins, labels=labels, right=False)

    df_e = pd.DataFrame({"true": y_true, "pred": y_pred, "bucket": buckets})
    grp = df_e.groupby("bucket", observed=True).apply(
        lambda g: accuracy_score(g["true"], g["pred"]) if len(g) >= 3 else np.nan
    ).dropna()

    if grp.empty:
        print("Not enough samples for length-bucket plot.")
        return

    fig, ax = plt.subplots(figsize=(8, 4))
    grp.plot(kind="bar", ax=ax, color=PALETTE[1])
    ax.set_ylim(0, 1)
    ax.set_ylabel("Accuracy")
    ax.set_title(f"{model_name} — Accuracy by Length Bucket")
    ax.set_xlabel("Word-count bucket")
    plt.xticks(rotation=30, ha="right")
    path = os.path.join(save_dir, f"{model_name}_length_bucket.png")
    plt.tight_layout()
    plt.savefig(path, dpi=160, bbox_inches="tight")
    plt.show()
    plt.close()
    print(f"Saved → {path}")

def write_inference_script():
    inference_code = f'''import json
import os
import re
import unicodedata
import torch
import torch.nn as nn
from sentence_transformers import SentenceTransformer

ROOT_DIR = os.path.dirname(os.path.abspath(__file__))
MODEL_DIR = os.path.join(ROOT_DIR, "model")
THRESH_PATH = os.path.join(ROOT_DIR, "threshold.json")
LABEL_MAP_PATH = os.path.join(ROOT_DIR, "label_map.json")
BEST_CKPT = os.path.join(ROOT_DIR, "best_model.pt")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def normalize_text(text):
    if text is None:
        return ""
    text = unicodedata.normalize("NFC", str(text))
    text = re.sub(r"\\[\\d[\\d,;\\s\\-]*\\]", " ", text)
    text = re.sub(r"\\(\\d[\\d,;\\s\\-]*\\)", " ", text)
    text = re.sub(r"\\s+", " ", text).strip()
    return text

def clean_text(text, max_words=300):
    text = normalize_text(text)
    if not text:
        return ""
    words = text.split()
    return " ".join(words[:max_words])

class SBERTClassifier(nn.Module):
    def __init__(self, emb_dim=768, dropout=0.25):
        super().__init__()
        feat_dim = emb_dim * 4 + 1
        self.net = nn.Sequential(
            nn.Linear(feat_dim, 1024),
            nn.GELU(),
            nn.LayerNorm(1024),
            nn.Dropout(dropout),
            nn.Linear(1024, 256),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(256, 64),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(64, 2),
        )

    def forward(self, a, c):
        a = nn.functional.normalize(a, p=2, dim=-1)
        c = nn.functional.normalize(c, p=2, dim=-1)
        diff = torch.abs(a - c)
        prod = a * c
        cos = torch.sum(a * c, dim=-1, keepdim=True)
        x = torch.cat([a, c, diff, prod, cos], dim=-1)
        return self.net(x)

def load_json(path, default):
    if os.path.exists(path):
        with open(path, "r") as f:
            return json.load(f)
    return default

THRESHOLD = float(load_json(THRESH_PATH, {{"threshold": 0.5}})["threshold"])
LABEL_MAP = load_json(LABEL_MAP_PATH, {{"0": "Different subdomain", "1": "Same subdomain"}})

# Local backbone saved in the package, no internet dependency
backbone = SentenceTransformer(MODEL_DIR, device=str(DEVICE))
backbone.max_seq_length = {MAX_TOKEN_LEN}
backbone.eval()

ckpt = torch.load(BEST_CKPT, map_location=DEVICE)
emb_dim = int(ckpt.get("emb_dim", 768))
dropout = float(ckpt.get("dropout", 0.25))

classifier = SBERTClassifier(emb_dim=emb_dim, dropout=dropout).to(DEVICE)
classifier.load_state_dict(ckpt["state_dict"])
classifier.eval()

def encode_text(text):
    text = clean_text(text)
    emb = backbone.encode(
        [text],
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False,
    )
    return torch.tensor(emb, dtype=torch.float32, device=DEVICE)

def predict_pair(abstract, conclusion):
    a = encode_text(abstract)
    c = encode_text(conclusion)
    with torch.no_grad():
        logits = classifier(a, c)
        prob_1 = torch.softmax(logits, dim=-1)[0, 1].item()
        pred = int(prob_1 >= THRESHOLD)
    return {{
        "label": pred,
        "label_name": LABEL_MAP.get(str(pred), str(pred)),
        "probability_same_subdomain": prob_1,
        "threshold": THRESHOLD
    }}

if __name__ == "__main__":
    print(predict_pair("Sample abstract.", "Sample conclusion."))
'''
    with open(INFER_PATH, "w") as f:
        f.write(inference_code)
    print(f"✅ Inference script saved to {INFER_PATH}")

def save_model_package():
    # Save the full SBERT module locally for offline inference
    sbert.save(MODEL_DIR)
    print(f"✅ Saved SBERT model to {MODEL_DIR}")

    # Save tokenizer separately as well
    try:
        transformer_module = sbert._first_module()
        transformer_module.tokenizer.save_pretrained(TOKENIZER_DIR)
        print(f"✅ Saved tokenizer to {TOKENIZER_DIR}")
    except Exception:
        pass

    save_label_map(ROOT_DIR)
    save_threshold(ROOT_DIR, best_val_threshold, best_val_f1)
    save_training_config(ROOT_DIR)
    write_inference_script()

    zip_base = os.path.join(MODELS_DIR, f"{MODEL_NAME}_package")
    zip_path = shutil.make_archive(
        zip_base,
        "zip",
        root_dir=MODELS_DIR,
        base_dir=MODEL_NAME
    )
    print(f"📦 Package archived → {zip_path}")

# ── Train ───────────────────────────────────────────────────────────────────
best_ckpt_path = BEST_CKPT
t0 = time.time()

for epoch in range(1, N_EPOCHS + 1):
    clf.train()
    ep_loss, ep_correct, ep_total = 0.0, 0, 0
    optimizer.zero_grad(set_to_none=True)

    for step, (ab, co, lb) in enumerate(trn_loader, start=1):
        ab = ab.to(DEVICE, non_blocking=True)
        co = co.to(DEVICE, non_blocking=True)
        lb = lb.to(DEVICE, non_blocking=True)

        with autocast(enabled=USE_FP16):
            logits = clf(ab, co)
            loss = criterion(logits, lb) / GRAD_ACCUM

        scaler.scale(loss).backward()

        ep_loss += loss.item() * GRAD_ACCUM
        preds = logits.argmax(dim=-1)
        ep_correct += (preds == lb).sum().item()
        ep_total += lb.size(0)

        if (step % GRAD_ACCUM == 0) or (step == len(trn_loader)):
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(clf.parameters(), MAX_GRAD_NORM)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)

    train_loss = ep_loss / max(len(trn_loader), 1)
    train_acc = ep_correct / max(ep_total, 1)
    train_losses.append(train_loss)
    train_accs.append(train_acc)

    y_val, p_val, pred_val, val_loss = predict_loader(clf, val_loader)
    val_acc_05 = accuracy_score(y_val, pred_val)
    val_f1_default = f1_score(y_val, pred_val, average="macro", zero_division=0)

    val_thr, val_f1_thr = best_threshold_for_f1(y_val, p_val)
    val_preds_best = (p_val >= val_thr).astype(int)
    val_acc_best = accuracy_score(y_val, val_preds_best)

    val_losses.append(val_loss)
    val_accs.append(val_acc_best)

    if val_f1_thr > best_val_f1:
        best_val_f1 = val_f1_thr
        best_val_threshold = val_thr
        patience_ctr = 0
        torch.save({
            "state_dict": clf.state_dict(),
            "emb_dim": EMB_DIM,
            "dropout": DROPOUT,
            "backbone": SBERT_CKPT,
        }, best_ckpt_path)
    else:
        patience_ctr += 1

    scheduler.step()

    print(
        f"Epoch {epoch:02d} | "
        f"train_loss={train_loss:.4f} | train_acc={train_acc:.3f} | "
        f"val_loss={val_loss:.4f} | val_acc@0.5={val_acc_05:.3f} | "
        f"val_f1@0.5={val_f1_default:.3f} | val_f1@best_thr={val_f1_thr:.3f} | "
        f"thr={val_thr:.3f} | patience={patience_ctr}"
    )

    if patience_ctr >= PATIENCE:
        print(f"Early stopping at epoch {epoch}.")
        break

train_time = time.time() - t0
print(f"\nTraining time: {train_time:.1f} seconds")
print(f"Best validation threshold: {best_val_threshold:.3f}")

# ── Evaluate best checkpoint ───────────────────────────────────────────────
ckpt = torch.load(best_ckpt_path, map_location=DEVICE)
clf.load_state_dict(ckpt["state_dict"])
clf.eval()

y_true, probs, raw_preds, _ = predict_loader(clf, tst_loader)
preds = (probs >= best_val_threshold).astype(int)

# ── Metrics ────────────────────────────────────────────────────────────────
metrics = {
    "accuracy": float(accuracy_score(y_true, preds)),
    "precision_macro": float(precision_score(y_true, preds, average="macro", zero_division=0)),
    "recall_macro": float(recall_score(y_true, preds, average="macro", zero_division=0)),
    "f1_macro": float(f1_score(y_true, preds, average="macro", zero_division=0)),
    "precision_weighted": float(precision_score(y_true, preds, average="weighted", zero_division=0)),
    "recall_weighted": float(recall_score(y_true, preds, average="weighted", zero_division=0)),
    "f1_weighted": float(f1_score(y_true, preds, average="weighted", zero_division=0)),
    "precision_class0": float(classification_report(y_true, preds, output_dict=True, zero_division=0)["0"]["precision"]),
    "recall_class0": float(classification_report(y_true, preds, output_dict=True, zero_division=0)["0"]["recall"]),
    "f1_class0": float(classification_report(y_true, preds, output_dict=True, zero_division=0)["0"]["f1-score"]),
    "precision_class1": float(classification_report(y_true, preds, output_dict=True, zero_division=0)["1"]["precision"]),
    "recall_class1": float(classification_report(y_true, preds, output_dict=True, zero_division=0)["1"]["recall"]),
    "f1_class1": float(classification_report(y_true, preds, output_dict=True, zero_division=0)["1"]["f1-score"]),
    "roc_auc": float(roc_auc_score(y_true, probs)) if len(np.unique(y_true)) > 1 else float("nan"),
    "pr_auc": float(average_precision_score(y_true, probs)),
    "log_loss": float(log_loss(y_true, np.column_stack([1 - np.array(probs), probs]))),
    "training_time_s": float(train_time),
    "best_val_threshold": float(best_val_threshold),
    "best_val_f1_macro": float(best_val_f1),
}
metrics.update(compute_hard_negative_accuracy(
    y_true, preds, test_df["abs_wc"].tolist(), test_df["concl_wc"].tolist()
))

print("\nFinal metrics:")
for k, v in metrics.items():
    try:
        print(f"{k:<30s}: {v:.4f}")
    except Exception:
        print(f"{k:<30s}: {v}")

# ── Plots ──────────────────────────────────────────────────────────────────
plot_confusion_matrix(y_true, preds, MODEL_NAME, RESULTS_DIR)
plot_roc_curve(y_true, probs, MODEL_NAME, RESULTS_DIR)
plot_pr_curve(y_true, probs, MODEL_NAME, RESULTS_DIR)
plot_training_curves(train_losses, val_losses, train_accs, val_accs, MODEL_NAME, RESULTS_DIR)

if "abstract_subdomain" in test_df.columns:
    plot_subdomain_performance(
        y_true, preds,
        test_df["abstract_subdomain"].fillna("unknown").astype(str).tolist(),
        MODEL_NAME, RESULTS_DIR
    )

plot_length_bucket_performance(
    y_true, preds,
    (test_df["abs_wc"].fillna(0) + test_df["concl_wc"].fillna(0)).tolist(),
    MODEL_NAME, RESULTS_DIR
)

# ── Save metrics/predictions/checkpoint ────────────────────────────────────
save_metrics(metrics, METRICS_PATH)
print(f"✅ Metrics saved to {METRICS_PATH}")

pred_out = pd.DataFrame({
    "y_true": y_true,
    "prob_1": probs,
    "pred": preds
})
pred_path = os.path.join(ROOT_DIR, "test_predictions.csv")
pred_out.to_csv(pred_path, index=False)
print(f"✅ Test predictions saved to {pred_path}")

save_model_package()

ALL_RESULTS[MODEL_NAME] = metrics

print("\n✅ SBERT cell complete.")
print(f"Elapsed time: {train_time/60:.1f} minutes")
print(f"Best checkpoint: {BEST_CKPT}")
print(f"Archive: {PACKAGE_ZIP}")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 8: BioBERT training + export (optimized, SBERT-style structure)
# Updated for stronger performance, safer artifact handling, and better T4 use
# ─────────────────────────────────────────────────────────────────────────────

!pip -q install transformers sentencepiece

import os
import re
import json
import time
import math
import random
import shutil
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.cuda.amp import autocast, GradScaler
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, precision_recall_curve,
    confusion_matrix, roc_curve, classification_report, log_loss
)
from sklearn.utils.class_weight import compute_class_weight
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup
)

# ── Reproducibility ─────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
try:
    torch.set_float32_matmul_precision("high")
except Exception:
    pass

os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# ── Paths ───────────────────────────────────────────────────────────────────
BASE_DIR = "/content"
PROCESSED_DIR = os.path.join(BASE_DIR, "processed_splits")
MODELS_DIR = globals().get("MODELS_DIR", os.path.join(BASE_DIR, "saved_models"))

MODEL_NAME = "BioBERT"
BACKBONE_NAME = "dmis-lab/biobert-base-cased-v1.2"

ROOT_DIR = os.path.join(MODELS_DIR, MODEL_NAME)
MODEL_DIR = os.path.join(ROOT_DIR, "model")
TOKENIZER_DIR = os.path.join(ROOT_DIR, "tokenizer")
RESULTS_DIR = os.path.join(ROOT_DIR, "results")
CACHE_DIR = os.path.join(ROOT_DIR, "cache")

BEST_CKPT = os.path.join(ROOT_DIR, "best_model.pt")
METRICS_PATH = os.path.join(ROOT_DIR, "metrics.json")
LABEL_MAP_PATH = os.path.join(ROOT_DIR, "label_map.json")
THRESH_PATH = os.path.join(ROOT_DIR, "threshold.json")
INFER_PATH = os.path.join(ROOT_DIR, "inference.py")
CONFIG_PATH = os.path.join(ROOT_DIR, "training_config.json")
PACKAGE_ZIP = os.path.join(MODELS_DIR, f"{MODEL_NAME}_package.zip")

for d in [ROOT_DIR, MODEL_DIR, TOKENIZER_DIR, RESULTS_DIR, CACHE_DIR]:
    os.makedirs(d, exist_ok=True)

PALETTE = ["#4C72B0", "#DD8452", "#55A868", "#C44E52", "#8172B2", "#937860"]

print(f"Root      : {ROOT_DIR}")
print(f"Model     : {MODEL_DIR}")
print(f"Tokenizer : {TOKENIZER_DIR}")
print(f"Results   : {RESULTS_DIR}")
print(f"Archive   : {PACKAGE_ZIP}")

ALL_RESULTS = globals().get("ALL_RESULTS", {})

# ── Data loading ────────────────────────────────────────────────────────────
train_df = pd.read_csv(os.path.join(PROCESSED_DIR, "train.csv"))
val_df   = pd.read_csv(os.path.join(PROCESSED_DIR, "val.csv"))
test_df  = pd.read_csv(os.path.join(PROCESSED_DIR, "test.csv"))

print(f"Loaded splits — train:{len(train_df)}  val:{len(val_df)}  test:{len(test_df)}")

def resolve_text_columns(df):
    abs_candidates = ["abstract_clean", "abstract_raw", "abstract_model", "abstract"]
    con_candidates = ["conclusion_clean", "conclusion_raw", "conclusion_model", "conclusion"]
    abs_col = next((c for c in abs_candidates if c in df.columns), None)
    con_col = next((c for c in con_candidates if c in df.columns), None)
    if abs_col is None or con_col is None:
        raise ValueError("Could not find abstract/conclusion columns in the split files.")
    return abs_col, con_col

ABS_COL, CONCL_COL = resolve_text_columns(train_df)
print(f"Using text columns: {ABS_COL}, {CONCL_COL}")

# ── Cleaning ────────────────────────────────────────────────────────────────
MAX_ABS_WORDS = 300
MAX_CONCL_WORDS = 300

# Stronger BioBERT settings for a T4, still usually safe with AMP.
MAX_LEN = 256
BATCH_SIZE = 16
GRAD_ACCUM = 2
LR = 2e-5
WEIGHT_DECAY = 0.01
N_EPOCHS = 12
PATIENCE = 3
FREEZE_LAYERS = 0
MAX_GRAD_NORM = 1.0
NUM_WORKERS = 2
WARMUP_RATIO = 0.10
LABEL_SMOOTHING = 0.01
USE_GRAD_CHECKPOINTING = False

PIPELINE_NAME = "biobert_pair_classifier"

def clean_text(text: str, max_words: int = None) -> str:
    if not isinstance(text, str) or not text.strip():
        return ""
    text = unicodedata.normalize("NFC", text)
    text = re.sub(r"\[\d[\d,;\s\-]*\]", " ", text)
    text = re.sub(r"\(\d[\d,;\s\-]*\)", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    if max_words and max_words > 0:
        words = text.split()
        if len(words) > max_words:
            text = " ".join(words[:max_words])
    return text

for split_name, split_df in [("train", train_df), ("val", val_df), ("test", test_df)]:
    split_df[ABS_COL] = split_df[ABS_COL].apply(lambda x: clean_text(x, MAX_ABS_WORDS))
    split_df[CONCL_COL] = split_df[CONCL_COL].apply(lambda x: clean_text(x, MAX_CONCL_WORDS))
    split_df = split_df.dropna(subset=["label"]).copy()
    split_df["label"] = split_df["label"].astype(int)

    before = len(split_df)
    split_df = split_df[(split_df[ABS_COL].str.len() > 0) & (split_df[CONCL_COL].str.len() > 0)].copy()
    split_df["abs_wc"] = split_df[ABS_COL].str.split().str.len()
    split_df["concl_wc"] = split_df[CONCL_COL].str.split().str.len()
    split_df = split_df[(split_df["abs_wc"] <= 1000) & (split_df["concl_wc"] <= 1000)].copy()

    if split_name == "train":
        train_df = split_df.reset_index(drop=True)
    elif split_name == "val":
        val_df = split_df.reset_index(drop=True)
    else:
        test_df = split_df.reset_index(drop=True)

    print(f"{split_name}: dropped {before - len(split_df):,} rows; remaining {len(split_df):,}")

for split_df in [train_df, val_df, test_df]:
    split_df["input_text_clean"] = "Abstract: " + split_df[ABS_COL] + " [SEP] Conclusion: " + split_df[CONCL_COL]

TOTAL_N = len(train_df) + len(val_df) + len(test_df)
TRAIN_FRAC = len(train_df) / TOTAL_N if TOTAL_N else 0
VAL_FRAC = len(val_df) / TOTAL_N if TOTAL_N else 0
TEST_FRAC = len(test_df) / TOTAL_N if TOTAL_N else 0

# ── Model / tokenizer ──────────────────────────────────────────────────────
print(f"Loading backbone: {BACKBONE_NAME}")
tokenizer = AutoTokenizer.from_pretrained(BACKBONE_NAME, use_fast=True)
model = AutoModelForSequenceClassification.from_pretrained(BACKBONE_NAME, num_labels=2)

try:
    model.config.problem_type = "single_label_classification"
except Exception:
    pass

try:
    model.config.use_cache = False
except Exception:
    pass

if USE_GRAD_CHECKPOINTING:
    try:
        model.gradient_checkpointing_enable()
        print("Gradient checkpointing enabled.")
    except Exception as e:
        print(f"Gradient checkpointing not available: {e}")

def freeze_lower_layers(mdl, n_layers):
    if n_layers <= 0:
        return

    backbone = None
    for attr in ("bert", "roberta", "electra", "deberta", "base_model"):
        if hasattr(mdl, attr):
            backbone = getattr(mdl, attr)
            break

    if backbone is None:
        return

    if hasattr(backbone, "embeddings"):
        for p in backbone.embeddings.parameters():
            p.requires_grad = False

    encoder = None
    if hasattr(backbone, "encoder"):
        encoder = backbone.encoder
    elif hasattr(backbone, "base_model") and hasattr(backbone.base_model, "encoder"):
        encoder = backbone.base_model.encoder

    if encoder is not None and hasattr(encoder, "layer"):
        for layer in encoder.layer[:n_layers]:
            for p in layer.parameters():
                p.requires_grad = False

freeze_lower_layers(model, FREEZE_LAYERS)
model.to(DEVICE)

try:
    model = torch.compile(model)
    print("torch.compile enabled")
except Exception:
    pass

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable params: {trainable:,} / {total:,}")

# ── Dataset / collator ─────────────────────────────────────────────────────
class PairDataset(Dataset):
    def __init__(self, df, abs_col, concl_col):
        self.abstracts = df[abs_col].astype(str).tolist()
        self.conclusions = df[concl_col].astype(str).tolist()
        self.labels = df["label"].astype(int).tolist()

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, i):
        return self.abstracts[i], self.conclusions[i], self.labels[i]

def collate(batch):
    abstracts, conclusions, labels = zip(*batch)
    enc = tokenizer(
        list(abstracts),
        list(conclusions),
        max_length=MAX_LEN,
        truncation=True,
        padding=True,
        return_tensors="pt",
        pad_to_multiple_of=8 if DEVICE.type == "cuda" else None,
    )
    enc["labels"] = torch.tensor(labels, dtype=torch.long)
    return enc

trn_ds = PairDataset(train_df, ABS_COL, CONCL_COL)
val_ds = PairDataset(val_df, ABS_COL, CONCL_COL)
tst_ds = PairDataset(test_df, ABS_COL, CONCL_COL)

loader_kwargs = dict(
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    pin_memory=(DEVICE.type == "cuda"),
    persistent_workers=(NUM_WORKERS > 0),
    collate_fn=collate,
)
if NUM_WORKERS > 0:
    loader_kwargs["prefetch_factor"] = 2

trn_loader = DataLoader(trn_ds, shuffle=True, **loader_kwargs)
val_loader = DataLoader(val_ds, shuffle=False, **loader_kwargs)
tst_loader = DataLoader(tst_ds, shuffle=False, **loader_kwargs)

# ── Loss / optimizer / scheduler ───────────────────────────────────────────
def get_class_weights(labels):
    labels = np.asarray(labels, dtype=int)
    classes = np.unique(labels)
    weights = compute_class_weight(class_weight="balanced", classes=classes, y=labels)
    out = np.ones(2, dtype=np.float32)
    for c, w in zip(classes, weights):
        out[int(c)] = float(w)
    return {0: out[0], 1: out[1]}

cw = get_class_weights(train_df["label"].tolist())
w_tensor = torch.tensor([cw[0], cw[1]], dtype=torch.float32, device=DEVICE)
print("Class weights:", cw)

criterion = nn.CrossEntropyLoss(weight=w_tensor, label_smoothing=LABEL_SMOOTHING)

optimizer_kwargs = dict(
    params=filter(lambda p: p.requires_grad, model.parameters()),
    lr=LR,
    weight_decay=WEIGHT_DECAY,
)

try:
    optimizer = torch.optim.AdamW(**optimizer_kwargs, fused=True)
    print("Using fused AdamW.")
except TypeError:
    optimizer = torch.optim.AdamW(**optimizer_kwargs)

num_update_steps_per_epoch = math.ceil(len(trn_loader) / GRAD_ACCUM)
total_steps = num_update_steps_per_epoch * N_EPOCHS
warmup_steps = int(WARMUP_RATIO * total_steps)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)

scaler = GradScaler(enabled=(DEVICE.type == "cuda"))

# ── Helpers ────────────────────────────────────────────────────────────────
def save_json(path, payload):
    with open(path, "w") as f:
        json.dump(payload, f, indent=2)

def best_threshold_for_f1(y_true, probs):
    thresholds = np.linspace(0.05, 0.95, 181)
    best_thr, best_f1 = 0.5, -1.0
    y_true = np.asarray(y_true, dtype=int)
    probs = np.asarray(probs, dtype=float)
    for thr in thresholds:
        preds = (probs >= thr).astype(int)
        f1 = f1_score(y_true, preds, average="macro", zero_division=0)
        if f1 > best_f1:
            best_f1 = f1
            best_thr = float(thr)
    return best_thr, float(best_f1)

def predict_loader(loader, threshold=0.5):
    model.eval()
    all_probs, all_preds, all_true = [], [], []
    total_loss, total_n = 0.0, 0

    with torch.no_grad():
        for batch in loader:
            labels = batch.pop("labels").to(DEVICE, non_blocking=True)
            batch = {k: v.to(DEVICE, non_blocking=True) for k, v in batch.items()}

            with autocast(enabled=(DEVICE.type == "cuda")):
                out = model(**batch)
                loss = criterion(out.logits, labels)

            probs = torch.softmax(out.logits, dim=-1)[:, 1]
            preds = (probs >= threshold).long()

            bs = labels.size(0)
            total_loss += loss.item() * bs
            total_n += bs

            all_probs.extend(probs.detach().cpu().tolist())
            all_preds.extend(preds.detach().cpu().tolist())
            all_true.extend(labels.detach().cpu().tolist())

    avg_loss = total_loss / max(total_n, 1)
    return all_true, all_probs, all_preds, avg_loss

def compute_hard_negative_accuracy(y_true, y_pred, abs_wc, concl_wc):
    y_true = np.asarray(y_true, dtype=int)
    y_pred = np.asarray(y_pred, dtype=int)
    abs_wc = np.asarray(abs_wc, dtype=int)
    concl_wc = np.asarray(concl_wc, dtype=int)

    mask = (y_true == 0) & (np.abs(abs_wc - concl_wc) < 50)
    idx = np.where(mask)[0]
    if len(idx) == 0:
        return {"hard_neg_count": 0, "hard_neg_accuracy": float("nan")}
    return {
        "hard_neg_count": int(len(idx)),
        "hard_neg_accuracy": float(accuracy_score(y_true[idx], y_pred[idx]))
    }

def compute_all_metrics(y_true, y_pred, y_prob):
    y_true = np.asarray(y_true, dtype=int)
    y_pred = np.asarray(y_pred, dtype=int)
    y_prob = np.asarray(y_prob, dtype=float)

    out = {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "precision_macro": float(precision_score(y_true, y_pred, average="macro", zero_division=0)),
        "recall_macro": float(recall_score(y_true, y_pred, average="macro", zero_division=0)),
        "f1_macro": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "precision_weighted": float(precision_score(y_true, y_pred, average="weighted", zero_division=0)),
        "recall_weighted": float(recall_score(y_true, y_pred, average="weighted", zero_division=0)),
        "f1_weighted": float(f1_score(y_true, y_pred, average="weighted", zero_division=0)),
    }

    report = classification_report(y_true, y_pred, output_dict=True, zero_division=0)
    if "0" in report:
        out["precision_class0"] = float(report["0"]["precision"])
        out["recall_class0"] = float(report["0"]["recall"])
        out["f1_class0"] = float(report["0"]["f1-score"])
    if "1" in report:
        out["precision_class1"] = float(report["1"]["precision"])
        out["recall_class1"] = float(report["1"]["recall"])
        out["f1_class1"] = float(report["1"]["f1-score"])

    out["roc_auc"] = float(roc_auc_score(y_true, y_prob)) if len(np.unique(y_true)) > 1 else float("nan")
    out["pr_auc"] = float(average_precision_score(y_true, y_prob))
    out["log_loss"] = float(log_loss(y_true, np.column_stack([1 - y_prob, y_prob])))
    return out

def plot_confusion_matrix(y_true, y_pred, model_name, save_dir):
    cm = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(5, 4))
    sns.heatmap(
        cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax,
        xticklabels=["Diff (0)", "Same (1)"],
        yticklabels=["Diff (0)", "Same (1)"]
    )
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_title(f"{model_name} — Confusion Matrix")
    path = os.path.join(save_dir, f"{model_name}_confusion_matrix.png")
    plt.tight_layout()
    plt.savefig(path, dpi=160, bbox_inches="tight")
    plt.show()
    plt.close()
    print(f"Saved → {path}")

def plot_roc_curve(y_true, y_prob, model_name, save_dir):
    if len(np.unique(y_true)) < 2:
        print("ROC curve skipped (only one class present).")
        return
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    auc_score = roc_auc_score(y_true, y_prob)
    fig, ax = plt.subplots(figsize=(5, 4))
    ax.plot(fpr, tpr, lw=2, label=f"AUC={auc_score:.3f}")
    ax.plot([0, 1], [0, 1], "k--", lw=1)
    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.set_title(f"{model_name} — ROC Curve")
    ax.legend(loc="lower right")
    path = os.path.join(save_dir, f"{model_name}_roc_curve.png")
    plt.tight_layout()
    plt.savefig(path, dpi=160, bbox_inches="tight")
    plt.show()
    plt.close()
    print(f"Saved → {path}")

def plot_pr_curve(y_true, y_prob, model_name, save_dir):
    precision, recall, _ = precision_recall_curve(y_true, y_prob)
    ap = average_precision_score(y_true, y_prob)
    fig, ax = plt.subplots(figsize=(5, 4))
    ax.plot(recall, precision, lw=2, label=f"AP={ap:.3f}")
    ax.set_xlabel("Recall")
    ax.set_ylabel("Precision")
    ax.set_title(f"{model_name} — Precision-Recall Curve")
    ax.legend(loc="upper right")
    path = os.path.join(save_dir, f"{model_name}_pr_curve.png")
    plt.tight_layout()
    plt.savefig(path, dpi=160, bbox_inches="tight")
    plt.show()
    plt.close()
    print(f"Saved → {path}")

def plot_training_curves(train_losses, val_losses, train_accs, val_accs, model_name, save_dir):
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    epochs = range(1, len(train_losses) + 1)

    axes[0].plot(epochs, train_losses, "o-", label="Train")
    if val_losses:
        axes[0].plot(epochs, val_losses, "s--", label="Val")
    axes[0].set_title(f"{model_name} — Loss")
    axes[0].set_xlabel("Epoch")
    axes[0].legend()

    axes[1].plot(epochs, train_accs, "o-", label="Train")
    if val_accs:
        axes[1].plot(epochs, val_accs, "s--", label="Val")
    axes[1].set_title(f"{model_name} — Accuracy")
    axes[1].set_xlabel("Epoch")
    axes[1].legend()

    path = os.path.join(save_dir, f"{model_name}_training_curves.png")
    plt.tight_layout()
    plt.savefig(path, dpi=160, bbox_inches="tight")
    plt.show()
    plt.close()
    print(f"Saved → {path}")

def plot_subdomain_performance(y_true, y_pred, subdomains, model_name, save_dir):
    df_e = pd.DataFrame({"true": y_true, "pred": y_pred, "subdomain": subdomains})
    grp = df_e.groupby("subdomain").apply(
        lambda g: accuracy_score(g["true"], g["pred"]) if len(g) >= 3 else np.nan
    ).dropna().sort_values(ascending=False)

    if grp.empty:
        print("Not enough samples for subdomain plot.")
        return

    fig, ax = plt.subplots(figsize=(max(8, len(grp) * 0.75), 4))
    grp.plot(kind="bar", ax=ax, color=PALETTE[0])
    ax.axhline(accuracy_score(y_true, y_pred), color="red", ls="--", label="Overall")
    ax.set_ylabel("Accuracy")
    ax.set_title(f"{model_name} — Subdomain Accuracy")
    ax.legend()
    plt.xticks(rotation=45, ha="right")
    path = os.path.join(save_dir, f"{model_name}_subdomain_performance.png")
    plt.tight_layout()
    plt.savefig(path, dpi=160, bbox_inches="tight")
    plt.show()
    plt.close()
    print(f"Saved → {path}")

def plot_length_bucket_performance(y_true, y_pred, lengths, model_name, save_dir):
    lengths = np.asarray(lengths)
    bins = [0, 100, 200, 300, 400, 600, 100000]
    labels = ["<100", "100-200", "200-300", "300-400", "400-600", "600+"]
    buckets = pd.cut(lengths, bins=bins, labels=labels, right=False)

    df_e = pd.DataFrame({"true": y_true, "pred": y_pred, "bucket": buckets})
    grp = df_e.groupby("bucket", observed=True).apply(
        lambda g: accuracy_score(g["true"], g["pred"]) if len(g) >= 3 else np.nan
    ).dropna()

    if grp.empty:
        print("Not enough samples for length-bucket plot.")
        return

    fig, ax = plt.subplots(figsize=(8, 4))
    grp.plot(kind="bar", ax=ax, color=PALETTE[1])
    ax.set_ylim(0, 1)
    ax.set_ylabel("Accuracy")
    ax.set_title(f"{model_name} — Accuracy by Length Bucket")
    ax.set_xlabel("Word-count bucket")
    plt.xticks(rotation=30, ha="right")
    path = os.path.join(save_dir, f"{model_name}_length_bucket.png")
    plt.tight_layout()
    plt.savefig(path, dpi=160, bbox_inches="tight")
    plt.show()
    plt.close()
    print(f"Saved → {path}")

def save_label_map(root_dir):
    label_map = {"0": "Different subdomain", "1": "Same subdomain"}
    save_json(os.path.join(root_dir, "label_map.json"), label_map)

def save_threshold(root_dir, threshold, val_f1):
    save_json(os.path.join(root_dir, "threshold.json"), {
        "threshold": float(threshold),
        "val_f1_macro": float(val_f1)
    })

def save_training_config(root_dir):
    cfg = {
        "pipeline_name": PIPELINE_NAME,
        "model_name": MODEL_NAME,
        "backbone": BACKBONE_NAME,
        "max_abs_words": MAX_ABS_WORDS,
        "max_concl_words": MAX_CONCL_WORDS,
        "max_len": MAX_LEN,
        "seed": SEED,
        "batch_size": BATCH_SIZE,
        "grad_accum": GRAD_ACCUM,
        "lr": LR,
        "weight_decay": WEIGHT_DECAY,
        "epochs": N_EPOCHS,
        "patience": PATIENCE,
        "freeze_layers": FREEZE_LAYERS,
        "max_grad_norm": MAX_GRAD_NORM,
        "label_smoothing": LABEL_SMOOTHING,
        "warmup_ratio": WARMUP_RATIO,
        "gradient_checkpointing": USE_GRAD_CHECKPOINTING,
        "num_workers": NUM_WORKERS,
    }
    save_json(os.path.join(root_dir, "training_config.json"), cfg)

def write_inference_script():
    inference_code = f'''import json
import os
import re
import unicodedata
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

ROOT_DIR = os.path.dirname(os.path.abspath(__file__))
MODEL_DIR = os.path.join(ROOT_DIR, "model")
TOKENIZER_DIR = os.path.join(ROOT_DIR, "tokenizer")
THRESH_PATH = os.path.join(ROOT_DIR, "threshold.json")
LABEL_MAP_PATH = os.path.join(ROOT_DIR, "label_map.json")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def load_json(path, default):
    if os.path.exists(path):
        with open(path, "r") as f:
            return json.load(f)
    return default

def normalize_text(text):
    if text is None:
        return ""
    text = unicodedata.normalize("NFC", str(text))
    text = re.sub(r"\\[\\d[\\d,;\\s\\-]*\\]", " ", text)
    text = re.sub(r"\\(\\d[\\d,;\\s\\-]*\\)", " ", text)
    text = re.sub(r"\\s+", " ", text).strip()
    return text

def clean_text(text, max_words=300):
    text = normalize_text(text)
    if not text:
        return ""
    return " ".join(text.split()[:max_words])

tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_DIR, use_fast=True)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR).to(DEVICE)
model.eval()

thr_payload = load_json(THRESH_PATH, {{"threshold": 0.5}})
threshold = float(thr_payload.get("threshold", 0.5))
label_map = load_json(LABEL_MAP_PATH, {{"0": "Different subdomain", "1": "Same subdomain"}})

def predict_pair(abstract, conclusion, max_len={MAX_LEN}):
    abstract = clean_text(abstract)
    conclusion = clean_text(conclusion)
    enc = tokenizer(
        abstract,
        conclusion,
        max_length=max_len,
        truncation=True,
        padding="max_length",
        return_tensors="pt"
    )
    enc = {{k: v.to(DEVICE) for k, v in enc.items()}}
    with torch.no_grad():
        logits = model(**enc).logits
        prob_1 = torch.softmax(logits, dim=-1)[0, 1].item()
    pred = int(prob_1 >= threshold)
    return {{
        "label": pred,
        "label_name": label_map.get(str(pred), str(pred)),
        "probability_same_subdomain": prob_1,
        "threshold": threshold
    }}

if __name__ == "__main__":
    print(predict_pair("Sample abstract.", "Sample conclusion."))
'''
    with open(INFER_PATH, "w") as f:
        f.write(inference_code)
    print(f"✅ Inference script saved to {INFER_PATH}")

def zip_model_package():
    zip_base = os.path.join(MODELS_DIR, f"{MODEL_NAME}_package")
    zip_path = shutil.make_archive(zip_base, "zip", root_dir=MODELS_DIR, base_dir=MODEL_NAME)
    print(f"📦 Package archived → {zip_path}")
    return zip_path

# ── Training ───────────────────────────────────────────────────────────────
best_val_f1 = 0.0
best_val_threshold = 0.5
patience_ctr = 0
train_losses, val_losses, train_accs, val_accs = [], [], [], []
best_ckpt_path = BEST_CKPT

t0 = time.time()
print("\nTraining BioBERT …")

for epoch in range(1, N_EPOCHS + 1):
    model.train()
    ep_loss, ep_correct, ep_total = 0.0, 0, 0
    optimizer.zero_grad(set_to_none=True)

    for step, batch in enumerate(trn_loader, start=1):
        labels = batch.pop("labels").to(DEVICE, non_blocking=True)
        batch = {k: v.to(DEVICE, non_blocking=True) for k, v in batch.items()}

        with autocast(enabled=(DEVICE.type == "cuda")):
            out = model(**batch)
            loss = criterion(out.logits, labels) / GRAD_ACCUM

        scaler.scale(loss).backward()

        ep_loss += loss.item() * GRAD_ACCUM
        preds = out.logits.argmax(dim=-1)
        ep_correct += (preds == labels).sum().item()
        ep_total += labels.size(0)

        do_step = (step % GRAD_ACCUM == 0) or (step == len(trn_loader))
        if do_step:
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            scheduler.step()

    train_loss = ep_loss / max(len(trn_loader), 1)
    train_acc = ep_correct / max(ep_total, 1)
    train_losses.append(train_loss)
    train_accs.append(train_acc)

    y_val, p_val, pred_val_05, v_loss = predict_loader(val_loader, threshold=0.5)
    val_acc_05 = accuracy_score(y_val, pred_val_05)
    val_f1_05 = f1_score(y_val, pred_val_05, average="macro", zero_division=0)

    val_thr, val_f1_thr = best_threshold_for_f1(y_val, p_val)
    val_preds_best = (np.asarray(p_val) >= val_thr).astype(int)
    val_acc_best = accuracy_score(y_val, val_preds_best)

    val_losses.append(v_loss)
    val_accs.append(val_acc_best)

    if val_f1_thr > best_val_f1:
        best_val_f1 = val_f1_thr
        best_val_threshold = val_thr
        patience_ctr = 0
        torch.save(model.state_dict(), best_ckpt_path)
    else:
        patience_ctr += 1

    print(
        f"Epoch {epoch:02d} | "
        f"train_loss={train_loss:.4f} | train_acc={train_acc:.3f} | "
        f"val_loss={v_loss:.4f} | val_acc@0.5={val_acc_05:.3f} | "
        f"val_f1@0.5={val_f1_05:.4f} | val_f1@best_thr={val_f1_thr:.4f} | "
        f"thr={val_thr:.3f} | patience={patience_ctr}"
    )

    if patience_ctr >= PATIENCE:
        print(f"Early stopping at epoch {epoch}.")
        break

train_time = time.time() - t0
print(f"\nTraining time: {train_time:.1f} seconds")
print(f"Best validation threshold: {best_val_threshold:.3f}")

# ── Load best checkpoint ───────────────────────────────────────────────────
model.load_state_dict(torch.load(best_ckpt_path, map_location=DEVICE))
model.eval()

# ── Test evaluation ────────────────────────────────────────────────────────
y_true, probs, raw_preds, _ = predict_loader(tst_loader, threshold=best_val_threshold)
preds = (np.asarray(probs) >= best_val_threshold).astype(int)

report = classification_report(y_true, preds, output_dict=True, zero_division=0)
metrics = {
    "accuracy": float(accuracy_score(y_true, preds)),
    "precision_macro": float(precision_score(y_true, preds, average="macro", zero_division=0)),
    "recall_macro": float(recall_score(y_true, preds, average="macro", zero_division=0)),
    "f1_macro": float(f1_score(y_true, preds, average="macro", zero_division=0)),
    "precision_weighted": float(precision_score(y_true, preds, average="weighted", zero_division=0)),
    "recall_weighted": float(recall_score(y_true, preds, average="weighted", zero_division=0)),
    "f1_weighted": float(f1_score(y_true, preds, average="weighted", zero_division=0)),
    "precision_class0": float(report["0"]["precision"]),
    "recall_class0": float(report["0"]["recall"]),
    "f1_class0": float(report["0"]["f1-score"]),
    "precision_class1": float(report["1"]["precision"]),
    "recall_class1": float(report["1"]["recall"]),
    "f1_class1": float(report["1"]["f1-score"]),
    "roc_auc": float(roc_auc_score(y_true, probs)) if len(np.unique(y_true)) > 1 else float("nan"),
    "pr_auc": float(average_precision_score(y_true, probs)),
    "log_loss": float(log_loss(y_true, np.column_stack([1 - np.array(probs), probs]))),
    "training_time_s": float(train_time),
    "best_val_threshold": float(best_val_threshold),
    "best_val_f1_macro": float(best_val_f1),
}
metrics.update(compute_hard_negative_accuracy(
    y_true, preds,
    test_df["abs_wc"].tolist(),
    test_df["concl_wc"].tolist()
))

print("\nFinal metrics:")
for k, v in metrics.items():
    try:
        print(f"{k:<30s}: {v:.4f}")
    except Exception:
        print(f"{k:<30s}: {v}")

# ── Plots ──────────────────────────────────────────────────────────────────
plot_confusion_matrix(y_true, preds, MODEL_NAME, RESULTS_DIR)
plot_roc_curve(y_true, probs, MODEL_NAME, RESULTS_DIR)
plot_pr_curve(y_true, probs, MODEL_NAME, RESULTS_DIR)
plot_training_curves(train_losses, val_losses, train_accs, val_accs, MODEL_NAME, RESULTS_DIR)

if "abstract_subdomain" in test_df.columns:
    plot_subdomain_performance(
        y_true, preds,
        test_df["abstract_subdomain"].fillna("unknown").astype(str).tolist(),
        MODEL_NAME, RESULTS_DIR
    )

plot_length_bucket_performance(
    y_true, preds,
    (test_df["abs_wc"].fillna(0) + test_df["concl_wc"].fillna(0)).tolist(),
    MODEL_NAME, RESULTS_DIR
)

# ── Save artifacts ──────────────────────────────────────────────────────────
save_json(METRICS_PATH, {k: (float(v) if isinstance(v, (np.floating, float)) and np.isfinite(v) else v)
                         for k, v in metrics.items()})
print(f"✅ Metrics saved to {METRICS_PATH}")

pred_out = pd.DataFrame({
    "y_true": y_true,
    "prob_1": probs,
    "pred": preds
})
pred_path = os.path.join(ROOT_DIR, "test_predictions.csv")
pred_out.to_csv(pred_path, index=False)
print(f"✅ Test predictions saved to {pred_path}")

save_label_map(ROOT_DIR)
save_threshold(ROOT_DIR, best_val_threshold, best_val_f1)
save_training_config(ROOT_DIR)
write_inference_script()

model.save_pretrained(MODEL_DIR)
tokenizer.save_pretrained(TOKENIZER_DIR)
print(f"✅ Saved model to {MODEL_DIR}")
print(f"✅ Saved tokenizer to {TOKENIZER_DIR}")

zip_model_package()
ALL_RESULTS[MODEL_NAME] = metrics

print("\n✅ BioBERT cell complete.")
print(f"Elapsed time: {train_time/60:.1f} minutes")
print(f"Best checkpoint: {BEST_CKPT}")
print(f"Archive: {PACKAGE_ZIP}")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 9: Longformer training + export (fast Longformer variant)
# Optimized to finish faster on T4 while preserving Longformer long-context use
# ─────────────────────────────────────────────────────────────────────────────

!pip -q install transformers sentencepiece

import os
import re
import json
import time
import math
import random
import shutil
import unicodedata
import gc
from collections import defaultdict

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.cuda.amp import autocast, GradScaler
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, precision_recall_curve, average_precision_score,
    confusion_matrix, roc_curve, classification_report, log_loss
)
from sklearn.utils.class_weight import compute_class_weight
from transformers import (
    LongformerTokenizerFast,
    LongformerForSequenceClassification,
    get_linear_schedule_with_warmup
)

# ── Reproducibility ─────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
try:
    torch.set_float32_matmul_precision("high")
except Exception:
    pass

os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# ── Paths ───────────────────────────────────────────────────────────────────
BASE_DIR = "/content"
PROCESSED_DIR = os.path.join(BASE_DIR, "processed_splits")
MODELS_DIR = globals().get("MODELS_DIR", os.path.join(BASE_DIR, "saved_models"))

MODEL_NAME = "Longformer"
BACKBONE_NAME = "allenai/longformer-base-4096"

ROOT_DIR = os.path.join(MODELS_DIR, MODEL_NAME)
MODEL_DIR = os.path.join(ROOT_DIR, "model")
TOKENIZER_DIR = os.path.join(ROOT_DIR, "tokenizer")
RESULTS_DIR = os.path.join(ROOT_DIR, "results")
CACHE_DIR = os.path.join(ROOT_DIR, "cache")

BEST_CKPT = os.path.join(ROOT_DIR, "best_model.pt")
METRICS_PATH = os.path.join(ROOT_DIR, "metrics.json")
LABEL_MAP_PATH = os.path.join(ROOT_DIR, "label_map.json")
THRESH_PATH = os.path.join(ROOT_DIR, "threshold.json")
INFER_PATH = os.path.join(ROOT_DIR, "inference.py")
CONFIG_PATH = os.path.join(ROOT_DIR, "training_config.json")
PACKAGE_ZIP = os.path.join(MODELS_DIR, f"{MODEL_NAME}_package.zip")

for d in [ROOT_DIR, MODEL_DIR, TOKENIZER_DIR, RESULTS_DIR, CACHE_DIR]:
    os.makedirs(d, exist_ok=True)

ALL_RESULTS = globals().get("ALL_RESULTS", {})

print(f"Root      : {ROOT_DIR}")
print(f"Model     : {MODEL_DIR}")
print(f"Tokenizer : {TOKENIZER_DIR}")
print(f"Results   : {RESULTS_DIR}")
print(f"Archive   : {PACKAGE_ZIP}")

# ── Data loading ───────────────────────────────────────────────────────────
train_df = pd.read_csv(os.path.join(PROCESSED_DIR, "train.csv"))
val_df   = pd.read_csv(os.path.join(PROCESSED_DIR, "val.csv"))
test_df  = pd.read_csv(os.path.join(PROCESSED_DIR, "test.csv"))

print(f"Loaded splits — train:{len(train_df)}  val:{len(val_df)}  test:{len(test_df)}")

def resolve_text_columns(df):
    abs_candidates = ["abstract_clean", "abstract_raw", "abstract_model", "abstract"]
    con_candidates = ["conclusion_clean", "conclusion_raw", "conclusion_model", "conclusion"]
    abs_col = next((c for c in abs_candidates if c in df.columns), None)
    con_col = next((c for c in con_candidates if c in df.columns), None)
    if abs_col is None or con_col is None:
        raise ValueError("Could not find abstract/conclusion columns in the split files.")
    return abs_col, con_col

ABS_COL, CONCL_COL = resolve_text_columns(train_df)
print(f"Using text columns: {ABS_COL}, {CONCL_COL}")

# ── Cleaning ────────────────────────────────────────────────────────────────
# Smaller word caps help the tokenizer and training speed; Longformer still
# keeps more context than standard BERT-style models.
MAX_ABS_WORDS = 220
MAX_CONCL_WORDS = 220

# Faster and memory-safe Longformer length for T4.
# Increase to 512 only if you have room and can afford the runtime.
MAX_LEN = 384

# Faster training target
BATCH_SIZE = 4
GRAD_ACCUM = 4          # effective batch size = 16
LR = 1.5e-5
WEIGHT_DECAY = 0.01
N_EPOCHS = 6
PATIENCE = 2
FREEZE_LAYERS = 10      # leave only the top layers trainable
MAX_GRAD_NORM = 1.0
LABEL_SMOOTHING = 0.02
NUM_WORKERS = 2
WARMUP_RATIO = 0.08
USE_GRAD_CHECKPOINTING = False   # faster; enable only if you hit memory limits

PIPELINE_NAME = "longformer_pair_classifier_fast"

def clean_text(text: str, max_words: int = None) -> str:
    if not isinstance(text, str) or not text.strip():
        return ""
    text = unicodedata.normalize("NFC", text)
    text = re.sub(r"\[\d[\d,;\s\-]*\]", " ", text)
    text = re.sub(r"\(\d[\d,;\s\-]*\)", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    if max_words and max_words > 0:
        words = text.split()
        if len(words) > max_words:
            text = " ".join(words[:max_words])
    return text

for split_name, split_df in [("train", train_df), ("val", val_df), ("test", test_df)]:
    split_df[ABS_COL] = split_df[ABS_COL].apply(lambda x: clean_text(x, MAX_ABS_WORDS))
    split_df[CONCL_COL] = split_df[CONCL_COL].apply(lambda x: clean_text(x, MAX_CONCL_WORDS))
    split_df = split_df.dropna(subset=["label"]).copy()
    split_df["label"] = split_df["label"].astype(int)

    before = len(split_df)
    split_df = split_df[(split_df[ABS_COL].str.len() > 0) & (split_df[CONCL_COL].str.len() > 0)].copy()
    split_df["abs_wc"] = split_df[ABS_COL].str.split().str.len()
    split_df["concl_wc"] = split_df[CONCL_COL].str.split().str.len()
    split_df = split_df[(split_df["abs_wc"] <= 1000) & (split_df["concl_wc"] <= 1000)].copy()

    if split_name == "train":
        train_df = split_df.reset_index(drop=True)
    elif split_name == "val":
        val_df = split_df.reset_index(drop=True)
    else:
        test_df = split_df.reset_index(drop=True)

    print(f"{split_name}: dropped {before - len(split_df):,} rows; remaining {len(split_df):,}")

for split_df in [train_df, val_df, test_df]:
    split_df["input_text_clean"] = "Abstract: " + split_df[ABS_COL] + " [SEP] Conclusion: " + split_df[CONCL_COL]

TOTAL_N = len(train_df) + len(val_df) + len(test_df)
TRAIN_FRAC = len(train_df) / TOTAL_N if TOTAL_N else 0
VAL_FRAC = len(val_df) / TOTAL_N if TOTAL_N else 0
TEST_FRAC = len(test_df) / TOTAL_N if TOTAL_N else 0

# ── Model / tokenizer ──────────────────────────────────────────────────────
print(f"Loading backbone: {BACKBONE_NAME}")
tokenizer = LongformerTokenizerFast.from_pretrained(BACKBONE_NAME)
model = LongformerForSequenceClassification.from_pretrained(BACKBONE_NAME, num_labels=2)

try:
    model.config.use_cache = False
except Exception:
    pass

if USE_GRAD_CHECKPOINTING:
    try:
        model.gradient_checkpointing_enable()
        print("Gradient checkpointing enabled.")
    except Exception as e:
        print(f"Gradient checkpointing not available: {e}")

def freeze_lower_layers(mdl, n_layers):
    if n_layers <= 0:
        return

    backbone = getattr(mdl, "longformer", None)
    if backbone is None:
        backbone = getattr(mdl, "base_model", None)

    if backbone is None:
        return

    if hasattr(backbone, "embeddings"):
        for p in backbone.embeddings.parameters():
            p.requires_grad = False

    encoder = None
    if hasattr(backbone, "encoder"):
        encoder = backbone.encoder
    elif hasattr(backbone, "base_model") and hasattr(backbone.base_model, "encoder"):
        encoder = backbone.base_model.encoder

    if encoder is not None and hasattr(encoder, "layer"):
        for layer in encoder.layer[:n_layers]:
            for p in layer.parameters():
                p.requires_grad = False

freeze_lower_layers(model, FREEZE_LAYERS)
model.to(DEVICE)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable params: {trainable:,} / {total:,}")

if DEVICE.type == "cuda":
    torch.cuda.empty_cache()
gc.collect()

# ── Dataset / collator ─────────────────────────────────────────────────────
class PairDataset(Dataset):
    def __init__(self, df, abs_col, concl_col):
        self.abstracts = df[abs_col].astype(str).tolist()
        self.conclusions = df[concl_col].astype(str).tolist()
        self.labels = df["label"].astype(int).tolist()

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, i):
        return self.abstracts[i], self.conclusions[i], self.labels[i]

def collate_fn(batch):
    abstracts, conclusions, labels = zip(*batch)
    enc = tokenizer(
        list(abstracts),
        list(conclusions),
        max_length=MAX_LEN,
        truncation=True,
        padding=True,
        return_tensors="pt",
        pad_to_multiple_of=8 if DEVICE.type == "cuda" else None,
    )
    global_attention_mask = torch.zeros_like(enc["input_ids"])
    global_attention_mask[:, 0] = 1
    enc["global_attention_mask"] = global_attention_mask
    enc["labels"] = torch.tensor(labels, dtype=torch.long)
    return enc

trn_ds = PairDataset(train_df, ABS_COL, CONCL_COL)
val_ds = PairDataset(val_df, ABS_COL, CONCL_COL)
tst_ds = PairDataset(test_df, ABS_COL, CONCL_COL)

loader_kwargs = dict(
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    pin_memory=(DEVICE.type == "cuda"),
    persistent_workers=(NUM_WORKERS > 0),
    collate_fn=collate_fn,
)
if NUM_WORKERS > 0:
    loader_kwargs["prefetch_factor"] = 2

trn_loader = DataLoader(trn_ds, shuffle=True, **loader_kwargs)
val_loader = DataLoader(val_ds, shuffle=False, **loader_kwargs)
tst_loader = DataLoader(tst_ds, shuffle=False, **loader_kwargs)

# ── Loss / optimizer / scheduler ───────────────────────────────────────────
def get_class_weights(labels):
    labels = np.asarray(labels, dtype=int)
    classes = np.unique(labels)
    weights = compute_class_weight(class_weight="balanced", classes=classes, y=labels)
    out = np.ones(2, dtype=np.float32)
    for c, w in zip(classes, weights):
        out[int(c)] = float(w)
    return {0: out[0], 1: out[1]}

cw = get_class_weights(train_df["label"].tolist())
w_tensor = torch.tensor([cw[0], cw[1]], dtype=torch.float32, device=DEVICE)
print("Class weights:", cw)

criterion = nn.CrossEntropyLoss(weight=w_tensor, label_smoothing=LABEL_SMOOTHING)

try:
    optimizer = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=LR,
        weight_decay=WEIGHT_DECAY,
        fused=True
    )
    print("Using fused AdamW.")
except TypeError:
    optimizer = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=LR,
        weight_decay=WEIGHT_DECAY
    )

num_update_steps_per_epoch = math.ceil(len(trn_loader) / GRAD_ACCUM)
total_steps = num_update_steps_per_epoch * N_EPOCHS
warmup_steps = int(WARMUP_RATIO * total_steps)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)

scaler = GradScaler(enabled=(DEVICE.type == "cuda"))

# ── Helpers ────────────────────────────────────────────────────────────────
def save_json(path, payload):
    with open(path, "w") as f:
        json.dump(payload, f, indent=2)

def best_threshold_for_f1(y_true, probs):
    thresholds = np.linspace(0.05, 0.95, 181)
    best_thr, best_f1 = 0.5, -1.0
    y_true = np.asarray(y_true, dtype=int)
    probs = np.asarray(probs, dtype=float)
    for thr in thresholds:
        preds = (probs >= thr).astype(int)
        f1 = f1_score(y_true, preds, average="macro", zero_division=0)
        if f1 > best_f1:
            best_f1 = f1
            best_thr = float(thr)
    return best_thr, float(best_f1)

def predict_loader(loader, threshold=0.5):
    model.eval()
    all_true, all_probs, all_preds = [], [], []
    total_loss, total_n = 0.0, 0

    with torch.no_grad():
        for batch in loader:
            labels = batch.pop("labels").to(DEVICE, non_blocking=True)
            batch = {k: v.to(DEVICE, non_blocking=True) for k, v in batch.items()}

            with autocast(enabled=(DEVICE.type == "cuda")):
                out = model(**batch)
                loss = criterion(out.logits, labels)

            probs = torch.softmax(out.logits, dim=-1)[:, 1]
            preds = (probs >= threshold).long()

            bs = labels.size(0)
            total_loss += loss.item() * bs
            total_n += bs

            all_true.extend(labels.detach().cpu().tolist())
            all_probs.extend(probs.detach().cpu().tolist())
            all_preds.extend(preds.detach().cpu().tolist())

    return all_true, all_probs, all_preds, total_loss / max(total_n, 1)

def compute_hard_negative_accuracy(y_true, y_pred, abs_wc, concl_wc):
    y_true = np.asarray(y_true, dtype=int)
    y_pred = np.asarray(y_pred, dtype=int)
    abs_wc = np.asarray(abs_wc, dtype=int)
    concl_wc = np.asarray(concl_wc, dtype=int)

    mask = (y_true == 0) & (np.abs(abs_wc - concl_wc) < 50)
    idx = np.where(mask)[0]
    if len(idx) == 0:
        return {"hard_neg_count": 0, "hard_neg_accuracy": float("nan")}
    return {
        "hard_neg_count": int(len(idx)),
        "hard_neg_accuracy": float(accuracy_score(y_true[idx], y_pred[idx]))
    }

def compute_all_metrics(y_true, y_pred, y_prob):
    y_true = np.asarray(y_true, dtype=int)
    y_pred = np.asarray(y_pred, dtype=int)
    y_prob = np.asarray(y_prob, dtype=float)

    metrics = {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "precision_macro": float(precision_score(y_true, y_pred, average="macro", zero_division=0)),
        "recall_macro": float(recall_score(y_true, y_pred, average="macro", zero_division=0)),
        "f1_macro": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "precision_weighted": float(precision_score(y_true, y_pred, average="weighted", zero_division=0)),
        "recall_weighted": float(recall_score(y_true, y_pred, average="weighted", zero_division=0)),
        "f1_weighted": float(f1_score(y_true, y_pred, average="weighted", zero_division=0)),
    }

    rep = classification_report(y_true, y_pred, output_dict=True, zero_division=0)
    if "0" in rep:
        metrics["precision_class0"] = float(rep["0"]["precision"])
        metrics["recall_class0"] = float(rep["0"]["recall"])
        metrics["f1_class0"] = float(rep["0"]["f1-score"])
    if "1" in rep:
        metrics["precision_class1"] = float(rep["1"]["precision"])
        metrics["recall_class1"] = float(rep["1"]["recall"])
        metrics["f1_class1"] = float(rep["1"]["f1-score"])

    metrics["roc_auc"] = float(roc_auc_score(y_true, y_prob)) if len(np.unique(y_true)) > 1 else float("nan")
    metrics["pr_auc"] = float(average_precision_score(y_true, y_prob))
    metrics["log_loss"] = float(log_loss(y_true, np.column_stack([1 - y_prob, y_prob])))
    return metrics

def plot_confusion_matrix(y_true, y_pred, model_name, save_dir):
    cm = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(5, 4))
    sns.heatmap(
        cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax,
        xticklabels=["Diff (0)", "Same (1)"],
        yticklabels=["Diff (0)", "Same (1)"]
    )
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_title(f"{model_name} — Confusion Matrix")
    path = os.path.join(save_dir, f"{model_name}_confusion_matrix.png")
    plt.tight_layout()
    plt.savefig(path, dpi=160, bbox_inches="tight")
    plt.show()
    plt.close()
    print(f"Saved → {path}")

def plot_roc_curve(y_true, y_prob, model_name, save_dir):
    if len(np.unique(y_true)) < 2:
        print("ROC curve skipped (only one class present).")
        return
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    auc_score = roc_auc_score(y_true, y_prob)
    fig, ax = plt.subplots(figsize=(5, 4))
    ax.plot(fpr, tpr, lw=2, label=f"AUC={auc_score:.3f}")
    ax.plot([0, 1], [0, 1], "k--", lw=1)
    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.set_title(f"{model_name} — ROC Curve")
    ax.legend(loc="lower right")
    path = os.path.join(save_dir, f"{model_name}_roc_curve.png")
    plt.tight_layout()
    plt.savefig(path, dpi=160, bbox_inches="tight")
    plt.show()
    plt.close()
    print(f"Saved → {path}")

def plot_pr_curve(y_true, y_prob, model_name, save_dir):
    precision, recall, _ = precision_recall_curve(y_true, y_prob)
    ap = average_precision_score(y_true, y_prob)
    fig, ax = plt.subplots(figsize=(5, 4))
    ax.plot(recall, precision, lw=2, label=f"AP={ap:.3f}")
    ax.set_xlabel("Recall")
    ax.set_ylabel("Precision")
    ax.set_title(f"{model_name} — Precision-Recall Curve")
    ax.legend(loc="upper right")
    path = os.path.join(save_dir, f"{model_name}_pr_curve.png")
    plt.tight_layout()
    plt.savefig(path, dpi=160, bbox_inches="tight")
    plt.show()
    plt.close()
    print(f"Saved → {path}")

def plot_training_curves(train_losses, val_losses, train_accs, val_accs, model_name, save_dir):
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    epochs = range(1, len(train_losses) + 1)

    axes[0].plot(epochs, train_losses, "o-", label="Train")
    if val_losses:
        axes[0].plot(epochs, val_losses, "s--", label="Val")
    axes[0].set_title(f"{model_name} — Loss")
    axes[0].set_xlabel("Epoch")
    axes[0].legend()

    axes[1].plot(epochs, train_accs, "o-", label="Train")
    if val_accs:
        axes[1].plot(epochs, val_accs, "s--", label="Val")
    axes[1].set_title(f"{model_name} — Accuracy")
    axes[1].set_xlabel("Epoch")
    axes[1].legend()

    path = os.path.join(save_dir, f"{model_name}_training_curves.png")
    plt.tight_layout()
    plt.savefig(path, dpi=160, bbox_inches="tight")
    plt.show()
    plt.close()
    print(f"Saved → {path}")

def plot_subdomain_performance(y_true, y_pred, subdomains, model_name, save_dir):
    df_e = pd.DataFrame({"true": y_true, "pred": y_pred, "subdomain": subdomains})
    grp = df_e.groupby("subdomain").apply(
        lambda g: accuracy_score(g["true"], g["pred"]) if len(g) >= 3 else np.nan
    ).dropna().sort_values(ascending=False)

    if grp.empty:
        print("Not enough samples for subdomain plot.")
        return

    fig, ax = plt.subplots(figsize=(max(8, len(grp) * 0.75), 4))
    grp.plot(kind="bar", ax=ax)
    ax.axhline(accuracy_score(y_true, y_pred), color="red", ls="--", label="Overall")
    ax.set_ylabel("Accuracy")
    ax.set_title(f"{model_name} — Subdomain Accuracy")
    ax.legend()
    plt.xticks(rotation=45, ha="right")
    path = os.path.join(save_dir, f"{model_name}_subdomain_performance.png")
    plt.tight_layout()
    plt.savefig(path, dpi=160, bbox_inches="tight")
    plt.show()
    plt.close()
    print(f"Saved → {path}")

def plot_length_bucket_performance(y_true, y_pred, lengths, model_name, save_dir):
    lengths = np.asarray(lengths)
    bins = [0, 100, 200, 300, 400, 600, 100000]
    labels = ["<100", "100-200", "200-300", "300-400", "400-600", "600+"]
    buckets = pd.cut(lengths, bins=bins, labels=labels, right=False)

    df_e = pd.DataFrame({"true": y_true, "pred": y_pred, "bucket": buckets})
    grp = df_e.groupby("bucket", observed=True).apply(
        lambda g: accuracy_score(g["true"], g["pred"]) if len(g) >= 3 else np.nan
    ).dropna()

    if grp.empty:
        print("Not enough samples for length-bucket plot.")
        return

    fig, ax = plt.subplots(figsize=(8, 4))
    grp.plot(kind="bar", ax=ax)
    ax.set_ylim(0, 1)
    ax.set_ylabel("Accuracy")
    ax.set_title(f"{model_name} — Accuracy by Length Bucket")
    ax.set_xlabel("Word-count bucket")
    plt.xticks(rotation=30, ha="right")
    path = os.path.join(save_dir, f"{model_name}_length_bucket.png")
    plt.tight_layout()
    plt.savefig(path, dpi=160, bbox_inches="tight")
    plt.show()
    plt.close()
    print(f"Saved → {path}")

def save_label_map(root_dir):
    save_json(os.path.join(root_dir, "label_map.json"), {
        "0": "Different subdomain",
        "1": "Same subdomain"
    })

def save_threshold(root_dir, threshold, val_f1):
    save_json(os.path.join(root_dir, "threshold.json"), {
        "threshold": float(threshold),
        "val_f1_macro": float(val_f1)
    })

def save_training_config(root_dir):
    cfg = {
        "pipeline_name": PIPELINE_NAME,
        "model_name": MODEL_NAME,
        "backbone": BACKBONE_NAME,
        "max_abs_words": MAX_ABS_WORDS,
        "max_concl_words": MAX_CONCL_WORDS,
        "max_len": MAX_LEN,
        "seed": SEED,
        "batch_size": BATCH_SIZE,
        "grad_accum": GRAD_ACCUM,
        "lr": LR,
        "weight_decay": WEIGHT_DECAY,
        "epochs": N_EPOCHS,
        "patience": PATIENCE,
        "freeze_layers": FREEZE_LAYERS,
        "max_grad_norm": MAX_GRAD_NORM,
        "label_smoothing": LABEL_SMOOTHING,
        "warmup_ratio": WARMUP_RATIO,
        "gradient_checkpointing": USE_GRAD_CHECKPOINTING,
        "num_workers": NUM_WORKERS,
    }
    save_json(os.path.join(root_dir, "training_config.json"), cfg)

def write_inference_script():
    inference_code = f'''import json
import os
import re
import unicodedata
import torch
from transformers import LongformerTokenizerFast, LongformerForSequenceClassification

ROOT_DIR = os.path.dirname(os.path.abspath(__file__))
MODEL_DIR = os.path.join(ROOT_DIR, "model")
TOKENIZER_DIR = os.path.join(ROOT_DIR, "tokenizer")
THRESH_PATH = os.path.join(ROOT_DIR, "threshold.json")
LABEL_MAP_PATH = os.path.join(ROOT_DIR, "label_map.json")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def load_json(path, default):
    if os.path.exists(path):
        with open(path, "r") as f:
            return json.load(f)
    return default

def normalize_text(text):
    if text is None:
        return ""
    text = unicodedata.normalize("NFC", str(text))
    text = re.sub(r"\\[\\d[\\d,;\\s\\-]*\\]", " ", text)
    text = re.sub(r"\\(\\d[\\d,;\\s\\-]*\\)", " ", text)
    text = re.sub(r"\\s+", " ", text).strip()
    return text

def clean_text(text, max_words=300):
    text = normalize_text(text)
    if not text:
        return ""
    return " ".join(text.split()[:max_words])

tokenizer = LongformerTokenizerFast.from_pretrained(TOKENIZER_DIR)
model = LongformerForSequenceClassification.from_pretrained(MODEL_DIR).to(DEVICE)
model.eval()

thr_payload = load_json(THRESH_PATH, {{"threshold": 0.5}})
threshold = float(thr_payload.get("threshold", 0.5))
label_map = load_json(LABEL_MAP_PATH, {{"0": "Different subdomain", "1": "Same subdomain"}})

def predict_pair(abstract, conclusion, max_len={MAX_LEN}):
    abstract = clean_text(abstract)
    conclusion = clean_text(conclusion)
    enc = tokenizer(
        abstract,
        conclusion,
        max_length=max_len,
        truncation=True,
        padding="max_length",
        return_tensors="pt"
    )
    enc = {{k: v.to(DEVICE) for k, v in enc.items()}}
    global_attention_mask = torch.zeros_like(enc["input_ids"])
    global_attention_mask[:, 0] = 1

    with torch.no_grad():
        logits = model(**enc, global_attention_mask=global_attention_mask).logits
        prob_1 = torch.softmax(logits, dim=-1)[0, 1].item()

    pred = int(prob_1 >= threshold)
    return {{
        "label": pred,
        "label_name": label_map.get(str(pred), str(pred)),
        "probability_same_subdomain": prob_1,
        "threshold": threshold
    }}

if __name__ == "__main__":
    print(predict_pair("Sample abstract.", "Sample conclusion."))
'''
    with open(INFER_PATH, "w") as f:
        f.write(inference_code)
    print(f"✅ Inference script saved to {INFER_PATH}")

def save_model_package():
    model.save_pretrained(MODEL_DIR)
    tokenizer.save_pretrained(TOKENIZER_DIR)
    print(f"✅ Saved model to {MODEL_DIR}")
    print(f"✅ Saved tokenizer to {TOKENIZER_DIR}")

    save_label_map(ROOT_DIR)
    save_threshold(ROOT_DIR, best_val_threshold, best_val_f1)
    save_training_config(ROOT_DIR)
    write_inference_script()

    zip_base = os.path.join(MODELS_DIR, f"{MODEL_NAME}_package")
    zip_path = shutil.make_archive(zip_base, "zip", root_dir=os.path.join(BASE_DIR, "saved_models"), base_dir=MODEL_NAME)
    print(f"📦 Package archived → {zip_path}")
    return zip_path

# ── Training ───────────────────────────────────────────────────────────────
best_val_f1 = 0.0
best_val_threshold = 0.5
patience_ctr = 0
train_losses, val_losses, train_accs, val_accs = [], [], [], []
best_ckpt_path = BEST_CKPT

t0 = time.time()
print(f"\nTraining {MODEL_NAME} …")

for epoch in range(1, N_EPOCHS + 1):
    model.train()
    ep_loss, ep_correct, ep_total = 0.0, 0, 0
    optimizer.zero_grad(set_to_none=True)

    for step, batch in enumerate(trn_loader, start=1):
        labels = batch.pop("labels").to(DEVICE, non_blocking=True)
        batch = {k: v.to(DEVICE, non_blocking=True) for k, v in batch.items()}

        with autocast(enabled=(DEVICE.type == "cuda")):
            out = model(**batch)
            loss = criterion(out.logits, labels) / GRAD_ACCUM

        scaler.scale(loss).backward()

        ep_loss += loss.item() * GRAD_ACCUM
        preds = out.logits.argmax(dim=-1)
        ep_correct += (preds == labels).sum().item()
        ep_total += labels.size(0)

        do_step = (step % GRAD_ACCUM == 0) or (step == len(trn_loader))
        if do_step:
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            scheduler.step()

    train_loss = ep_loss / max(len(trn_loader), 1)
    train_acc = ep_correct / max(ep_total, 1)
    train_losses.append(train_loss)
    train_accs.append(train_acc)

    y_val, p_val, pred_val_05, v_loss = predict_loader(val_loader, threshold=0.5)
    val_acc_05 = accuracy_score(y_val, pred_val_05)
    val_f1_05 = f1_score(y_val, pred_val_05, average="macro", zero_division=0)

    val_thr, val_f1_thr = best_threshold_for_f1(y_val, p_val)
    val_preds_best = (np.asarray(p_val) >= val_thr).astype(int)
    val_acc_best = accuracy_score(y_val, val_preds_best)

    val_losses.append(v_loss)
    val_accs.append(val_acc_best)

    if val_f1_thr > best_val_f1:
        best_val_f1 = val_f1_thr
        best_val_threshold = val_thr
        patience_ctr = 0
        torch.save(model.state_dict(), best_ckpt_path)
    else:
        patience_ctr += 1

    print(
        f"Epoch {epoch:02d} | "
        f"train_loss={train_loss:.4f} | train_acc={train_acc:.3f} | "
        f"val_loss={v_loss:.4f} | val_acc@0.5={val_acc_05:.3f} | "
        f"val_f1@0.5={val_f1_05:.4f} | val_f1@best_thr={val_f1_thr:.4f} | "
        f"thr={val_thr:.3f} | patience={patience_ctr}"
    )

    if patience_ctr >= PATIENCE:
        print(f"Early stopping at epoch {epoch}.")
        break

train_time = time.time() - t0
print(f"\nTraining time: {train_time:.1f} seconds")
print(f"Best validation threshold: {best_val_threshold:.3f}")

# ── Load best checkpoint and evaluate ──────────────────────────────────────
model.load_state_dict(torch.load(best_ckpt_path, map_location=DEVICE))
model.eval()

y_true, probs, raw_preds, _ = predict_loader(tst_loader, threshold=best_val_threshold)
preds = (np.asarray(probs) >= best_val_threshold).astype(int)

metrics = compute_all_metrics(y_true, preds, probs)
metrics.update(compute_hard_negative_accuracy(
    y_true, preds,
    test_df["abs_wc"].tolist(),
    test_df["concl_wc"].tolist()
))
metrics["training_time_s"] = float(train_time)
metrics["best_val_threshold"] = float(best_val_threshold)
metrics["best_val_f1_macro"] = float(best_val_f1)

print("\nFinal metrics:")
for k, v in metrics.items():
    try:
        print(f"{k:<30s}: {v:.4f}")
    except Exception:
        print(f"{k:<30s}: {v}")

# ── Plots ──────────────────────────────────────────────────────────────────
plot_confusion_matrix(y_true, preds, MODEL_NAME, RESULTS_DIR)
plot_roc_curve(y_true, probs, MODEL_NAME, RESULTS_DIR)
plot_pr_curve(y_true, probs, MODEL_NAME, RESULTS_DIR)
plot_training_curves(train_losses, val_losses, train_accs, val_accs, MODEL_NAME, RESULTS_DIR)

if "abstract_subdomain" in test_df.columns:
    plot_subdomain_performance(
        y_true, preds,
        test_df["abstract_subdomain"].fillna("unknown").astype(str).tolist(),
        MODEL_NAME, RESULTS_DIR
    )

plot_length_bucket_performance(
    y_true, preds,
    (test_df["abs_wc"].fillna(0) + test_df["concl_wc"].fillna(0)).tolist(),
    MODEL_NAME, RESULTS_DIR
)

# ── Save artifacts ─────────────────────────────────────────────────────────
save_metrics(metrics, METRICS_PATH)
print(f"✅ Metrics saved to {METRICS_PATH}")

pred_out = pd.DataFrame({
    "y_true": y_true,
    "prob_1": probs,
    "pred": preds
})
pred_path = os.path.join(ROOT_DIR, "test_predictions.csv")
pred_out.to_csv(pred_path, index=False)
print(f"✅ Test predictions saved to {pred_path}")

save_model_package()
ALL_RESULTS[MODEL_NAME] = metrics

print("\n✅ Longformer cell complete.")
print(f"Elapsed time: {train_time/60:.1f} minutes")
print(f"Best checkpoint: {BEST_CKPT}")
print(f"Archive: {PACKAGE_ZIP}")

In [ ]:
# ── Cell 10: BigBird training (high-performance, T4-heavy version) ─────────
# Direct sequence classification, no freezing, no checkpointing.
# Designed to push GPU utilization higher on T4.

import os
import gc
import json
import time
import math
import random
import warnings
from collections import defaultdict

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

from transformers import AutoTokenizer, BigBirdForSequenceClassification, get_linear_schedule_with_warmup
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, precision_recall_curve,
    confusion_matrix, roc_curve, classification_report, log_loss, auc
)
from sklearn.utils.class_weight import compute_class_weight

import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")

# ── Reproducibility & speed ────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
try:
    torch.set_float32_matmul_precision("high")
except Exception:
    pass

os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# ── Config ────────────────────────────────────────────────────────────────
BASE_DIR = "/content"
MODELS_DIR = globals().get("MODELS_DIR", os.path.join(BASE_DIR, "saved_models"))
PROCESSED_DIR = globals().get("PROCESSED_DIR", os.path.join(BASE_DIR, "processed_splits"))

MODEL_NAME = "BigBird"
BD_CKPT = "google/bigbird-roberta-base"

MAX_LEN = 512
BATCH_SIZE = 4
GRAD_ACCUM = 4
LR = 2e-5
HEAD_LR = 5e-5
WEIGHT_DECAY = 0.01
N_EPOCHS = 12
PATIENCE = 4
MAX_GRAD_NORM = 1.0
LABEL_SMOOTHING = 0.01
NUM_WORKERS = 2
WARMUP_RATIO = 0.10
DROPOUT = 0.20

PIPELINE_NAME = "bigbird_pair_classifier"

BD_DIR = os.path.join(MODELS_DIR, MODEL_NAME)
MODEL_DIR = os.path.join(BD_DIR, "model")
TOKENIZER_DIR = os.path.join(BD_DIR, "tokenizer")
RESULTS_DIR = os.path.join(BD_DIR, "results")
CACHE_DIR = os.path.join(BD_DIR, "cache")
BEST_CKPT = os.path.join(BD_DIR, "best_model.pt")
METRICS_PATH = os.path.join(BD_DIR, "metrics.json")
LABEL_MAP_PATH = os.path.join(BD_DIR, "label_map.json")
THRESH_PATH = os.path.join(BD_DIR, "threshold.json")
INFER_PATH = os.path.join(BD_DIR, "inference.py")
CONFIG_PATH = os.path.join(BD_DIR, "training_config.json")
PACKAGE_ZIP = os.path.join(MODELS_DIR, f"{MODEL_NAME}_package.zip")

for d in [BD_DIR, MODEL_DIR, TOKENIZER_DIR, RESULTS_DIR, CACHE_DIR]:
    os.makedirs(d, exist_ok=True)

ALL_RESULTS = globals().get("ALL_RESULTS", {})

print(f"Root      : {BD_DIR}")
print(f"Model     : {MODEL_DIR}")
print(f"Tokenizer : {TOKENIZER_DIR}")
print(f"Results   : {RESULTS_DIR}")
print(f"Archive   : {PACKAGE_ZIP}")

# ── Load data ─────────────────────────────────────────────────────────────
train_df = pd.read_csv(os.path.join(PROCESSED_DIR, "train.csv"))
val_df   = pd.read_csv(os.path.join(PROCESSED_DIR, "val.csv"))
test_df  = pd.read_csv(os.path.join(PROCESSED_DIR, "test.csv"))

print(f"Loaded splits — train:{len(train_df)}  val:{len(val_df)}  test:{len(test_df)}")

def resolve_text_columns(df):
    abs_candidates = ["abstract_clean", "abstract_raw", "abstract_model", "abstract"]
    con_candidates = ["conclusion_clean", "conclusion_raw", "conclusion_model", "conclusion"]
    abs_col = next((c for c in abs_candidates if c in df.columns), None)
    con_col = next((c for c in con_candidates if c in df.columns), None)
    if abs_col is None or con_col is None:
        raise ValueError("Could not find abstract/conclusion columns in the split files.")
    return abs_col, con_col

ABS_COL, CONCL_COL = resolve_text_columns(train_df)
print(f"Using text columns: {ABS_COL}, {CONCL_COL}")

# ── Cleaning ───────────────────────────────────────────────────────────────
MAX_ABS_WORDS = 300
MAX_CONCL_WORDS = 300

def clean_text(text: str, max_words: int = None) -> str:
    if not isinstance(text, str) or not text.strip():
        return ""
    text = " ".join(text.replace("\n", " ").split())
    if max_words and max_words > 0:
        words = text.split()
        if len(words) > max_words:
            text = " ".join(words[:max_words])
    return text

for split_name, split_df in [("train", train_df), ("val", val_df), ("test", test_df)]:
    split_df[ABS_COL] = split_df[ABS_COL].apply(lambda x: clean_text(x, MAX_ABS_WORDS))
    split_df[CONCL_COL] = split_df[CONCL_COL].apply(lambda x: clean_text(x, MAX_CONCL_WORDS))
    split_df.dropna(subset=["label"], inplace=True)
    split_df["label"] = split_df["label"].astype(int)

    before = len(split_df)
    split_df = split_df[(split_df[ABS_COL].str.len() > 0) & (split_df[CONCL_COL].str.len() > 0)].copy()
    split_df["abs_wc"] = split_df[ABS_COL].str.split().str.len()
    split_df["concl_wc"] = split_df[CONCL_COL].str.split().str.len()
    split_df = split_df[(split_df["abs_wc"] <= 1000) & (split_df["concl_wc"] <= 1000)].copy()

    if split_name == "train":
        train_df = split_df.reset_index(drop=True)
    elif split_name == "val":
        val_df = split_df.reset_index(drop=True)
    else:
        test_df = split_df.reset_index(drop=True)

    print(f"{split_name}: dropped {before - len(split_df):,} rows; remaining {len(split_df):,}")

for split_df in [train_df, val_df, test_df]:
    split_df["input_text_clean"] = "Abstract: " + split_df[ABS_COL] + " [SEP] Conclusion: " + split_df[CONCL_COL]

# ── Model / tokenizer ──────────────────────────────────────────────────────
print(f"Loading backbone: {BD_CKPT}")
tokenizer = AutoTokenizer.from_pretrained(BD_CKPT, use_fast=True)
model = BigBirdForSequenceClassification.from_pretrained(
    BD_CKPT,
    num_labels=2,
    attention_type="block_sparse"
)

try:
    model.config.classifier_dropout = DROPOUT
except Exception:
    pass

model.to(DEVICE)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable params: {trainable:,} / {total:,}")

if DEVICE.type == "cuda":
    torch.cuda.empty_cache()
gc.collect()

# ── Dataset / collator ─────────────────────────────────────────────────────
class PairDataset(Dataset):
    def __init__(self, df, abs_col, concl_col):
        self.abstracts = df[abs_col].astype(str).tolist()
        self.conclusions = df[concl_col].astype(str).tolist()
        self.labels = df["label"].astype(int).tolist()

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, i):
        return self.abstracts[i], self.conclusions[i], self.labels[i]

def collate_fn(batch):
    abstracts, conclusions, labels = zip(*batch)
    enc = tokenizer(
        list(abstracts),
        list(conclusions),
        max_length=MAX_LEN,
        truncation=True,
        padding=True,
        return_tensors="pt",
        pad_to_multiple_of=8 if DEVICE.type == "cuda" else None,
    )
    enc["labels"] = torch.tensor(labels, dtype=torch.long)
    return enc

train_ds = PairDataset(train_df, ABS_COL, CONCL_COL)
val_ds = PairDataset(val_df, ABS_COL, CONCL_COL)
test_ds = PairDataset(test_df, ABS_COL, CONCL_COL)

train_labels = train_df["label"].astype(int).tolist()
class_counts = np.bincount(np.asarray(train_labels), minlength=2)
sample_weights = [1.0 / class_counts[y] for y in train_labels]
train_sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)

loader_kwargs = dict(
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    pin_memory=(DEVICE.type == "cuda"),
    persistent_workers=(NUM_WORKERS > 0),
    collate_fn=collate_fn,
)
if NUM_WORKERS > 0:
    loader_kwargs["prefetch_factor"] = 2

trn_loader = DataLoader(train_ds, sampler=train_sampler, shuffle=False, **loader_kwargs)
val_loader = DataLoader(val_ds, shuffle=False, **loader_kwargs)
tst_loader = DataLoader(test_ds, shuffle=False, **loader_kwargs)

# ── Loss / optimizer / scheduler ───────────────────────────────────────────
def get_class_weights(labels):
    labels = np.asarray(labels, dtype=int)
    classes = np.unique(labels)
    weights = compute_class_weight(class_weight="balanced", classes=classes, y=labels)
    out = np.ones(2, dtype=np.float32)
    for c, w in zip(classes, weights):
        out[int(c)] = float(w)
    return {0: out[0], 1: out[1]}

cw = get_class_weights(train_df["label"].tolist())
w_tensor = torch.tensor([cw[0], cw[1]], dtype=torch.float32, device=DEVICE)
print("Class weights:", cw)

criterion = nn.CrossEntropyLoss(weight=w_tensor, label_smoothing=LABEL_SMOOTHING)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=WEIGHT_DECAY
)

num_update_steps_per_epoch = math.ceil(len(trn_loader) / GRAD_ACCUM)
total_steps = num_update_steps_per_epoch * N_EPOCHS
warmup_steps = int(WARMUP_RATIO * total_steps)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)

scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE.type == "cuda"))

# ── Helper functions ──────────────────────────────────────────────────────
LABEL2TEXT = {0: "different_subdomain", 1: "same_subdomain"}

def save_json(path, payload):
    with open(path, "w") as f:
        json.dump(payload, f, indent=2)

def save_metrics(metrics, path):
    clean = {}
    for k, v in metrics.items():
        if isinstance(v, (np.floating, float)):
            clean[k] = float(v) if np.isfinite(v) else None
        elif isinstance(v, (np.integer, int)):
            clean[k] = int(v)
        else:
            clean[k] = v
    save_json(path, clean)

def best_threshold_for_f1(y_true, probs):
    thresholds = np.linspace(0.05, 0.95, 181)
    best_thr, best_f1 = 0.5, -1.0
    y_true = np.asarray(y_true, dtype=int)
    probs = np.asarray(probs, dtype=float)
    for thr in thresholds:
        preds = (probs >= thr).astype(int)
        f1 = f1_score(y_true, preds, average="macro", zero_division=0)
        if f1 > best_f1:
            best_f1 = f1
            best_thr = float(thr)
    return best_thr, float(best_f1)

def compute_hard_negative_accuracy(y_true, y_pred, abs_wc, concl_wc):
    y_true = np.asarray(y_true, dtype=int)
    y_pred = np.asarray(y_pred, dtype=int)
    abs_wc = np.asarray(abs_wc, dtype=int)
    concl_wc = np.asarray(concl_wc, dtype=int)

    mask = (y_true == 0) & (np.abs(abs_wc - concl_wc) < 50)
    idx = np.where(mask)[0]
    if len(idx) == 0:
        return {"hard_neg_count": 0, "hard_neg_accuracy": float("nan")}
    return {
        "hard_neg_count": int(len(idx)),
        "hard_neg_accuracy": float(accuracy_score(y_true[idx], y_pred[idx]))
    }

def compute_all_metrics(y_true, y_pred, y_prob):
    y_true = np.asarray(y_true, dtype=int)
    y_pred = np.asarray(y_pred, dtype=int)
    y_prob = np.asarray(y_prob, dtype=float)

    out = {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "precision_macro": float(precision_score(y_true, y_pred, average="macro", zero_division=0)),
        "recall_macro": float(recall_score(y_true, y_pred, average="macro", zero_division=0)),
        "f1_macro": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "precision_weighted": float(precision_score(y_true, y_pred, average="weighted", zero_division=0)),
        "recall_weighted": float(recall_score(y_true, y_pred, average="weighted", zero_division=0)),
        "f1_weighted": float(f1_score(y_true, y_pred, average="weighted", zero_division=0)),
    }

    rep = classification_report(y_true, y_pred, output_dict=True, zero_division=0)
    if "0" in rep:
        out["precision_class0"] = float(rep["0"]["precision"])
        out["recall_class0"] = float(rep["0"]["recall"])
        out["f1_class0"] = float(rep["0"]["f1-score"])
    if "1" in rep:
        out["precision_class1"] = float(rep["1"]["precision"])
        out["recall_class1"] = float(rep["1"]["recall"])
        out["f1_class1"] = float(rep["1"]["f1-score"])

    out["roc_auc"] = float(roc_auc_score(y_true, y_prob)) if len(np.unique(y_true)) > 1 else float("nan")
    out["pr_auc"] = float(average_precision_score(y_true, y_prob))
    out["log_loss"] = float(log_loss(y_true, np.column_stack([1 - y_prob, y_prob])))
    return out

def plot_confusion_matrix(y_true, y_pred, model_name, save_dir):
    cm = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(5, 4))
    sns.heatmap(
        cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax,
        xticklabels=["Diff (0)", "Same (1)"],
        yticklabels=["Diff (0)", "Same (1)"]
    )
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_title(f"{model_name} — Confusion Matrix")
    path = os.path.join(save_dir, f"{model_name}_confusion_matrix.png")
    plt.tight_layout()
    plt.savefig(path, dpi=160, bbox_inches="tight")
    plt.show()
    plt.close()
    print(f"Saved → {path}")

def plot_roc_curve(y_true, y_prob, model_name, save_dir):
    if len(np.unique(y_true)) < 2:
        print("ROC curve skipped (only one class present).")
        return
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    auc_score = roc_auc_score(y_true, y_prob)
    fig, ax = plt.subplots(figsize=(5, 4))
    ax.plot(fpr, tpr, lw=2, label=f"AUC={auc_score:.3f}")
    ax.plot([0, 1], [0, 1], "k--", lw=1)
    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.set_title(f"{model_name} — ROC Curve")
    ax.legend(loc="lower right")
    path = os.path.join(save_dir, f"{model_name}_roc_curve.png")
    plt.tight_layout()
    plt.savefig(path, dpi=160, bbox_inches="tight")
    plt.show()
    plt.close()
    print(f"Saved → {path}")

def plot_pr_curve(y_true, y_prob, model_name, save_dir):
    precision, recall, _ = precision_recall_curve(y_true, y_prob)
    ap = average_precision_score(y_true, y_prob)
    fig, ax = plt.subplots(figsize=(5, 4))
    ax.plot(recall, precision, lw=2, label=f"AP={ap:.3f}")
    ax.set_xlabel("Recall")
    ax.set_ylabel("Precision")
    ax.set_title(f"{model_name} — Precision-Recall Curve")
    ax.legend(loc="upper right")
    path = os.path.join(save_dir, f"{model_name}_pr_curve.png")
    plt.tight_layout()
    plt.savefig(path, dpi=160, bbox_inches="tight")
    plt.show()
    plt.close()
    print(f"Saved → {path}")

def plot_training_curves(train_losses, val_losses, train_accs, val_accs, model_name, save_dir):
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    epochs = range(1, len(train_losses) + 1)

    axes[0].plot(epochs, train_losses, "o-", label="Train")
    if val_losses:
        axes[0].plot(epochs, val_losses, "s--", label="Val")
    axes[0].set_title(f"{model_name} — Loss")
    axes[0].set_xlabel("Epoch")
    axes[0].legend()

    axes[1].plot(epochs, train_accs, "o-", label="Train")
    if val_accs:
        axes[1].plot(epochs, val_accs, "s--", label="Val")
    axes[1].set_title(f"{model_name} — Accuracy")
    axes[1].set_xlabel("Epoch")
    axes[1].legend()

    path = os.path.join(save_dir, f"{model_name}_training_curves.png")
    plt.tight_layout()
    plt.savefig(path, dpi=160, bbox_inches="tight")
    plt.show()
    plt.close()
    print(f"Saved → {path}")

def plot_subdomain_performance(y_true, y_pred, subdomains, model_name, save_dir):
    df_e = pd.DataFrame({"true": y_true, "pred": y_pred, "subdomain": subdomains})
    grp = df_e.groupby("subdomain").apply(
        lambda g: accuracy_score(g["true"], g["pred"]) if len(g) >= 3 else np.nan
    ).dropna().sort_values(ascending=False)

    if grp.empty:
        print("Not enough samples for subdomain plot.")
        return

    fig, ax = plt.subplots(figsize=(max(8, len(grp) * 0.75), 4))
    grp.plot(kind="bar", ax=ax)
    ax.axhline(accuracy_score(y_true, y_pred), color="red", ls="--", label="Overall")
    ax.set_ylabel("Accuracy")
    ax.set_title(f"{model_name} — Subdomain Accuracy")
    ax.legend()
    plt.xticks(rotation=45, ha="right")
    path = os.path.join(save_dir, f"{model_name}_subdomain_performance.png")
    plt.tight_layout()
    plt.savefig(path, dpi=160, bbox_inches="tight")
    plt.show()
    plt.close()
    print(f"Saved → {path}")

def plot_length_bucket_performance(y_true, y_pred, lengths, model_name, save_dir):
    lengths = np.asarray(lengths)
    bins = [0, 100, 200, 300, 400, 600, 100000]
    labels = ["<100", "100-200", "200-300", "300-400", "400-600", "600+"]
    buckets = pd.cut(lengths, bins=bins, labels=labels, right=False)

    df_e = pd.DataFrame({"true": y_true, "pred": y_pred, "bucket": buckets})
    grp = df_e.groupby("bucket", observed=True).apply(
        lambda g: accuracy_score(g["true"], g["pred"]) if len(g) >= 3 else np.nan
    ).dropna()

    if grp.empty:
        print("Not enough samples for length-bucket plot.")
        return

    fig, ax = plt.subplots(figsize=(8, 4))
    grp.plot(kind="bar", ax=ax)
    ax.set_ylim(0, 1)
    ax.set_ylabel("Accuracy")
    ax.set_title(f"{model_name} — Accuracy by Length Bucket")
    ax.set_xlabel("Word-count bucket")
    plt.xticks(rotation=30, ha="right")
    path = os.path.join(save_dir, f"{model_name}_length_bucket.png")
    plt.tight_layout()
    plt.savefig(path, dpi=160, bbox_inches="tight")
    plt.show()
    plt.close()
    print(f"Saved → {path}")

def save_label_map(root_dir):
    save_json(os.path.join(root_dir, "label_map.json"), {
        "0": "Different subdomain",
        "1": "Same subdomain"
    })

def save_threshold(root_dir, threshold, val_f1):
    save_json(os.path.join(root_dir, "threshold.json"), {
        "threshold": float(threshold),
        "val_f1_macro": float(val_f1)
    })

def save_training_config(root_dir):
    cfg = {
        "pipeline_name": PIPELINE_NAME,
        "model_name": MODEL_NAME,
        "backbone": BD_CKPT,
        "max_abs_words": MAX_ABS_WORDS,
        "max_concl_words": MAX_CONCL_WORDS,
        "max_len": MAX_LEN,
        "seed": SEED,
        "batch_size": BATCH_SIZE,
        "grad_accum": GRAD_ACCUM,
        "lr": LR,
        "head_lr": HEAD_LR,
        "weight_decay": WEIGHT_DECAY,
        "epochs": N_EPOCHS,
        "patience": PATIENCE,
        "max_grad_norm": MAX_GRAD_NORM,
        "label_smoothing": LABEL_SMOOTHING,
        "warmup_ratio": WARMUP_RATIO,
        "dropout": DROPOUT,
        "num_workers": NUM_WORKERS,
    }
    save_json(os.path.join(root_dir, "training_config.json"), cfg)

def write_inference_script():
    inference_code = f'''import json
import os
import torch
from transformers import AutoTokenizer, BigBirdForSequenceClassification

ROOT_DIR = os.path.dirname(os.path.abspath(__file__))
MODEL_DIR = os.path.join(ROOT_DIR, "model")
TOKENIZER_DIR = os.path.join(ROOT_DIR, "tokenizer")
THRESH_PATH = os.path.join(ROOT_DIR, "threshold.json")
LABEL_MAP_PATH = os.path.join(ROOT_DIR, "label_map.json")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def load_json(path, default):
    if os.path.exists(path):
        with open(path, "r") as f:
            return json.load(f)
    return default

tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_DIR, use_fast=True)
model = BigBirdForSequenceClassification.from_pretrained(MODEL_DIR).to(DEVICE)
model.eval()

thr_payload = load_json(THRESH_PATH, {{"threshold": 0.5}})
threshold = float(thr_payload.get("threshold", 0.5))
label_map = load_json(LABEL_MAP_PATH, {{"0": "Different subdomain", "1": "Same subdomain"}})

def predict_pair(abstract, conclusion, max_len={MAX_LEN}):
    prompt = f"Abstract: {{abstract}} [SEP] Conclusion: {{conclusion}}"
    enc = tokenizer(
        prompt,
        max_length=max_len,
        truncation=True,
        padding="max_length",
        return_tensors="pt"
    )
    enc = {{k: v.to(DEVICE) for k, v in enc.items()}}
    with torch.no_grad():
        logits = model(**enc).logits
        prob_1 = torch.softmax(logits, dim=-1)[0, 1].item()
    pred = int(prob_1 >= threshold)
    return {{
        "label": pred,
        "label_name": label_map.get(str(pred), str(pred)),
        "probability_same_subdomain": prob_1,
        "threshold": threshold
    }}

if __name__ == "__main__":
    print(predict_pair("Sample abstract.", "Sample conclusion."))
'''
    with open(INFER_PATH, "w") as f:
        f.write(inference_code)
    print(f"✅ Inference script saved to {INFER_PATH}")

def zip_model_package():
    zip_base = os.path.join(MODELS_DIR, f"{MODEL_NAME}_package")
    zip_path = shutil.make_archive(zip_base, "zip", root_dir=MODELS_DIR, base_dir=MODEL_NAME)
    print(f"📦 Package archived → {zip_path}")
    return zip_path

@torch.no_grad()
def predict_loader(loader, threshold=0.5):
    model.eval()
    all_true, all_probs, all_preds = [], [], []
    total_loss, total_n = 0.0, 0

    for batch in loader:
        labels = batch.pop("labels").to(DEVICE, non_blocking=True)
        batch = {k: v.to(DEVICE, non_blocking=True) for k, v in batch.items()}

        with torch.cuda.amp.autocast(enabled=(DEVICE.type == "cuda")):
            logits = model(**batch).logits
            loss = criterion(logits, labels)

        probs = torch.softmax(logits, dim=-1)[:, 1]
        preds = (probs >= threshold).long()

        bs = labels.size(0)
        total_loss += loss.item() * bs
        total_n += bs

        all_true.extend(labels.detach().cpu().tolist())
        all_probs.extend(probs.detach().cpu().tolist())
        all_preds.extend(preds.detach().cpu().tolist())

    return all_true, all_probs, all_preds, total_loss / max(total_n, 1)

# ── Training ───────────────────────────────────────────────────────────────
best_val_f1 = 0.0
best_val_threshold = 0.5
patience_ctr = 0
train_losses, val_losses, train_accs, val_accs = [], [], [], []
best_ckpt_path = BEST_CKPT

t0 = time.time()
print(f"\nTraining {MODEL_NAME} …")

for epoch in range(1, N_EPOCHS + 1):
    model.train()
    ep_loss, ep_correct, ep_total = 0.0, 0, 0
    optimizer.zero_grad(set_to_none=True)

    for step, batch in enumerate(trn_loader, start=1):
        labels = batch.pop("labels").to(DEVICE, non_blocking=True)
        batch = {k: v.to(DEVICE, non_blocking=True) for k, v in batch.items()}

        with torch.cuda.amp.autocast(enabled=(DEVICE.type == "cuda")):
            logits = model(**batch).logits
            loss = criterion(logits, labels) / GRAD_ACCUM

        scaler.scale(loss).backward()

        ep_loss += loss.item() * GRAD_ACCUM
        preds = logits.argmax(dim=-1)
        ep_correct += (preds == labels).sum().item()
        ep_total += labels.size(0)

        do_step = (step % GRAD_ACCUM == 0) or (step == len(trn_loader))
        if do_step:
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            scheduler.step()

    train_loss = ep_loss / max(len(trn_loader), 1)
    train_acc = ep_correct / max(ep_total, 1)
    train_losses.append(train_loss)
    train_accs.append(train_acc)

    y_val, p_val, pred_val_05, v_loss = predict_loader(val_loader, threshold=0.5)
    val_acc_05 = accuracy_score(y_val, pred_val_05)
    val_f1_05 = f1_score(y_val, pred_val_05, average="macro", zero_division=0)

    val_thr, val_f1_thr = best_threshold_for_f1(y_val, p_val)
    val_preds_best = (np.asarray(p_val) >= val_thr).astype(int)
    val_acc_best = accuracy_score(y_val, val_preds_best)

    val_losses.append(v_loss)
    val_accs.append(val_acc_best)

    if val_f1_thr > best_val_f1:
        best_val_f1 = val_f1_thr
        best_val_threshold = val_thr
        patience_ctr = 0
        torch.save(model.state_dict(), best_ckpt_path)
    else:
        patience_ctr += 1

    print(
        f"Epoch {epoch:02d} | "
        f"train_loss={train_loss:.4f} | train_acc={train_acc:.3f} | "
        f"val_loss={v_loss:.4f} | val_acc@0.5={val_acc_05:.3f} | "
        f"val_f1@0.5={val_f1_05:.4f} | val_f1@best_thr={val_f1_thr:.4f} | "
        f"thr={val_thr:.3f} | patience={patience_ctr}"
    )

    if patience_ctr >= PATIENCE:
        print(f"Early stopping at epoch {epoch}.")
        break

    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

train_time = time.time() - t0
print(f"\nTraining time: {train_time:.1f} seconds")
print(f"Best validation threshold: {best_val_threshold:.3f}")

# ── Load best checkpoint ───────────────────────────────────────────────────
model.load_state_dict(torch.load(best_ckpt_path, map_location=DEVICE))
model.eval()

# ── Test evaluation ────────────────────────────────────────────────────────
y_true, probs, raw_preds, _ = predict_loader(tst_loader, threshold=best_val_threshold)
preds = (np.asarray(probs) >= best_val_threshold).astype(int)

metrics = compute_all_metrics(y_true, preds, probs)
metrics.update(compute_hard_negative_accuracy(
    y_true, preds,
    test_df["abs_wc"].tolist(),
    test_df["concl_wc"].tolist()
))
metrics["training_time_s"] = float(train_time)
metrics["best_val_threshold"] = float(best_val_threshold)
metrics["best_val_f1_macro"] = float(best_val_f1)

print("\nFinal metrics:")
for k, v in metrics.items():
    try:
        print(f"{k:<30s}: {v:.4f}")
    except Exception:
        print(f"{k:<30s}: {v}")

# ── Plots ──────────────────────────────────────────────────────────────────
plot_confusion_matrix(y_true, preds, MODEL_NAME, RESULTS_DIR)
plot_roc_curve(y_true, probs, MODEL_NAME, RESULTS_DIR)
plot_pr_curve(y_true, probs, MODEL_NAME, RESULTS_DIR)
plot_training_curves(train_losses, val_losses, train_accs, val_accs, MODEL_NAME, RESULTS_DIR)

if "abstract_subdomain" in test_df.columns:
    plot_subdomain_performance(
        y_true, preds,
        test_df["abstract_subdomain"].fillna("unknown").astype(str).tolist(),
        MODEL_NAME, RESULTS_DIR
    )

plot_length_bucket_performance(
    y_true, preds,
    (test_df["abs_wc"].fillna(0) + test_df["concl_wc"].fillna(0)).tolist(),
    MODEL_NAME, RESULTS_DIR
)

# ── Save artifacts ─────────────────────────────────────────────────────────
save_metrics(metrics, METRICS_PATH)
print(f"✅ Metrics saved to {METRICS_PATH}")

pred_out = pd.DataFrame({
    "y_true": y_true,
    "prob_1": probs,
    "pred": preds
})
pred_path = os.path.join(BD_DIR, "test_predictions.csv")
pred_out.to_csv(pred_path, index=False)
print(f"✅ Test predictions saved to {pred_path}")

save_label_map(BD_DIR)
save_threshold(BD_DIR, best_val_threshold, best_val_f1)
save_training_config(BD_DIR)
write_inference_script()

model.save_pretrained(MODEL_DIR)
tokenizer.save_pretrained(TOKENIZER_DIR)
print(f"✅ Saved model to {MODEL_DIR}")
print(f"✅ Saved tokenizer to {TOKENIZER_DIR}")

zip_model_package()
ALL_RESULTS[MODEL_NAME] = metrics

print("\n✅ BigBird complete.")
print(f"Elapsed time: {train_time/60:.1f} minutes")
print(f"Best checkpoint: {BEST_CKPT}")
print(f"Archive: {PACKAGE_ZIP}")

In [ ]:
# ── Cell 11: T5 training (high-performance, T4-heavy version) ──────────────
# Uses T5 as a single-token classifier with yes/no labels.
# This is faster and lighter than full generation, but still pushes T4 well.

import os
import gc
import json
import time
import math
import random
import warnings
from collections import defaultdict

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

from transformers import AutoTokenizer, T5ForConditionalGeneration, get_linear_schedule_with_warmup
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, precision_recall_curve,
    confusion_matrix, roc_curve, classification_report, log_loss, auc
)
from sklearn.utils.class_weight import compute_class_weight

import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")

# ── Reproducibility & speed ────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
try:
    torch.set_float32_matmul_precision("high")
except Exception:
    pass

os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# ── Config ────────────────────────────────────────────────────────────────
BASE_DIR = "/content"
MODELS_DIR = globals().get("MODELS_DIR", os.path.join(BASE_DIR, "saved_models"))
PROCESSED_DIR = globals().get("PROCESSED_DIR", os.path.join(BASE_DIR, "processed_splits"))

MODEL_NAME = "T5"
T5_CKPT = "t5-base"

MAX_IN_LEN = 512
BATCH_SIZE = 2
GRAD_ACCUM = 8
LR = 2e-5
WEIGHT_DECAY = 0.01
N_EPOCHS = 12
PATIENCE = 4
MAX_GRAD_NORM = 1.0
LABEL_SMOOTHING = 0.0
NUM_WORKERS = 2
WARMUP_RATIO = 0.10
DROPOUT = 0.10

PIPELINE_NAME = "t5_pair_classifier"

T5_DIR = os.path.join(MODELS_DIR, MODEL_NAME)
MODEL_DIR = os.path.join(T5_DIR, "model")
TOKENIZER_DIR = os.path.join(T5_DIR, "tokenizer")
RESULTS_DIR = os.path.join(T5_DIR, "results")
CACHE_DIR = os.path.join(T5_DIR, "cache")
BEST_CKPT = os.path.join(T5_DIR, "best_model.pt")
METRICS_PATH = os.path.join(T5_DIR, "metrics.json")
LABEL_MAP_PATH = os.path.join(T5_DIR, "label_map.json")
THRESH_PATH = os.path.join(T5_DIR, "threshold.json")
INFER_PATH = os.path.join(T5_DIR, "inference.py")
CONFIG_PATH = os.path.join(T5_DIR, "training_config.json")
PACKAGE_ZIP = os.path.join(MODELS_DIR, f"{MODEL_NAME}_package.zip")

for d in [T5_DIR, MODEL_DIR, TOKENIZER_DIR, RESULTS_DIR, CACHE_DIR]:
    os.makedirs(d, exist_ok=True)

ALL_RESULTS = globals().get("ALL_RESULTS", {})

print(f"Root      : {T5_DIR}")
print(f"Model     : {MODEL_DIR}")
print(f"Tokenizer : {TOKENIZER_DIR}")
print(f"Results   : {RESULTS_DIR}")
print(f"Archive   : {PACKAGE_ZIP}")

# ── Load data ─────────────────────────────────────────────────────────────
train_df = pd.read_csv(os.path.join(PROCESSED_DIR, "train.csv"))
val_df   = pd.read_csv(os.path.join(PROCESSED_DIR, "val.csv"))
test_df  = pd.read_csv(os.path.join(PROCESSED_DIR, "test.csv"))

print(f"Loaded splits — train:{len(train_df)}  val:{len(val_df)}  test:{len(test_df)}")

def resolve_text_columns(df):
    abs_candidates = ["abstract_clean", "abstract_raw", "abstract_model", "abstract"]
    con_candidates = ["conclusion_clean", "conclusion_raw", "conclusion_model", "conclusion"]
    abs_col = next((c for c in abs_candidates if c in df.columns), None)
    con_col = next((c for c in con_candidates if c in df.columns), None)
    if abs_col is None or con_col is None:
        raise ValueError("Could not find abstract/conclusion columns in the split files.")
    return abs_col, con_col

ABS_COL, CONCL_COL = resolve_text_columns(train_df)
print(f"Using text columns: {ABS_COL}, {CONCL_COL}")

MAX_ABS_WORDS = 300
MAX_CONCL_WORDS = 300

def clean_text(text: str, max_words: int = None) -> str:
    if not isinstance(text, str) or not text.strip():
        return ""
    text = " ".join(text.replace("\n", " ").split())
    if max_words and max_words > 0:
        words = text.split()
        if len(words) > max_words:
            text = " ".join(words[:max_words])
    return text

for split_name, split_df in [("train", train_df), ("val", val_df), ("test", test_df)]:
    split_df[ABS_COL] = split_df[ABS_COL].apply(lambda x: clean_text(x, MAX_ABS_WORDS))
    split_df[CONCL_COL] = split_df[CONCL_COL].apply(lambda x: clean_text(x, MAX_CONCL_WORDS))
    split_df.dropna(subset=["label"], inplace=True)
    split_df["label"] = split_df["label"].astype(int)

    before = len(split_df)
    split_df = split_df[(split_df[ABS_COL].str.len() > 0) & (split_df[CONCL_COL].str.len() > 0)].copy()
    split_df["abs_wc"] = split_df[ABS_COL].str.split().str.len()
    split_df["concl_wc"] = split_df[CONCL_COL].str.split().str.len()
    split_df = split_df[(split_df["abs_wc"] <= 1000) & (split_df["concl_wc"] <= 1000)].copy()

    if split_name == "train":
        train_df = split_df.reset_index(drop=True)
    elif split_name == "val":
        val_df = split_df.reset_index(drop=True)
    else:
        test_df = split_df.reset_index(drop=True)

    print(f"{split_name}: dropped {before - len(split_df):,} rows; remaining {len(split_df):,}")

# ── Tokenizer / model ──────────────────────────────────────────────────────
print(f"Loading backbone: {T5_CKPT}")
tokenizer = AutoTokenizer.from_pretrained(T5_CKPT, use_fast=True)
tokenizer.padding_side = "right"

model = T5ForConditionalGeneration.from_pretrained(T5_CKPT)

try:
    model.config.use_cache = False
except Exception:
    pass

try:
    model.config.dropout_rate = DROPOUT
except Exception:
    pass

# No checkpointing: higher memory usage, faster training
model.to(DEVICE)

# ── Label words ────────────────────────────────────────────────────────────
LABEL2TEXT = {0: "no", 1: "yes"}
LABEL_MAP = {"0": "Different subdomain", "1": "Same subdomain"}

def get_single_token_id(word: str) -> int:
    ids = tokenizer(word, add_special_tokens=False).input_ids
    if len(ids) == 0:
        raise ValueError(f"Could not tokenize label word: {word}")
    return int(ids[0])

NO_ID = get_single_token_id("no")
YES_ID = get_single_token_id("yes")
print(f"Label token ids — no: {NO_ID}, yes: {YES_ID}")

# ── Dataset / collator ─────────────────────────────────────────────────────
class T5Dataset(Dataset):
    def __init__(self, df):
        self.abstracts = df[ABS_COL].astype(str).tolist()
        self.conclusions = df[CONCL_COL].astype(str).tolist()
        self.labels = df["label"].astype(int).tolist()

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, i):
        prompt = (
            "classify whether the abstract and conclusion are in the same subdomain: "
            f"abstract: {self.abstracts[i]} conclusion: {self.conclusions[i]}"
        )
        return prompt, self.labels[i]

def collate_t5(batch):
    prompts, labels = zip(*batch)
    enc = tokenizer(
        list(prompts),
        max_length=MAX_IN_LEN,
        truncation=True,
        padding=True,
        return_tensors="pt",
        pad_to_multiple_of=8 if DEVICE.type == "cuda" else None,
    )
    enc["labels"] = torch.tensor(labels, dtype=torch.long)
    return enc

train_ds = T5Dataset(train_df)
val_ds = T5Dataset(val_df)
test_ds = T5Dataset(test_df)

train_labels = train_df["label"].astype(int).tolist()
class_counts = np.bincount(np.asarray(train_labels), minlength=2)
sample_weights = [1.0 / class_counts[y] for y in train_labels]
train_sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)

loader_kwargs = dict(
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    pin_memory=(DEVICE.type == "cuda"),
    persistent_workers=(NUM_WORKERS > 0),
    collate_fn=collate_t5,
)
if NUM_WORKERS > 0:
    loader_kwargs["prefetch_factor"] = 2

trn_loader = DataLoader(train_ds, sampler=train_sampler, shuffle=False, **loader_kwargs)
val_loader = DataLoader(val_ds, shuffle=False, **loader_kwargs)
tst_loader = DataLoader(test_ds, shuffle=False, **loader_kwargs)

# ── Optimizer / scheduler ────────────────────────────────────────────────
def get_class_weights(labels):
    labels = np.asarray(labels, dtype=int)
    classes = np.unique(labels)
    weights = compute_class_weight(class_weight="balanced", classes=classes, y=labels)
    out = np.ones(2, dtype=np.float32)
    for c, w in zip(classes, weights):
        out[int(c)] = float(w)
    return {0: out[0], 1: out[1]}

cw = get_class_weights(train_df["label"].tolist())
w_tensor = torch.tensor([cw[0], cw[1]], dtype=torch.float32, device=DEVICE)
print("Class weights:", cw)

criterion = nn.CrossEntropyLoss(weight=w_tensor, label_smoothing=LABEL_SMOOTHING)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=WEIGHT_DECAY
)

num_update_steps_per_epoch = math.ceil(len(trn_loader) / GRAD_ACCUM)
total_steps = num_update_steps_per_epoch * N_EPOCHS
warmup_steps = int(WARMUP_RATIO * total_steps)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)

scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE.type == "cuda"))

# ── Helper functions ──────────────────────────────────────────────────────
def save_json(path, payload):
    with open(path, "w") as f:
        json.dump(payload, f, indent=2)

def save_metrics(metrics, path):
    clean = {}
    for k, v in metrics.items():
        if isinstance(v, (np.floating, float)):
            clean[k] = float(v) if np.isfinite(v) else None
        elif isinstance(v, (np.integer, int)):
            clean[k] = int(v)
        else:
            clean[k] = v
    save_json(path, clean)

def best_threshold_for_f1(y_true, probs):
    thresholds = np.linspace(0.05, 0.95, 181)
    best_thr, best_f1 = 0.5, -1.0
    y_true = np.asarray(y_true, dtype=int)
    probs = np.asarray(probs, dtype=float)
    for thr in thresholds:
        preds = (probs >= thr).astype(int)
        f1 = f1_score(y_true, preds, average="macro", zero_division=0)
        if f1 > best_f1:
            best_f1 = f1
            best_thr = float(thr)
    return best_thr, float(best_f1)

def classifier_probs(model, input_ids, attention_mask):
    bs = input_ids.size(0)
    decoder_input_ids = torch.full(
        (bs, 1),
        tokenizer.pad_token_id,
        dtype=torch.long,
        device=input_ids.device
    )
    out = model(
        input_ids=input_ids,
        attention_mask=attention_mask,
        decoder_input_ids=decoder_input_ids
    )
    logits_0 = out.logits[:, 0, :]
    two_logits = logits_0[:, [NO_ID, YES_ID]]
    probs = torch.softmax(two_logits, dim=-1)[:, 1]
    preds = (probs >= 0.5).long()
    return probs, preds

def compute_hard_negative_accuracy(y_true, y_pred, abs_wc, concl_wc):
    y_true = np.asarray(y_true, dtype=int)
    y_pred = np.asarray(y_pred, dtype=int)
    abs_wc = np.asarray(abs_wc, dtype=int)
    concl_wc = np.asarray(concl_wc, dtype=int)

    mask = (y_true == 0) & (np.abs(abs_wc - concl_wc) < 50)
    idx = np.where(mask)[0]
    if len(idx) == 0:
        return {"hard_neg_count": 0, "hard_neg_accuracy": float("nan")}
    return {
        "hard_neg_count": int(len(idx)),
        "hard_neg_accuracy": float(accuracy_score(y_true[idx], y_pred[idx]))
    }

def compute_all_metrics(y_true, y_pred, y_prob):
    y_true = np.asarray(y_true, dtype=int)
    y_pred = np.asarray(y_pred, dtype=int)
    y_prob = np.asarray(y_prob, dtype=float)

    out = {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "precision_macro": float(precision_score(y_true, y_pred, average="macro", zero_division=0)),
        "recall_macro": float(recall_score(y_true, y_pred, average="macro", zero_division=0)),
        "f1_macro": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "precision_weighted": float(precision_score(y_true, y_pred, average="weighted", zero_division=0)),
        "recall_weighted": float(recall_score(y_true, y_pred, average="weighted", zero_division=0)),
        "f1_weighted": float(f1_score(y_true, y_pred, average="weighted", zero_division=0)),
    }

    rep = classification_report(y_true, y_pred, output_dict=True, zero_division=0)
    if "0" in rep:
        out["precision_class0"] = float(rep["0"]["precision"])
        out["recall_class0"] = float(rep["0"]["recall"])
        out["f1_class0"] = float(rep["0"]["f1-score"])
    if "1" in rep:
        out["precision_class1"] = float(rep["1"]["precision"])
        out["recall_class1"] = float(rep["1"]["recall"])
        out["f1_class1"] = float(rep["1"]["f1-score"])

    out["roc_auc"] = float(roc_auc_score(y_true, y_prob)) if len(np.unique(y_true)) > 1 else float("nan")
    out["pr_auc"] = float(average_precision_score(y_true, y_prob))
    out["log_loss"] = float(log_loss(y_true, np.column_stack([1 - y_prob, y_prob])))
    return out

def plot_confusion_matrix(y_true, y_pred, model_name, save_dir):
    cm = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(5, 4))
    sns.heatmap(
        cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax,
        xticklabels=["Diff (0)", "Same (1)"],
        yticklabels=["Diff (0)", "Same (1)"]
    )
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_title(f"{model_name} — Confusion Matrix")
    path = os.path.join(save_dir, f"{model_name}_confusion_matrix.png")
    plt.tight_layout()
    plt.savefig(path, dpi=160, bbox_inches="tight")
    plt.show()
    plt.close()
    print(f"Saved → {path}")

def plot_roc_curve(y_true, y_prob, model_name, save_dir):
    if len(np.unique(y_true)) < 2:
        print("ROC curve skipped (only one class present).")
        return
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    auc_score = roc_auc_score(y_true, y_prob)
    fig, ax = plt.subplots(figsize=(5, 4))
    ax.plot(fpr, tpr, lw=2, label=f"AUC={auc_score:.3f}")
    ax.plot([0, 1], [0, 1], "k--", lw=1)
    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.set_title(f"{model_name} — ROC Curve")
    ax.legend(loc="lower right")
    path = os.path.join(save_dir, f"{model_name}_roc_curve.png")
    plt.tight_layout()
    plt.savefig(path, dpi=160, bbox_inches="tight")
    plt.show()
    plt.close()
    print(f"Saved → {path}")

def plot_pr_curve(y_true, y_prob, model_name, save_dir):
    precision, recall, _ = precision_recall_curve(y_true, y_prob)
    ap = average_precision_score(y_true, y_prob)
    fig, ax = plt.subplots(figsize=(5, 4))
    ax.plot(recall, precision, lw=2, label=f"AP={ap:.3f}")
    ax.set_xlabel("Recall")
    ax.set_ylabel("Precision")
    ax.set_title(f"{model_name} — Precision-Recall Curve")
    ax.legend(loc="upper right")
    path = os.path.join(save_dir, f"{model_name}_pr_curve.png")
    plt.tight_layout()
    plt.savefig(path, dpi=160, bbox_inches="tight")
    plt.show()
    plt.close()
    print(f"Saved → {path}")

def plot_training_curves(train_losses, val_losses, train_accs, val_accs, model_name, save_dir):
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    epochs = range(1, len(train_losses) + 1)

    axes[0].plot(epochs, train_losses, "o-", label="Train")
    if val_losses:
        axes[0].plot(epochs, val_losses, "s--", label="Val")
    axes[0].set_title(f"{model_name} — Loss")
    axes[0].set_xlabel("Epoch")
    axes[0].legend()

    axes[1].plot(epochs, train_accs, "o-", label="Train")
    if val_accs:
        axes[1].plot(epochs, val_accs, "s--", label="Val")
    axes[1].set_title(f"{model_name} — Accuracy")
    axes[1].set_xlabel("Epoch")
    axes[1].legend()

    path = os.path.join(save_dir, f"{model_name}_training_curves.png")
    plt.tight_layout()
    plt.savefig(path, dpi=160, bbox_inches="tight")
    plt.show()
    plt.close()
    print(f"Saved → {path}")

def plot_subdomain_performance(y_true, y_pred, subdomains, model_name, save_dir):
    df_e = pd.DataFrame({"true": y_true, "pred": y_pred, "subdomain": subdomains})
    grp = df_e.groupby("subdomain").apply(
        lambda g: accuracy_score(g["true"], g["pred"]) if len(g) >= 3 else np.nan
    ).dropna().sort_values(ascending=False)

    if grp.empty:
        print("Not enough samples for subdomain plot.")
        return

    fig, ax = plt.subplots(figsize=(max(8, len(grp) * 0.75), 4))
    grp.plot(kind="bar", ax=ax)
    ax.axhline(accuracy_score(y_true, y_pred), color="red", ls="--", label="Overall")
    ax.set_ylabel("Accuracy")
    ax.set_title(f"{model_name} — Subdomain Accuracy")
    ax.legend()
    plt.xticks(rotation=45, ha="right")
    path = os.path.join(save_dir, f"{model_name}_subdomain_performance.png")
    plt.tight_layout()
    plt.savefig(path, dpi=160, bbox_inches="tight")
    plt.show()
    plt.close()
    print(f"Saved → {path}")

def plot_length_bucket_performance(y_true, y_pred, lengths, model_name, save_dir):
    lengths = np.asarray(lengths)
    bins = [0, 100, 200, 300, 400, 600, 100000]
    labels = ["<100", "100-200", "200-300", "300-400", "400-600", "600+"]
    buckets = pd.cut(lengths, bins=bins, labels=labels, right=False)

    df_e = pd.DataFrame({"true": y_true, "pred": y_pred, "bucket": buckets})
    grp = df_e.groupby("bucket", observed=True).apply(
        lambda g: accuracy_score(g["true"], g["pred"]) if len(g) >= 3 else np.nan
    ).dropna()

    if grp.empty:
        print("Not enough samples for length-bucket plot.")
        return

    fig, ax = plt.subplots(figsize=(8, 4))
    grp.plot(kind="bar", ax=ax)
    ax.set_ylim(0, 1)
    ax.set_ylabel("Accuracy")
    ax.set_title(f"{model_name} — Accuracy by Length Bucket")
    ax.set_xlabel("Word-count bucket")
    plt.xticks(rotation=30, ha="right")
    path = os.path.join(save_dir, f"{model_name}_length_bucket.png")
    plt.tight_layout()
    plt.savefig(path, dpi=160, bbox_inches="tight")
    plt.show()
    plt.close()
    print(f"Saved → {path}")

def save_label_map(root_dir):
    save_json(os.path.join(root_dir, "label_map.json"), LABEL_MAP)

def save_threshold(root_dir, threshold, val_f1):
    save_json(os.path.join(root_dir, "threshold.json"), {
        "threshold": float(threshold),
        "val_f1_macro": float(val_f1)
    })

def save_training_config(root_dir):
    cfg = {
        "pipeline_name": PIPELINE_NAME,
        "model_name": MODEL_NAME,
        "backbone": T5_CKPT,
        "max_abs_words": MAX_ABS_WORDS,
        "max_concl_words": MAX_CONCL_WORDS,
        "max_in_len": MAX_IN_LEN,
        "seed": SEED,
        "batch_size": BATCH_SIZE,
        "grad_accum": GRAD_ACCUM,
        "lr": LR,
        "weight_decay": WEIGHT_DECAY,
        "epochs": N_EPOCHS,
        "patience": PATIENCE,
        "max_grad_norm": MAX_GRAD_NORM,
        "label_smoothing": LABEL_SMOOTHING,
        "warmup_ratio": WARMUP_RATIO,
        "dropout": DROPOUT,
        "num_workers": NUM_WORKERS,
        "yes_token_id": YES_ID,
        "no_token_id": NO_ID,
    }
    save_json(os.path.join(root_dir, "training_config.json"), cfg)

def write_inference_script():
    inference_code = f'''import json
import os
import torch
from transformers import AutoTokenizer, T5ForConditionalGeneration

ROOT_DIR = os.path.dirname(os.path.abspath(__file__))
MODEL_DIR = os.path.join(ROOT_DIR, "model")
TOKENIZER_DIR = os.path.join(ROOT_DIR, "tokenizer")
THRESH_PATH = os.path.join(ROOT_DIR, "threshold.json")
LABEL_MAP_PATH = os.path.join(ROOT_DIR, "label_map.json")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def load_json(path, default):
    if os.path.exists(path):
        with open(path, "r") as f:
            return json.load(f)
    return default

tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_DIR, use_fast=True)
model = T5ForConditionalGeneration.from_pretrained(MODEL_DIR).to(DEVICE)
model.eval()

thr_payload = load_json(THRESH_PATH, {{"threshold": 0.5}})
threshold = float(thr_payload.get("threshold", 0.5))
label_map = load_json(LABEL_MAP_PATH, {{"0": "Different subdomain", "1": "Same subdomain"}})

NO_ID = {NO_ID}
YES_ID = {YES_ID}

def predict_pair(abstract, conclusion, max_len={MAX_IN_LEN}):
    prompt = (
        "classify whether the abstract and conclusion are in the same subdomain: "
        f"abstract: {{abstract}} conclusion: {{conclusion}}"
    )
    enc = tokenizer(
        prompt,
        max_length=max_len,
        truncation=True,
        padding="max_length",
        return_tensors="pt"
    )
    enc = {{k: v.to(DEVICE) for k, v in enc.items()}}
    decoder_input_ids = torch.full(
        (enc["input_ids"].size(0), 1),
        tokenizer.pad_token_id,
        dtype=torch.long,
        device=DEVICE
    )
    with torch.no_grad():
        out = model(
            input_ids=enc["input_ids"],
            attention_mask=enc["attention_mask"],
            decoder_input_ids=decoder_input_ids
        )
        step0 = out.logits[:, 0, :]
        two_logits = step0[:, [NO_ID, YES_ID]]
        prob_same = torch.softmax(two_logits, dim=-1)[0, 1].item()
    pred = int(prob_same >= threshold)
    return {{
        "label": pred,
        "label_name": label_map.get(str(pred), str(pred)),
        "probability_same_subdomain": prob_same,
        "threshold": threshold
    }}

if __name__ == "__main__":
    print(predict_pair("Sample abstract.", "Sample conclusion."))
'''
    with open(INFER_PATH, "w") as f:
        f.write(inference_code)
    print(f"✅ Inference script saved to {INFER_PATH}")

def zip_model_package():
    zip_base = os.path.join(MODELS_DIR, f"{MODEL_NAME}_package")
    zip_path = shutil.make_archive(zip_base, "zip", root_dir=MODELS_DIR, base_dir=MODEL_NAME)
    print(f"📦 Package archived → {zip_path}")
    return zip_path

@torch.no_grad()
def predict_loader(loader, threshold=0.5):
    model.eval()
    all_true, all_probs, all_preds = [], [], []
    total_loss, total_n = 0.0, 0

    for batch in loader:
        labels = batch.pop("labels").to(DEVICE, non_blocking=True)
        batch = {k: v.to(DEVICE, non_blocking=True) for k, v in batch.items()}

        with torch.cuda.amp.autocast(enabled=(DEVICE.type == "cuda")):
            decoder_input_ids = torch.full(
                (batch["input_ids"].size(0), 1),
                tokenizer.pad_token_id,
                dtype=torch.long,
                device=DEVICE
            )
            out = model(
                input_ids=batch["input_ids"],
                attention_mask=batch["attention_mask"],
                decoder_input_ids=decoder_input_ids
            )
            step0 = out.logits[:, 0, :]
            logits = step0[:, [NO_ID, YES_ID]]
            loss = criterion(logits, labels)

        probs = torch.softmax(logits, dim=-1)[:, 1]
        preds = (probs >= threshold).long()

        bs = labels.size(0)
        total_loss += loss.item() * bs
        total_n += bs

        all_true.extend(labels.detach().cpu().tolist())
        all_probs.extend(probs.detach().cpu().tolist())
        all_preds.extend(preds.detach().cpu().tolist())

    return all_true, all_probs, all_preds, total_loss / max(total_n, 1)

# ── Training ───────────────────────────────────────────────────────────────
best_val_f1 = 0.0
best_val_threshold = 0.5
patience_ctr = 0
train_losses, val_losses, train_accs, val_accs = [], [], [], []
best_ckpt_path = BEST_CKPT

t0 = time.time()
print(f"\nTraining {MODEL_NAME} …")

for epoch in range(1, N_EPOCHS + 1):
    model.train()
    ep_loss, ep_correct, ep_total = 0.0, 0, 0
    optimizer.zero_grad(set_to_none=True)

    for step, batch in enumerate(trn_loader, start=1):
        labels = batch.pop("labels").to(DEVICE, non_blocking=True)
        batch = {k: v.to(DEVICE, non_blocking=True) for k, v in batch.items()}

        with torch.cuda.amp.autocast(enabled=(DEVICE.type == "cuda")):
            decoder_input_ids = torch.full(
                (batch["input_ids"].size(0), 1),
                tokenizer.pad_token_id,
                dtype=torch.long,
                device=DEVICE
            )
            out = model(
                input_ids=batch["input_ids"],
                attention_mask=batch["attention_mask"],
                decoder_input_ids=decoder_input_ids
            )
            step0 = out.logits[:, 0, :]
            logits = step0[:, [NO_ID, YES_ID]]
            loss = criterion(logits, labels) / GRAD_ACCUM

        scaler.scale(loss).backward()

        ep_loss += loss.item() * GRAD_ACCUM
        preds = logits.argmax(dim=-1)
        ep_correct += (preds == labels).sum().item()
        ep_total += labels.size(0)

        do_step = (step % GRAD_ACCUM == 0) or (step == len(trn_loader))
        if do_step:
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            scheduler.step()

    train_loss = ep_loss / max(len(trn_loader), 1)
    train_acc = ep_correct / max(ep_total, 1)
    train_losses.append(train_loss)
    train_accs.append(train_acc)

    y_val, p_val, pred_val_05, v_loss = predict_loader(val_loader, threshold=0.5)
    val_acc_05 = accuracy_score(y_val, pred_val_05)
    val_f1_05 = f1_score(y_val, pred_val_05, average="macro", zero_division=0)

    val_thr, val_f1_thr = best_threshold_for_f1(y_val, p_val)
    val_preds_best = (np.asarray(p_val) >= val_thr).astype(int)
    val_acc_best = accuracy_score(y_val, val_preds_best)

    val_losses.append(v_loss)
    val_accs.append(val_acc_best)

    if val_f1_thr > best_val_f1:
        best_val_f1 = val_f1_thr
        best_val_threshold = val_thr
        patience_ctr = 0
        torch.save(model.state_dict(), best_ckpt_path)
    else:
        patience_ctr += 1

    print(
        f"Epoch {epoch:02d} | "
        f"train_loss={train_loss:.4f} | train_acc={train_acc:.3f} | "
        f"val_loss={v_loss:.4f} | val_acc@0.5={val_acc_05:.3f} | "
        f"val_f1@0.5={val_f1_05:.4f} | val_f1@best_thr={val_f1_thr:.4f} | "
        f"thr={val_thr:.3f} | patience={patience_ctr}"
    )

    if patience_ctr >= PATIENCE:
        print(f"Early stopping at epoch {epoch}.")
        break

    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

train_time = time.time() - t0
print(f"\nTraining time: {train_time:.1f} seconds")
print(f"Best validation threshold: {best_val_threshold:.3f}")

# ── Load best checkpoint ───────────────────────────────────────────────────
model.load_state_dict(torch.load(best_ckpt_path, map_location=DEVICE))
model.eval()

# ── Test evaluation ────────────────────────────────────────────────────────
y_true, probs, raw_preds, _ = predict_loader(tst_loader, threshold=best_val_threshold)
preds = (np.asarray(probs) >= best_val_threshold).astype(int)

metrics = compute_all_metrics(y_true, preds, probs)
metrics.update(compute_hard_negative_accuracy(
    y_true, preds,
    test_df["abs_wc"].tolist(),
    test_df["concl_wc"].tolist()
))
metrics["training_time_s"] = float(train_time)
metrics["best_val_threshold"] = float(best_val_threshold)
metrics["best_val_f1_macro"] = float(best_val_f1)

print("\nFinal metrics:")
for k, v in metrics.items():
    try:
        print(f"{k:<30s}: {v:.4f}")
    except Exception:
        print(f"{k:<30s}: {v}")

# ── Plots ──────────────────────────────────────────────────────────────────
plot_confusion_matrix(y_true, preds, MODEL_NAME, RESULTS_DIR)
plot_roc_curve(y_true, probs, MODEL_NAME, RESULTS_DIR)
plot_pr_curve(y_true, probs, MODEL_NAME, RESULTS_DIR)
plot_training_curves(train_losses, val_losses, train_accs, val_accs, MODEL_NAME, RESULTS_DIR)

if "abstract_subdomain" in test_df.columns:
    plot_subdomain_performance(
        y_true, preds,
        test_df["abstract_subdomain"].fillna("unknown").astype(str).tolist(),
        MODEL_NAME, RESULTS_DIR
    )

plot_length_bucket_performance(
    y_true, preds,
    (test_df["abs_wc"].fillna(0) + test_df["concl_wc"].fillna(0)).tolist(),
    MODEL_NAME, RESULTS_DIR
)

# ── Save artifacts ─────────────────────────────────────────────────────────
save_metrics(metrics, METRICS_PATH)
print(f"✅ Metrics saved to {METRICS_PATH}")

pred_out = pd.DataFrame({
    "y_true": y_true,
    "prob_1": probs,
    "pred": preds
})
pred_path = os.path.join(T5_DIR, "test_predictions.csv")
pred_out.to_csv(pred_path, index=False)
print(f"✅ Test predictions saved to {pred_path}")

save_label_map(T5_DIR)
save_threshold(T5_DIR, best_val_threshold, best_val_f1)
save_training_config(T5_DIR)
write_inference_script()

model.save_pretrained(MODEL_DIR)
tokenizer.save_pretrained(TOKENIZER_DIR)
print(f"✅ Saved model to {MODEL_DIR}")
print(f"✅ Saved tokenizer to {TOKENIZER_DIR}")

zip_model_package()
ALL_RESULTS[MODEL_NAME] = metrics

print("\n✅ T5 complete.")
print(f"Elapsed time: {train_time/60:.1f} minutes")
print(f"Best checkpoint: {BEST_CKPT}")
print(f"Archive: {PACKAGE_ZIP}")

In [ ]:
# ── Cell 12: BART training (high-performance, T4-heavy version) ────────────
# Direct sequence classification, no generation.
# Usually much faster and more stable for binary pair classification.

import os
import gc
import json
import time
import math
import random
import warnings
from collections import defaultdict

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

from transformers import AutoTokenizer, BartForSequenceClassification, get_linear_schedule_with_warmup
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, precision_recall_curve,
    confusion_matrix, roc_curve, classification_report, log_loss, auc
)
from sklearn.utils.class_weight import compute_class_weight

import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")

# ── Reproducibility & speed ────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
try:
    torch.set_float32_matmul_precision("high")
except Exception:
    pass

os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# ── Config ────────────────────────────────────────────────────────────────
BASE_DIR = "/content"
MODELS_DIR = globals().get("MODELS_DIR", os.path.join(BASE_DIR, "saved_models"))
PROCESSED_DIR = globals().get("PROCESSED_DIR", os.path.join(BASE_DIR, "processed_splits"))

MODEL_NAME = "BART"
BART_CKPT = "facebook/bart-base"

MAX_LEN = 512
BATCH_SIZE = 6
GRAD_ACCUM = 4
LR = 2e-5
HEAD_LR = 7e-5
WEIGHT_DECAY = 0.01
N_EPOCHS = 12
PATIENCE = 4
MAX_GRAD_NORM = 1.0
LABEL_SMOOTHING = 0.01
NUM_WORKERS = 2
WARMUP_RATIO = 0.10
DROPOUT = 0.20

PIPELINE_NAME = "bart_pair_classifier"

BART_DIR = os.path.join(MODELS_DIR, MODEL_NAME)
MODEL_DIR = os.path.join(BART_DIR, "model")
TOKENIZER_DIR = os.path.join(BART_DIR, "tokenizer")
RESULTS_DIR = os.path.join(BART_DIR, "results")
CACHE_DIR = os.path.join(BART_DIR, "cache")
BEST_CKPT = os.path.join(BART_DIR, "best_model.pt")
METRICS_PATH = os.path.join(BART_DIR, "metrics.json")
LABEL_MAP_PATH = os.path.join(BART_DIR, "label_map.json")
THRESH_PATH = os.path.join(BART_DIR, "threshold.json")
INFER_PATH = os.path.join(BART_DIR, "inference.py")
CONFIG_PATH = os.path.join(BART_DIR, "training_config.json")
PACKAGE_ZIP = os.path.join(MODELS_DIR, f"{MODEL_NAME}_package.zip")

for d in [BART_DIR, MODEL_DIR, TOKENIZER_DIR, RESULTS_DIR, CACHE_DIR]:
    os.makedirs(d, exist_ok=True)

ALL_RESULTS = globals().get("ALL_RESULTS", {})

print(f"Root      : {BART_DIR}")
print(f"Model     : {MODEL_DIR}")
print(f"Tokenizer : {TOKENIZER_DIR}")
print(f"Results   : {RESULTS_DIR}")
print(f"Archive   : {PACKAGE_ZIP}")

# ── Load data ─────────────────────────────────────────────────────────────
train_df = pd.read_csv(os.path.join(PROCESSED_DIR, "train.csv"))
val_df   = pd.read_csv(os.path.join(PROCESSED_DIR, "val.csv"))
test_df  = pd.read_csv(os.path.join(PROCESSED_DIR, "test.csv"))

print(f"Loaded splits — train:{len(train_df)}  val:{len(val_df)}  test:{len(test_df)}")

def resolve_text_columns(df):
    abs_candidates = ["abstract_clean", "abstract_raw", "abstract_model", "abstract"]
    con_candidates = ["conclusion_clean", "conclusion_raw", "conclusion_model", "conclusion"]
    abs_col = next((c for c in abs_candidates if c in df.columns), None)
    con_col = next((c for c in con_candidates if c in df.columns), None)
    if abs_col is None or con_col is None:
        raise ValueError("Could not find abstract/conclusion columns in the split files.")
    return abs_col, con_col

ABS_COL, CONCL_COL = resolve_text_columns(train_df)
print(f"Using text columns: {ABS_COL}, {CONCL_COL}")

MAX_ABS_WORDS = 300
MAX_CONCL_WORDS = 300

def clean_text(text: str, max_words: int = None) -> str:
    if not isinstance(text, str) or not text.strip():
        return ""
    text = " ".join(text.replace("\n", " ").split())
    if max_words and max_words > 0:
        words = text.split()
        if len(words) > max_words:
            text = " ".join(words[:max_words])
    return text

for split_name, split_df in [("train", train_df), ("val", val_df), ("test", test_df)]:
    split_df[ABS_COL] = split_df[ABS_COL].apply(lambda x: clean_text(x, MAX_ABS_WORDS))
    split_df[CONCL_COL] = split_df[CONCL_COL].apply(lambda x: clean_text(x, MAX_CONCL_WORDS))
    split_df.dropna(subset=["label"], inplace=True)
    split_df["label"] = split_df["label"].astype(int)

    before = len(split_df)
    split_df = split_df[(split_df[ABS_COL].str.len() > 0) & (split_df[CONCL_COL].str.len() > 0)].copy()
    split_df["abs_wc"] = split_df[ABS_COL].str.split().str.len()
    split_df["concl_wc"] = split_df[CONCL_COL].str.split().str.len()
    split_df = split_df[(split_df["abs_wc"] <= 1000) & (split_df["concl_wc"] <= 1000)].copy()

    if split_name == "train":
        train_df = split_df.reset_index(drop=True)
    elif split_name == "val":
        val_df = split_df.reset_index(drop=True)
    else:
        test_df = split_df.reset_index(drop=True)

    print(f"{split_name}: dropped {before - len(split_df):,} rows; remaining {len(split_df):,}")

for split_df in [train_df, val_df, test_df]:
    split_df["input_text_clean"] = "Abstract: " + split_df[ABS_COL] + " [SEP] Conclusion: " + split_df[CONCL_COL]

# ── Tokenizer / model ──────────────────────────────────────────────────────
print(f"Loading backbone: {BART_CKPT}")
tokenizer = AutoTokenizer.from_pretrained(BART_CKPT, use_fast=True)

model = BartForSequenceClassification.from_pretrained(
    BART_CKPT,
    num_labels=2,
    classifier_dropout=DROPOUT
)

model.to(DEVICE)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable params: {trainable:,} / {total:,}")

if DEVICE.type == "cuda":
    torch.cuda.empty_cache()
gc.collect()

# ── Dataset / collator ─────────────────────────────────────────────────────
class PairDataset(Dataset):
    def __init__(self, df, abs_col, concl_col):
        self.abstracts = df[abs_col].astype(str).tolist()
        self.conclusions = df[concl_col].astype(str).tolist()
        self.labels = df["label"].astype(int).tolist()

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, i):
        return self.abstracts[i], self.conclusions[i], self.labels[i]

def collate_fn(batch):
    abstracts, conclusions, labels = zip(*batch)
    enc = tokenizer(
        list(abstracts),
        list(conclusions),
        max_length=MAX_LEN,
        truncation=True,
        padding=True,
        return_tensors="pt",
        pad_to_multiple_of=8 if DEVICE.type == "cuda" else None,
    )
    enc["labels"] = torch.tensor(labels, dtype=torch.long)
    return enc

train_ds = PairDataset(train_df, ABS_COL, CONCL_COL)
val_ds = PairDataset(val_df, ABS_COL, CONCL_COL)
test_ds = PairDataset(test_df, ABS_COL, CONCL_COL)

train_labels = train_df["label"].astype(int).tolist()
class_counts = np.bincount(np.asarray(train_labels), minlength=2)
sample_weights = [1.0 / class_counts[y] for y in train_labels]
train_sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)

loader_kwargs = dict(
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    pin_memory=(DEVICE.type == "cuda"),
    persistent_workers=(NUM_WORKERS > 0),
    collate_fn=collate_fn,
)
if NUM_WORKERS > 0:
    loader_kwargs["prefetch_factor"] = 2

trn_loader = DataLoader(train_ds, sampler=train_sampler, shuffle=False, **loader_kwargs)
val_loader = DataLoader(val_ds, shuffle=False, **loader_kwargs)
tst_loader = DataLoader(test_ds, shuffle=False, **loader_kwargs)

# ── Loss / optimizer / scheduler ───────────────────────────────────────────
def get_class_weights(labels):
    labels = np.asarray(labels, dtype=int)
    classes = np.unique(labels)
    weights = compute_class_weight(class_weight="balanced", classes=classes, y=labels)
    out = np.ones(2, dtype=np.float32)
    for c, w in zip(classes, weights):
        out[int(c)] = float(w)
    return {0: out[0], 1: out[1]}

cw = get_class_weights(train_df["label"].tolist())
w_tensor = torch.tensor([cw[0], cw[1]], dtype=torch.float32, device=DEVICE)
print("Class weights:", cw)

criterion = nn.CrossEntropyLoss(weight=w_tensor, label_smoothing=LABEL_SMOOTHING)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=WEIGHT_DECAY
)

num_update_steps_per_epoch = math.ceil(len(trn_loader) / GRAD_ACCUM)
total_steps = num_update_steps_per_epoch * N_EPOCHS
warmup_steps = int(WARMUP_RATIO * total_steps)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)

scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE.type == "cuda"))

# ── Helper functions ──────────────────────────────────────────────────────
LABEL2TEXT = {0: "different_subdomain", 1: "same_subdomain"}

def save_json(path, payload):
    with open(path, "w") as f:
        json.dump(payload, f, indent=2)

def save_metrics(metrics, path):
    clean = {}
    for k, v in metrics.items():
        if isinstance(v, (np.floating, float)):
            clean[k] = float(v) if np.isfinite(v) else None
        elif isinstance(v, (np.integer, int)):
            clean[k] = int(v)
        else:
            clean[k] = v
    save_json(path, clean)

def best_threshold_for_f1(y_true, probs):
    thresholds = np.linspace(0.05, 0.95, 181)
    best_thr, best_f1 = 0.5, -1.0
    y_true = np.asarray(y_true, dtype=int)
    probs = np.asarray(probs, dtype=float)
    for thr in thresholds:
        preds = (probs >= thr).astype(int)
        f1 = f1_score(y_true, preds, average="macro", zero_division=0)
        if f1 > best_f1:
            best_f1 = f1
            best_thr = float(thr)
    return best_thr, float(best_f1)

def compute_hard_negative_accuracy(y_true, y_pred, abs_wc, concl_wc):
    y_true = np.asarray(y_true, dtype=int)
    y_pred = np.asarray(y_pred, dtype=int)
    abs_wc = np.asarray(abs_wc, dtype=int)
    concl_wc = np.asarray(concl_wc, dtype=int)

    mask = (y_true == 0) & (np.abs(abs_wc - concl_wc) < 50)
    idx = np.where(mask)[0]
    if len(idx) == 0:
        return {"hard_neg_count": 0, "hard_neg_accuracy": float("nan")}
    return {
        "hard_neg_count": int(len(idx)),
        "hard_neg_accuracy": float(accuracy_score(y_true[idx], y_pred[idx]))
    }

def compute_all_metrics(y_true, y_pred, y_prob):
    y_true = np.asarray(y_true, dtype=int)
    y_pred = np.asarray(y_pred, dtype=int)
    y_prob = np.asarray(y_prob, dtype=float)

    out = {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "precision_macro": float(precision_score(y_true, y_pred, average="macro", zero_division=0)),
        "recall_macro": float(recall_score(y_true, y_pred, average="macro", zero_division=0)),
        "f1_macro": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "precision_weighted": float(precision_score(y_true, y_pred, average="weighted", zero_division=0)),
        "recall_weighted": float(recall_score(y_true, y_pred, average="weighted", zero_division=0)),
        "f1_weighted": float(f1_score(y_true, y_pred, average="weighted", zero_division=0)),
    }

    rep = classification_report(y_true, y_pred, output_dict=True, zero_division=0)
    if "0" in rep:
        out["precision_class0"] = float(rep["0"]["precision"])
        out["recall_class0"] = float(rep["0"]["recall"])
        out["f1_class0"] = float(rep["0"]["f1-score"])
    if "1" in rep:
        out["precision_class1"] = float(rep["1"]["precision"])
        out["recall_class1"] = float(rep["1"]["recall"])
        out["f1_class1"] = float(rep["1"]["f1-score"])

    out["roc_auc"] = float(roc_auc_score(y_true, y_prob)) if len(np.unique(y_true)) > 1 else float("nan")
    out["pr_auc"] = float(average_precision_score(y_true, y_prob))
    out["log_loss"] = float(log_loss(y_true, np.column_stack([1 - y_prob, y_prob])))
    return out

def plot_confusion_matrix(y_true, y_pred, model_name, save_dir):
    cm = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(5, 4))
    sns.heatmap(
        cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax,
        xticklabels=["Diff (0)", "Same (1)"],
        yticklabels=["Diff (0)", "Same (1)"]
    )
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_title(f"{model_name} — Confusion Matrix")
    path = os.path.join(save_dir, f"{model_name}_confusion_matrix.png")
    plt.tight_layout()
    plt.savefig(path, dpi=160, bbox_inches="tight")
    plt.show()
    plt.close()
    print(f"Saved → {path}")

def plot_roc_curve(y_true, y_prob, model_name, save_dir):
    if len(np.unique(y_true)) < 2:
        print("ROC curve skipped (only one class present).")
        return
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    auc_score = roc_auc_score(y_true, y_prob)
    fig, ax = plt.subplots(figsize=(5, 4))
    ax.plot(fpr, tpr, lw=2, label=f"AUC={auc_score:.3f}")
    ax.plot([0, 1], [0, 1], "k--", lw=1)
    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.set_title(f"{model_name} — ROC Curve")
    ax.legend(loc="lower right")
    path = os.path.join(save_dir, f"{model_name}_roc_curve.png")
    plt.tight_layout()
    plt.savefig(path, dpi=160, bbox_inches="tight")
    plt.show()
    plt.close()
    print(f"Saved → {path}")

def plot_pr_curve(y_true, y_prob, model_name, save_dir):
    precision, recall, _ = precision_recall_curve(y_true, y_prob)
    ap = average_precision_score(y_true, y_prob)
    fig, ax = plt.subplots(figsize=(5, 4))
    ax.plot(recall, precision, lw=2, label=f"AP={ap:.3f}")
    ax.set_xlabel("Recall")
    ax.set_ylabel("Precision")
    ax.set_title(f"{model_name} — Precision-Recall Curve")
    ax.legend(loc="upper right")
    path = os.path.join(save_dir, f"{model_name}_pr_curve.png")
    plt.tight_layout()
    plt.savefig(path, dpi=160, bbox_inches="tight")
    plt.show()
    plt.close()
    print(f"Saved → {path}")

def plot_training_curves(train_losses, val_losses, train_accs, val_accs, model_name, save_dir):
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    epochs = range(1, len(train_losses) + 1)

    axes[0].plot(epochs, train_losses, "o-", label="Train")
    if val_losses:
        axes[0].plot(epochs, val_losses, "s--", label="Val")
    axes[0].set_title(f"{model_name} — Loss")
    axes[0].set_xlabel("Epoch")
    axes[0].legend()

    axes[1].plot(epochs, train_accs, "o-", label="Train")
    if val_accs:
        axes[1].plot(epochs, val_accs, "s--", label="Val")
    axes[1].set_title(f"{model_name} — Accuracy")
    axes[1].set_xlabel("Epoch")
    axes[1].legend()

    path = os.path.join(save_dir, f"{model_name}_training_curves.png")
    plt.tight_layout()
    plt.savefig(path, dpi=160, bbox_inches="tight")
    plt.show()
    plt.close()
    print(f"Saved → {path}")

def plot_subdomain_performance(y_true, y_pred, subdomains, model_name, save_dir):
    df_e = pd.DataFrame({"true": y_true, "pred": y_pred, "subdomain": subdomains})
    grp = df_e.groupby("subdomain").apply(
        lambda g: accuracy_score(g["true"], g["pred"]) if len(g) >= 3 else np.nan
    ).dropna().sort_values(ascending=False)

    if grp.empty:
        print("Not enough samples for subdomain plot.")
        return

    fig, ax = plt.subplots(figsize=(max(8, len(grp) * 0.75), 4))
    grp.plot(kind="bar", ax=ax)
    ax.axhline(accuracy_score(y_true, y_pred), color="red", ls="--", label="Overall")
    ax.set_ylabel("Accuracy")
    ax.set_title(f"{model_name} — Subdomain Accuracy")
    ax.legend()
    plt.xticks(rotation=45, ha="right")
    path = os.path.join(save_dir, f"{model_name}_subdomain_performance.png")
    plt.tight_layout()
    plt.savefig(path, dpi=160, bbox_inches="tight")
    plt.show()
    plt.close()
    print(f"Saved → {path}")

def plot_length_bucket_performance(y_true, y_pred, lengths, model_name, save_dir):
    lengths = np.asarray(lengths)
    bins = [0, 100, 200, 300, 400, 600, 100000]
    labels = ["<100", "100-200", "200-300", "300-400", "400-600", "600+"]
    buckets = pd.cut(lengths, bins=bins, labels=labels, right=False)

    df_e = pd.DataFrame({"true": y_true, "pred": y_pred, "bucket": buckets})
    grp = df_e.groupby("bucket", observed=True).apply(
        lambda g: accuracy_score(g["true"], g["pred"]) if len(g) >= 3 else np.nan
    ).dropna()

    if grp.empty:
        print("Not enough samples for length-bucket plot.")
        return

    fig, ax = plt.subplots(figsize=(8, 4))
    grp.plot(kind="bar", ax=ax)
    ax.set_ylim(0, 1)
    ax.set_ylabel("Accuracy")
    ax.set_title(f"{model_name} — Accuracy by Length Bucket")
    ax.set_xlabel("Word-count bucket")
    plt.xticks(rotation=30, ha="right")
    path = os.path.join(save_dir, f"{model_name}_length_bucket.png")
    plt.tight_layout()
    plt.savefig(path, dpi=160, bbox_inches="tight")
    plt.show()
    plt.close()
    print(f"Saved → {path}")

def save_label_map(root_dir):
    save_json(os.path.join(root_dir, "label_map.json"), LABEL2TEXT)

def save_threshold(root_dir, threshold, val_f1):
    save_json(os.path.join(root_dir, "threshold.json"), {
        "threshold": float(threshold),
        "val_f1_macro": float(val_f1)
    })

def save_training_config(root_dir):
    cfg = {
        "pipeline_name": PIPELINE_NAME,
        "model_name": MODEL_NAME,
        "backbone": BART_CKPT,
        "max_abs_words": MAX_ABS_WORDS,
        "max_concl_words": MAX_CONCL_WORDS,
        "max_len": MAX_LEN,
        "seed": SEED,
        "batch_size": BATCH_SIZE,
        "grad_accum": GRAD_ACCUM,
        "lr": LR,
        "head_lr": HEAD_LR,
        "weight_decay": WEIGHT_DECAY,
        "epochs": N_EPOCHS,
        "patience": PATIENCE,
        "max_grad_norm": MAX_GRAD_NORM,
        "label_smoothing": LABEL_SMOOTHING,
        "warmup_ratio": WARMUP_RATIO,
        "dropout": DROPOUT,
        "num_workers": NUM_WORKERS,
    }
    save_json(os.path.join(root_dir, "training_config.json"), cfg)

def write_inference_script():
    inference_code = f'''import json
import os
import torch
from transformers import AutoTokenizer, BartForSequenceClassification

ROOT_DIR = os.path.dirname(os.path.abspath(__file__))
MODEL_DIR = os.path.join(ROOT_DIR, "model")
TOKENIZER_DIR = os.path.join(ROOT_DIR, "tokenizer")
THRESH_PATH = os.path.join(ROOT_DIR, "threshold.json")
LABEL_MAP_PATH = os.path.join(ROOT_DIR, "label_map.json")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def load_json(path, default):
    if os.path.exists(path):
        with open(path, "r") as f:
            return json.load(f)
    return default

tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_DIR, use_fast=True)
model = BartForSequenceClassification.from_pretrained(MODEL_DIR).to(DEVICE)
model.eval()

thr_payload = load_json(THRESH_PATH, {{"threshold": 0.5}})
threshold = float(thr_payload.get("threshold", 0.5))
label_map = load_json(LABEL_MAP_PATH, {{"0": "Different subdomain", "1": "Same subdomain"}})

def predict_pair(abstract, conclusion, max_len={MAX_LEN}):
    prompt = f"Abstract: {{abstract}} [SEP] Conclusion: {{conclusion}}"
    enc = tokenizer(
        prompt,
        max_length=max_len,
        truncation=True,
        padding="max_length",
        return_tensors="pt"
    )
    enc = {{k: v.to(DEVICE) for k, v in enc.items()}}
    with torch.no_grad():
        logits = model(**enc).logits
        prob_1 = torch.softmax(logits, dim=-1)[0, 1].item()
    pred = int(prob_1 >= threshold)
    return {{
        "label": pred,
        "label_name": label_map.get(str(pred), str(pred)),
        "probability_same_subdomain": prob_1,
        "threshold": threshold
    }}

if __name__ == "__main__":
    print(predict_pair("Sample abstract.", "Sample conclusion."))
'''
    with open(INFER_PATH, "w") as f:
        f.write(inference_code)
    print(f"✅ Inference script saved to {INFER_PATH}")

def zip_model_package():
    zip_base = os.path.join(MODELS_DIR, f"{MODEL_NAME}_package")
    zip_path = shutil.make_archive(zip_base, "zip", root_dir=MODELS_DIR, base_dir=MODEL_NAME)
    print(f"📦 Package archived → {zip_path}")
    return zip_path

@torch.no_grad()
def predict_loader(loader, threshold=0.5):
    model.eval()
    all_true, all_probs, all_preds = [], [], []
    total_loss, total_n = 0.0, 0

    for batch in loader:
        labels = batch.pop("labels").to(DEVICE, non_blocking=True)
        batch = {k: v.to(DEVICE, non_blocking=True) for k, v in batch.items()}

        with torch.cuda.amp.autocast(enabled=(DEVICE.type == "cuda")):
            logits = model(**batch).logits
            loss = criterion(logits, labels)

        probs = torch.softmax(logits, dim=-1)[:, 1]
        preds = (probs >= threshold).long()

        bs = labels.size(0)
        total_loss += loss.item() * bs
        total_n += bs

        all_true.extend(labels.detach().cpu().tolist())
        all_probs.extend(probs.detach().cpu().tolist())
        all_preds.extend(preds.detach().cpu().tolist())

    return all_true, all_probs, all_preds, total_loss / max(total_n, 1)

# ── Training ───────────────────────────────────────────────────────────────
best_val_f1 = 0.0
best_val_threshold = 0.5
patience_ctr = 0
train_losses, val_losses, train_accs, val_accs = [], [], [], []
best_ckpt_path = BEST_CKPT

t0 = time.time()
print(f"\nTraining {MODEL_NAME} …")

for epoch in range(1, N_EPOCHS + 1):
    model.train()
    ep_loss, ep_correct, ep_total = 0.0, 0, 0
    optimizer.zero_grad(set_to_none=True)

    for step, batch in enumerate(trn_loader, start=1):
        labels = batch.pop("labels").to(DEVICE, non_blocking=True)
        batch = {k: v.to(DEVICE, non_blocking=True) for k, v in batch.items()}

        with torch.cuda.amp.autocast(enabled=(DEVICE.type == "cuda")):
            logits = model(**batch).logits
            loss = criterion(logits, labels) / GRAD_ACCUM

        scaler.scale(loss).backward()

        ep_loss += loss.item() * GRAD_ACCUM
        preds = logits.argmax(dim=-1)
        ep_correct += (preds == labels).sum().item()
        ep_total += labels.size(0)

        do_step = (step % GRAD_ACCUM == 0) or (step == len(trn_loader))
        if do_step:
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            scheduler.step()

    train_loss = ep_loss / max(len(trn_loader), 1)
    train_acc = ep_correct / max(ep_total, 1)
    train_losses.append(train_loss)
    train_accs.append(train_acc)

    y_val, p_val, pred_val_05, v_loss = predict_loader(val_loader, threshold=0.5)
    val_acc_05 = accuracy_score(y_val, pred_val_05)
    val_f1_05 = f1_score(y_val, pred_val_05, average="macro", zero_division=0)

    val_thr, val_f1_thr = best_threshold_for_f1(y_val, p_val)
    val_preds_best = (np.asarray(p_val) >= val_thr).astype(int)
    val_acc_best = accuracy_score(y_val, val_preds_best)

    val_losses.append(v_loss)
    val_accs.append(val_acc_best)

    if val_f1_thr > best_val_f1:
        best_val_f1 = val_f1_thr
        best_val_threshold = val_thr
        patience_ctr = 0
        torch.save(model.state_dict(), best_ckpt_path)
    else:
        patience_ctr += 1

    print(
        f"Epoch {epoch:02d} | "
        f"train_loss={train_loss:.4f} | train_acc={train_acc:.3f} | "
        f"val_loss={v_loss:.4f} | val_acc@0.5={val_acc_05:.3f} | "
        f"val_f1@0.5={val_f1_05:.4f} | val_f1@best_thr={val_f1_thr:.4f} | "
        f"thr={val_thr:.3f} | patience={patience_ctr}"
    )

    if patience_ctr >= PATIENCE:
        print(f"Early stopping at epoch {epoch}.")
        break

    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

train_time = time.time() - t0
print(f"\nTraining time: {train_time:.1f} seconds")
print(f"Best validation threshold: {best_val_threshold:.3f}")

# ── Load best checkpoint ───────────────────────────────────────────────────
model.load_state_dict(torch.load(best_ckpt_path, map_location=DEVICE))
model.eval()

# ── Test evaluation ────────────────────────────────────────────────────────
y_true, probs, raw_preds, _ = predict_loader(tst_loader, threshold=best_val_threshold)
preds = (np.asarray(probs) >= best_val_threshold).astype(int)

metrics = compute_all_metrics(y_true, preds, probs)
metrics.update(compute_hard_negative_accuracy(
    y_true, preds,
    test_df["abs_wc"].tolist(),
    test_df["concl_wc"].tolist()
))
metrics["training_time_s"] = float(train_time)
metrics["best_val_threshold"] = float(best_val_threshold)
metrics["best_val_f1_macro"] = float(best_val_f1)

print("\nFinal metrics:")
for k, v in metrics.items():
    try:
        print(f"{k:<30s}: {v:.4f}")
    except Exception:
        print(f"{k:<30s}: {v}")

# ── Plots ──────────────────────────────────────────────────────────────────
plot_confusion_matrix(y_true, preds, MODEL_NAME, RESULTS_DIR)
plot_roc_curve(y_true, probs, MODEL_NAME, RESULTS_DIR)
plot_pr_curve(y_true, probs, MODEL_NAME, RESULTS_DIR)
plot_training_curves(train_losses, val_losses, train_accs, val_accs, MODEL_NAME, RESULTS_DIR)

if "abstract_subdomain" in test_df.columns:
    plot_subdomain_performance(
        y_true, preds,
        test_df["abstract_subdomain"].fillna("unknown").astype(str).tolist(),
        MODEL_NAME, RESULTS_DIR
    )

plot_length_bucket_performance(
    y_true, preds,
    (test_df["abs_wc"].fillna(0) + test_df["concl_wc"].fillna(0)).tolist(),
    MODEL_NAME, RESULTS_DIR
)

# ── Save artifacts ─────────────────────────────────────────────────────────
save_metrics(metrics, METRICS_PATH)
print(f"✅ Metrics saved to {METRICS_PATH}")

pred_out = pd.DataFrame({
    "y_true": y_true,
    "prob_1": probs,
    "pred": preds
})
pred_path = os.path.join(BART_DIR, "test_predictions.csv")
pred_out.to_csv(pred_path, index=False)
print(f"✅ Test predictions saved to {pred_path}")

save_label_map(BART_DIR)
save_threshold(BART_DIR, best_val_threshold, best_val_f1)
save_training_config(BART_DIR)
write_inference_script()

model.save_pretrained(MODEL_DIR)
tokenizer.save_pretrained(TOKENIZER_DIR)
print(f"✅ Saved model to {MODEL_DIR}")
print(f"✅ Saved tokenizer to {TOKENIZER_DIR}")

zip_model_package()
ALL_RESULTS[MODEL_NAME] = metrics

print("\n✅ BART complete.")
print(f"Elapsed time: {train_time/60:.1f} minutes")
print(f"Best checkpoint: {BEST_CKPT}")
print(f"Archive: {PACKAGE_ZIP}")

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style="whitegrid", font_scale=1.1)

# ── Base dirs ─────────────────────────────────────────────────────────────
# Updated to use the requested /content/Data path
BASE_DIR = "/content"
MODELS_DIR = "/content/Data"
RESULTS_DIR = os.path.join(BASE_DIR, "comparison_results")
os.makedirs(RESULTS_DIR, exist_ok=True)

# ── Model registry ────────────────────────────────────────────────────────
MODEL_SPECS = [
    {"display_name": "SBERT", "aliases": ["SBERT", "sbert"]},
    {"display_name": "BioBERT", "aliases": ["BioBERT", "biobert"]},
    {"display_name": "Longformer", "aliases": ["Longformer", "longformer"]},
    {"display_name": "BigBird", "aliases": ["BigBird", "bigbird"]},
    {"display_name": "T5", "aliases": ["T5", "t5"]},
    {"display_name": "BART", "aliases": ["BART", "bart"]},
]

def safe_read_json(path):
    try:
        with open(path, "r") as f:
            return json.load(f)
    except Exception:
        return None

# ── Load results ───────────────────────────────────────────────────────────
ALL_RESULTS = {}
MODEL_FOLDERS = {}

print(f"Scanning {MODELS_DIR} for metrics.json files …")

for spec in MODEL_SPECS:
    found = False
    # Check each alias folder for a metrics.json
    for alias in spec["aliases"]:
        metrics_path = Path(MODELS_DIR) / alias / "metrics.json"
        if metrics_path.exists():
            metrics = safe_read_json(metrics_path)
            if metrics:
                ALL_RESULTS[spec["display_name"]] = metrics
                MODEL_FOLDERS[spec["display_name"]] = str(metrics_path.parent)
                print(f"  ✅ {spec['display_name']} found at {metrics_path}")
                found = True
                break
    if not found:
        print(f"  ⛔ {spec['display_name']}: metrics.json not found in any alias folders.")

if not ALL_RESULTS:
    print(f"❌ No metrics found in {MODELS_DIR}. Please ensure folders exist with metrics.json.")
else:
    MODEL_ORDER = [m for m in [s["display_name"] for s in MODEL_SPECS] if m in ALL_RESULTS]

    # ── Summary table ─────────────────────────────────────────────────────────
    rows = []
    for mn in MODEL_ORDER:
        m = ALL_RESULTS[mn]
        rows.append({
            "Model": mn,
            "Accuracy": m.get("accuracy", np.nan),
            "F1 (macro)": m.get("f1_macro", np.nan),
            "ROC-AUC": m.get("roc_auc", np.nan),
            "PR-AUC": m.get("pr_auc", np.nan),
            "Train time (s)": m.get("training_time_s", np.nan),
        })

    summary_df = pd.DataFrame(rows).set_index("Model")
    display(summary_df.round(4))

    # ── Plots ───────────────────────────────────────────────────────
    PALETTE = ["#4C72B0", "#DD8452", "#55A868", "#C44E52", "#8172B2", "#937860"]

    def bar_chart(metric_key, title, ylabel, fname):
        fig, ax = plt.subplots(figsize=(9, 4))
        vals = [ALL_RESULTS[m].get(metric_key, 0) for m in MODEL_ORDER]
        bars = ax.bar(MODEL_ORDER, vals, color=PALETTE[:len(MODEL_ORDER)], width=0.6)
        for bar, val in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width() / 2, val + 0.005, f"{val:.3f}", ha="center", va="bottom")
        ax.set_title(title, fontweight="bold")
        ax.set_ylabel(ylabel)
        ax.set_ylim(0, 1.1)
        plt.tight_layout()
        plt.savefig(os.path.join(RESULTS_DIR, fname), dpi=150)
        plt.show()

    bar_chart("accuracy", "Accuracy Comparison", "Score", "cmp_accuracy.png")
    bar_chart("f1_macro", "F1 Macro Comparison", "Score", "cmp_f1.png")

    best_model = summary_df["F1 (macro)"].idxmax()
    print(f"\n🏆 Best model by F1 (macro): {best_model}")

In [ ]:
import os
import re
import json
import unicodedata
from pathlib import Path

import torch
from transformers import AutoTokenizer, BartForSequenceClassification, BartForConditionalGeneration

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

BART_ROOT = Path("/content/saved_models/BART")
MODEL_DIR = BART_ROOT / "model"
TOKENIZER_DIR = BART_ROOT / "tokenizer"
BEST_CKPT = BART_ROOT / "best_model.pt"
THRESH_PATH = BART_ROOT / "threshold.json"
LABEL_MAP_PATH = BART_ROOT / "label_map.json"

def load_json(path, default):
    if path.exists():
        with open(path, "r") as f:
            return json.load(f)
    return default

def clean_text(text, max_words=300):
    if text is None:
        return ""
    text = unicodedata.normalize("NFC", str(text))
    text = re.sub(r"\s+", " ", text).strip()
    words = text.split()
    if len(words) > max_words:
        text = " ".join(words[:max_words])
    return text

def multiline_input(prompt):
    print(prompt)
    print("Paste text, then press Enter on an empty line to finish:")
    lines = []
    while True:
        line = input()
        if line.strip() == "":
            break
        lines.append(line)
    return "\n".join(lines)

thr_payload = load_json(THRESH_PATH, {"threshold": 0.5})
THRESHOLD = float(thr_payload.get("threshold", 0.5))
LABEL_MAP = load_json(LABEL_MAP_PATH, {"0": "Different subdomain", "1": "Same subdomain"})

tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_DIR, use_fast=True)

BART_MODE = "seqcls"
try:
    model = BartForSequenceClassification.from_pretrained(MODEL_DIR)
    ckpt = torch.load(BEST_CKPT, map_location=DEVICE)
    if isinstance(ckpt, dict) and "state_dict" in ckpt:
        model.load_state_dict(ckpt["state_dict"], strict=False)
    else:
        model.load_state_dict(ckpt, strict=False)
except Exception:
    BART_MODE = "gen"
    model = BartForConditionalGeneration.from_pretrained(MODEL_DIR)

model.to(DEVICE)
model.eval()

def predict_bart(abstract, conclusion, max_len=512):
    abstract = clean_text(abstract, 300)
    conclusion = clean_text(conclusion, 300)

    if BART_MODE == "seqcls":
        prompt = f"classify: Abstract: {abstract} [SEP] Conclusion: {conclusion}"
        enc = tokenizer(
            prompt,
            max_length=max_len,
            truncation=True,
            padding="max_length",
            return_tensors="pt"
        )
        enc = {k: v.to(DEVICE) for k, v in enc.items()}

        with torch.no_grad():
            logits = model(**enc).logits
            prob_same = torch.softmax(logits, dim=-1)[0, 1].item()
    else:
        prompt = f"classify: Abstract: {abstract} [SEP] Conclusion: {conclusion}"
        enc = tokenizer(
            prompt,
            max_length=max_len,
            truncation=True,
            padding="max_length",
            return_tensors="pt"
        )
        enc = {k: v.to(DEVICE) for k, v in enc.items()}

        same_ids = tokenizer("same_subdomain", add_special_tokens=False).input_ids
        diff_ids = tokenizer("different_subdomain", add_special_tokens=False).input_ids
        same_id = same_ids[0] if same_ids else tokenizer.eos_token_id
        diff_id = diff_ids[0] if diff_ids else tokenizer.eos_token_id

        decoder_input_ids = torch.full(
            (enc["input_ids"].size(0), 1),
            tokenizer.pad_token_id,
            dtype=torch.long,
            device=DEVICE
        )

        with torch.no_grad():
            out = model(
                input_ids=enc["input_ids"],
                attention_mask=enc["attention_mask"],
                decoder_input_ids=decoder_input_ids
            )
            step0 = out.logits[:, 0, :]
            two_logits = step0[:, [diff_id, same_id]]
            prob_same = torch.softmax(two_logits, dim=-1)[0, 1].item()

    pred = int(prob_same >= THRESHOLD)
    return {
        "label": pred,
        "label_name": LABEL_MAP.get(str(pred), str(pred)),
        "probability_same_subdomain": prob_same,
        "threshold": THRESHOLD
    }

abstract = multiline_input("Enter ABSTRACT")
conclusion = multiline_input("Enter CONCLUSION")

result = predict_bart(abstract, conclusion)
print(result)